# LaPA · FP8 · 560 вопросов
A100 40 GB, Python 3.13. Chat 5-shot, thinking off.
Выполните все ячейки; разрешите доступ к Google Drive. Одна среда за раз.
Общий лимит — 12 часов с подготовкой и продолжениями.


In [ ]:
import hashlib
import json
import os
import shutil
import signal
import subprocess
import sys
import threading
import time
import venv
import zipfile
from pathlib import Path
from uuid import uuid4

from google.colab import drive

if sys.version_info[:2] != (3, 13):
    raise RuntimeError('Choose Python 3.13 in Colab')
if 'BUDGET' in globals():
    raise RuntimeError('Restart Python before running all cells again')
if 'SESSION_STARTED' not in globals():
    started = time.monotonic()
    SESSION_ID = uuid4().hex
    drive.mount('/content/drive')
    WORK_ROOT = Path('/content/lapa_fp8_560')
    RUN_ROOT = WORK_ROOT / SESSION_ID
    STUDY = Path('/content/drive/MyDrive/LaPA/fp8-chat5-560-ea2fa680')
    RESULTS_DIR = STUDY / 'results'
    SESSION_DIR = STUDY / 'sessions' / SESSION_ID
    for directory in (RUN_ROOT, RESULTS_DIR, SESSION_DIR):
        directory.mkdir(parents=True, exist_ok=True)
    SESSION_STARTED = started
print('Results:', STUDY)


## Код и фиксированная выборка


In [ ]:
SOURCES = {
    'lm_eval_tasks/mmlu_prox_ru_chat/_mmlu_prox_ru_chat.yaml': r'''group: mmlu_prox_ru_chat
task:
  - mmlu_prox_ru_chat_biology
  - mmlu_prox_ru_chat_business
  - mmlu_prox_ru_chat_chemistry
  - mmlu_prox_ru_chat_computer_science
  - mmlu_prox_ru_chat_economics
  - mmlu_prox_ru_chat_engineering
  - mmlu_prox_ru_chat_health
  - mmlu_prox_ru_chat_history
  - mmlu_prox_ru_chat_law
  - mmlu_prox_ru_chat_math
  - mmlu_prox_ru_chat_other
  - mmlu_prox_ru_chat_philosophy
  - mmlu_prox_ru_chat_physics
  - mmlu_prox_ru_chat_psychology
aggregate_metric_list:
  - aggregation: mean
    metric: exact_match
    weight_by_size: true
    filter_list: custom-extract
metadata:
  version: 0.0
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/_ru_chat_template.yaml': r'''dataset_path: li-lab/MMLU-ProX
dataset_name: ru
dataset_kwargs:
  revision: 8e6106a6c6ce1c5027e66cc338143cf997b2aa09

test_split: test
fewshot_split: validation

description: |
  Перед вами вопрос с множественным выбором.
  Выберите правильный вариант ответа.
  Финальный ответ должен заканчиваться в формате "Ответ - (X)",
  где X — латинская буква правильного варианта.

fewshot_config:
  sampler: first_n
  doc_to_text: !function utils.doc_to_text
  doc_to_target: !function utils.fewshot_target

output_type: generate_until

doc_to_text: !function utils.doc_to_text
doc_to_target: answer

filter_list:
  - name: custom-extract
    filter:
      - function: regex
        regex_pattern: 'Ответ - \(?([ABCDEFGHIJ])\)?'
      - function: take_first

generation_kwargs:
  until:
    - "</s>"
    - "Q:"
    - "Вопрос:"
    - "<|im_end|>"
  do_sample: false
  temperature: 0.0
  max_gen_toks: 2048

num_fewshot: 5

metric_list:
  - metric: exact_match
    aggregation: mean
    higher_is_better: true
    ignore_case: true
    ignore_punctuation: true

metadata:
  version: 0.0
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_biology.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_biology
task_alias: biology
process_docs: !function utils.process_biology
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_business.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_business
task_alias: business
process_docs: !function utils.process_business
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_chemistry.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_chemistry
task_alias: chemistry
process_docs: !function utils.process_chemistry
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_computer_science.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_computer_science
task_alias: computer_science
process_docs: !function utils.process_computer_science
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_economics.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_economics
task_alias: economics
process_docs: !function utils.process_economics
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_engineering.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_engineering
task_alias: engineering
process_docs: !function utils.process_engineering
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_health.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_health
task_alias: health
process_docs: !function utils.process_health
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_history.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_history
task_alias: history
process_docs: !function utils.process_history
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_law.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_law
task_alias: law
process_docs: !function utils.process_law
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_math.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_math
task_alias: math
process_docs: !function utils.process_math
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_other.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_other
task_alias: other
process_docs: !function utils.process_other
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_philosophy.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_philosophy
task_alias: philosophy
process_docs: !function utils.process_philosophy
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_physics.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_physics
task_alias: physics
process_docs: !function utils.process_physics
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_psychology.yaml': r'''include: _ru_chat_template.yaml
task: mmlu_prox_ru_chat_psychology
task_alias: psychology
process_docs: !function utils.process_psychology
''',
    'lm_eval_tasks/mmlu_prox_ru_chat/utils.py': r'''from functools import partial

CHOICES = list("ABCDEFGHIJ")
MAX_OPTIONS = 10


def doc_to_text(example):
    prompt = "Вопрос:\n"
    prompt += example["question"] + "\n"
    prompt += "Варианты:\n"

    for i in range(MAX_OPTIONS):
        option = example[f"option_{i}"]
        if option is not None:
            prompt += f"{CHOICES[i]}. {option}\n"

    return prompt


def fewshot_target(example):
    return f"Ответ - ({example['answer']})"


def process_docs(dataset, subject):
    return dataset.filter(lambda x: x["category"] == subject)


process_biology = partial(process_docs, subject="biology")
process_business = partial(process_docs, subject="business")
process_chemistry = partial(process_docs, subject="chemistry")
process_computer_science = partial(
    process_docs, subject="computer science"
)
process_economics = partial(process_docs, subject="economics")
process_engineering = partial(process_docs, subject="engineering")
process_health = partial(process_docs, subject="health")
process_history = partial(process_docs, subject="history")
process_law = partial(process_docs, subject="law")
process_math = partial(process_docs, subject="math")
process_other = partial(process_docs, subject="other")
process_philosophy = partial(process_docs, subject="philosophy")
process_physics = partial(process_docs, subject="physics")
process_psychology = partial(process_docs, subject="psychology")
''',
    'requirements.txt': r'''--index-url https://pypi.org/simple
--extra-index-url https://download.pytorch.org/whl/cu129

absl-py==2.5.0
agent-detector==2.0.0
aiohappyeyeballs==2.7.1
aiohttp==3.14.4
aiosignal==1.4.0
annotated-doc==0.0.5
annotated-types==0.8.0
anthropic==1.11.0
anyio==4.15.1
apache-tvm-ffi==0.1.9
astor==0.8.1
attrs==26.1.0
blake3==1.0.10
cachetools==7.2.1
cbor2==6.1.5
certifi==2026.7.22
cffi==2.1.1
chardet==6.0.0.post1
charset-normalizer==3.5.2
click==8.5.0
cloudpickle==3.1.2
colorama==0.4.6
compressed-tensors==0.15.0.1
cryptography==50.0.2
cuda-bindings==12.9.9
cuda-pathfinder==1.8.3
cuda-tile==1.6.0
cuda-toolkit==12.9.1
dataproperty==1.1.1
datasets==4.6.1
defusedxml==0.7.1
depyf==0.20.0
detect-installer==0.2.1
dill==0.4.0
diskcache==5.6.3
dnspython==2.8.0
docstring-parser==0.18.0
einops==0.8.2
email-validator==2.3.0
evaluate==0.4.6
fastapi==0.142.2
fastapi-cli==0.0.32
fastapi-cloud-cli==0.26.0
fastar==0.12.0
fastsafetensors==0.4.0
filelock==4.0.12
flashinfer-cubin==0.6.8.post1
flashinfer-python==0.6.8.post1
frozenlist==1.8.0
fsspec==2026.2.0
gguf==0.19.0
googleapis-common-protos==1.75.5
grpcio==1.84.0
h11==0.16.0
hf-xet==1.7.0
httpcore==1.0.9
httpcore2==2.13.1
httptools==0.8.0
httpx==0.28.1
httpx2==2.13.1
huggingface-hub==1.29.0
idna==3.20
ijson==3.5.1
interegular==0.3.3
jinja2==3.1.6
jiter==0.17.0
jmespath==1.1.0
joblib==1.6.0
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
lark==1.2.2
llguidance==1.3.0
llvmlite==0.47.0
lm-eval @ git+https://github.com/EleutherAI/lm-evaluation-harness.git@d6de81643928d653435c431bae19945d41d32520
lm-format-enforcer==0.11.3
loguru==0.7.3
lxml==6.1.3
markdown-it-py==4.2.0
markupsafe==3.0.4
mbstrdecoder==1.1.5
mcp==2.3.0
mcp-types==2.3.0
mdurl==0.1.2
mistral-common==1.12.0
ml-dtypes==0.6.0
model-hosting-container-standards==0.1.16
more-itertools==11.1.0
mpmath==1.3.0
msgspec==0.22.0
multidict==6.9.1
multiprocess==0.70.18
narwhals==2.26.0
networkx==3.7
ninja==1.13.2
nltk==3.10.3
numba==0.65.0
numpy==2.4.6
nvidia-cublas-cu12==12.9.1.4
nvidia-cuda-cupti-cu12==12.9.79
nvidia-cuda-nvrtc-cu12==12.9.86
nvidia-cuda-runtime-cu12==12.9.79
nvidia-cudnn-cu12==9.17.1.4
nvidia-cudnn-frontend==1.18.0
nvidia-cufft-cu12==11.4.1.4
nvidia-cufile-cu12==1.14.1.1
nvidia-curand-cu12==10.3.10.19
nvidia-cusolver-cu12==11.7.5.82
nvidia-cusparse-cu12==12.5.10.65
nvidia-cusparselt-cu12==0.7.1
nvidia-cutlass-dsl==4.8.0
nvidia-cutlass-dsl-libs-base==4.8.0
nvidia-cutlass-dsl-libs-core==4.8.0
nvidia-cutlass-dsl-libs-cu12==4.8.0
nvidia-ml-py==13.615.71
nvidia-nccl-cu12==2.28.9
nvidia-nvjitlink-cu12==12.9.86
nvidia-nvshmem-cu12==3.4.5
nvidia-nvtx-cu12==12.9.79
openai==3.26.0
openai-harmony==0.0.8
opencv-python-headless==5.0.0.93
opentelemetry-api==1.45.1
opentelemetry-exporter-http-transport==0.66b1
opentelemetry-exporter-otlp==1.45.1
opentelemetry-exporter-otlp-common==0.66b1
opentelemetry-exporter-otlp-proto-common==1.45.1
opentelemetry-exporter-otlp-proto-grpc==1.45.1
opentelemetry-exporter-otlp-proto-http==1.45.1
opentelemetry-proto==1.45.1
opentelemetry-sdk==1.45.1
opentelemetry-semantic-conventions==0.66b1
opentelemetry-semantic-conventions-ai==0.5.1
outlines-core==0.2.14
packaging==26.3
pandas==3.0.6
partial-json-parser==0.2.1.1.post7
pathvalidate==3.3.1
pillow==12.3.0
portalocker==4.4.0
prometheus-client==0.26.0
prometheus-fastapi-instrumentator==8.1.0
propcache==0.5.4
protobuf==7.36.2
psutil==7.2.2
py-cpuinfo==9.0.0
pyarrow==25.0.1
pybase64==1.5.1
pycountry==26.2.16
pycparser==3.0
pydantic==2.13.5
pydantic-core==2.46.5
pydantic-extra-types==2.11.1
pydantic-settings==2.15.0
pygments==2.21.0
pyjwt==2.15.1
pytablewriter==1.2.1
python-dateutil==2.9.0.post0
python-dotenv==1.2.4
python-json-logger==4.2.0
python-multipart==0.0.32
pytz==2026.5
pyyaml==6.0.3
pyzmq==27.2.0
quack-kernels==0.6.5
referencing==0.37.0
regex==2026.9.29
requests==2.34.2
rich==15.0.0
rich-toolkit==0.20.5
rignore==0.8.1
rouge-score==0.1.2
rpds-py==2026.9.1
sacrebleu==2.6.0
safetensors==0.8.0
scikit-learn==1.9.1
scipy==1.18.1
sentencepiece==0.2.2
sentry-sdk==2.71.0
setproctitle==1.3.8
setuptools==80.10.2
shellingham==1.5.4
six==1.17.0
sniffio==1.3.1
sqlitedict==2.1.0
sse-starlette==3.5.0
starlette==1.7.0
supervisor==4.3.0
sympy==1.14.0
tabledata==1.3.5
tabulate==0.10.0
tcolorpy==0.1.7
threadpoolctl==3.7.0
tiktoken==0.14.0
tilelang==0.1.9
tokenizers==0.22.2
torch==2.11.0+cu129
torch-c-dlpack-ext==0.1.5
torchaudio==2.11.0+cu129
torchvision==0.26.0+cu129
tqdm==4.70.1
transformers==5.5.1
triton==3.6.0
truststore==0.10.4
typepy==1.3.5
typer==0.27.3
typing-extensions==4.16.0
typing-inspection==0.4.4
urllib3==2.8.0
uvicorn==0.54.0
uvloop==0.23.0
vllm @ https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0%2Bcu129-cp38-abi3-manylinux_2_31_x86_64.whl#sha256=b687351527b0bca7f6b0f367b6c93499e19b06f0c9a05b01084c043f7ec50ce7
watchfiles==1.3.0
websockets==17.2
word2number==1.1
xgrammar==0.2.3
xxhash==4.0.1
yarl==1.25.1
z3-solver==4.15.4.0
''',
    'baseline.py': r'''"""FP8 baseline; scoring and generation belong to lm-eval."""

import hashlib
import json
import os
import time
import traceback
from pathlib import Path
from uuid import uuid4

import lm_eval
from lm_eval.models.vllm_causallms import VLLM
from lm_eval.utils import handle_non_serializable

DATA_REVISION = "8e6106a6c6ce1c5027e66cc338143cf997b2aa09"
MODEL_REVISION = "017b9c7af6b5689d5dd426a76e0bc077eb5ca20a"
BASE_REVISION = "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0"
HARNESS_REVISION = "d6de81643928d653435c431bae19945d41d32520"
MODEL_ID = "Qwen/Qwen3.8-27B-FP8"


class AuditedVLLM(VLLM):
    """Record native outputs and reject changed or truncated inputs."""

    raw_path = None
    expected_input_hash = None

    def generate_until(self, requests, **kwargs):
        for request in requests:
            tokens = self.tok_encode(request.args[0])
            limit = request.args[1].get("max_gen_toks", 2048)
            if len(tokens) + limit > self.max_length:
                raise ValueError("Input exceeds context; no truncation")
            digest = hashlib.sha256(json.dumps(tokens).encode()).hexdigest()
            if self.expected_input_hash and digest != self.expected_input_hash:
                raise ValueError("Prompt tokens differ from frozen preflight")
        return super().generate_until(requests, **kwargs)

    def _model_generate(self, requests, generate=False, sampling_params=None):
        started = time.monotonic()
        outputs = super()._model_generate(
            requests, generate=generate, sampling_params=sampling_params,
        )
        elapsed = time.monotonic() - started
        if generate and self.raw_path is not None:
            with self.raw_path.open("a", encoding="utf-8") as stream:
                for output in outputs:
                    answer = output.outputs[0]
                    record = {
                        "input_ids": list(output.prompt_token_ids),
                        "generated_ids": list(answer.token_ids),
                        "raw_text": answer.text,
                        "finish_reason": answer.finish_reason,
                        "stop_reason": answer.stop_reason,
                        "generation_seconds": elapsed,
                        "note": "vLLM wall time; not paired NF4 speedup",
                    }
                    stream.write(json.dumps(record, ensure_ascii=False))
                    stream.write("\n")
        return outputs


def save_json(path, value):
    """Never overwrite a previous result."""
    path = Path(path)
    if path.exists():
        raise FileExistsError(path)
    temporary = path.with_name(path.name + '.' + uuid4().hex + '.tmp')
    with temporary.open("x", encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2,
                  default=handle_non_serializable)
    os.replace(temporary, path)


def seal(directory):
    """Hash saved files, including partial raw output after failure."""
    directory = Path(directory)
    files = {}
    for path in sorted(directory.rglob("*")):
        if path.is_file() and path.name != "manifest.json":
            with path.open("rb") as stream:
                digest = hashlib.file_digest(stream, "sha256").hexdigest()
            files[str(path.relative_to(directory))] = digest
    save_json(directory / "manifest.json", files)


def run_eval(lm, task_manager, directory, name, task, sample_index,
             document_hash, prepared_task=None):
    """Evaluate chat 5-shot without native thinking."""
    run_dir = Path(directory) / name
    run_dir.mkdir(parents=True, exist_ok=False)
    lm.enable_thinking = False
    lm.think_end_token = None
    lm.raw_path = run_dir / "raw_generations.jsonl"
    started = time.monotonic()
    save_json(run_dir / "settings.json", {
        "task": task, "shots": 5, "chat": True,
        "thinking": False, "limit": None,
        "dataset_revision": DATA_REVISION,
        "model_revision": MODEL_REVISION,
        "harness_revision": HARNESS_REVISION,
        "sample_index": sample_index, "document_hash": document_hash,
        "debug": False, "split": "test",
    })
    try:
        if prepared_task is None:
            prepared_task = task_manager._factory.build(
                task_manager.task_index[task],
                overrides={"dataset_kwargs": {"revision": DATA_REVISION}},
                registry=task_manager.task_index,
            )
        document = prepared_task.test_docs()[sample_index]
        if digest_json(document) != document_hash:
            raise ValueError("Selected document differs from the plan")
        result = lm_eval.simple_evaluate(
            model=lm,
            tasks=[prepared_task],
            task_manager=task_manager,
            num_fewshot=5,
            apply_chat_template=True,
            fewshot_as_multiturn=True,
            samples={task: [sample_index]},
            log_samples=True,
            bootstrap_iters=0,
            random_seed=0,
            numpy_random_seed=1234,
            torch_random_seed=1234,
            fewshot_random_seed=1234,
        )
        save_json(run_dir / f"{name}.json", result)
        save_json(run_dir / "timing.json", {
            "wall_seconds": time.monotonic() - started,
            "note": "Evaluation wall time, not paired inference E2E",
        })
        seal(run_dir)
        save_json(run_dir / "completed.json", {"status": "completed"})
        return result
    except BaseException:
        original_traceback = traceback.format_exc()
        try:
            save_json(run_dir / "failure.json", {
                "traceback": original_traceback,
                "wall_seconds": time.monotonic() - started,
            })
        except OSError as error:
            print("Could not save failure diagnostics:", error)
        raise
    finally:
        lm.raw_path = None


def digest_json(value):
    """Stable identity for documents and frozen plans."""
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False,
                         separators=(",", ":")).encode()
    return hashlib.sha256(encoded).hexdigest()


def verify_run(directory):
    """Accept only a sealed, completed harness result."""
    directory = Path(directory)
    marker = directory / 'completed.json'
    if not marker.exists():
        return False
    try:
        if json.loads(marker.read_text()) != {'status': 'completed'}:
            return False
    except json.JSONDecodeError:
        return False
    manifest = json.loads((directory / 'manifest.json').read_text())
    for name, expected in manifest.items():
        path = directory / name
        if not path.resolve().is_relative_to(directory.resolve()):
            raise ValueError('Unsafe run manifest path')
        with path.open('rb') as stream:
            actual = hashlib.file_digest(stream, 'sha256').hexdigest()
        if actual != expected:
            raise ValueError('Run checksum mismatch')
    required = {'settings.json', 'timing.json', 'raw_generations.jsonl',
                directory.name + '.json'}
    if not required <= manifest.keys():
        raise ValueError('Incomplete run manifest')
    return True


def freeze_json(path, value):
    """Keep a plan immutable across sessions."""
    path = Path(path)
    if path.exists():
        if json.loads(path.read_text()) != value:
            raise ValueError('Frozen plan differs; do not mix experiments')
    else:
        save_json(path, value)
    return value


def finished_attempt(study, phase, mode, item):
    """Find at most one completed attempt for the exact frozen document."""
    root = Path(study) / phase / mode / (
        item['subject'] + '_' + str(item['index'])
    )
    complete = [p for p in sorted(root.glob('attempt_*')) if verify_run(p)]
    if len(complete) > 1:
        raise ValueError('Duplicate completed document')
    if not complete:
        return None
    result = complete[0]
    settings = json.loads((result / 'settings.json').read_text())
    if settings['document_hash'] != item['sha256']:
        raise ValueError('Completed result belongs to another document')
    return result
''',
    'run.py': r'''"""Run one FP8 protocol; durable completed answers are reused."""

import hashlib
import importlib.metadata
import json
import os
import sys
import time
from pathlib import Path
from types import SimpleNamespace
from uuid import uuid4

os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'


def prepare(manager, selection, tokenizer):
    from baseline import AuditedVLLM, digest_json

    prefix = tokenizer.bos_token_id
    if prefix is None:
        prefix = tokenizer.eos_token_id
    proxy = SimpleNamespace(
        tokenizer=tokenizer, hf_chat_template=None,
        enable_thinking=False, chat_template_args={},
        prefix_token_id=prefix, add_bos_token=None,
    )

    def render(messages, **kwargs):
        roles = [m['role'] for m in messages]
        if roles != ['system'] + ['user', 'assistant'] * 5 + ['user']:
            raise ValueError('Expected five chat demonstration pairs')
        return AuditedVLLM.apply_chat_template(proxy, messages, **kwargs)

    tasks, prompts = {}, {}
    for subject, demos in selection['demonstrations'].items():
        name = 'mmlu_prox_ru_chat_' + subject
        task = manager._factory.build(
            manager.task_index[name], registry=manager.task_index,
            overrides={'dataset_kwargs': {
                'revision': selection['dataset_revision'],
            }},
        )
        actual = [{'question_id': d['question_id'], 'sha256': digest_json(d)}
                  for d in task.fewshot_docs()]
        if actual != demos:
            raise ValueError('Validation demonstrations changed')
        task.set_fewshot_seed(1234)
        tasks[subject] = task
        selected = [r for r in selection['items'] if r['subject'] == subject]
        for item in selected:
            doc = task.test_docs()[item['index']]
            if digest_json(doc) != item['sha256']:
                raise ValueError('Selected question changed')
            prompt = task.fewshot_context(
                doc, 5, apply_chat_template=True, fewshot_as_multiturn=True,
                chat_template=render, gen_prefix=task.doc_to_prefix(doc),
            )
            tokens = AuditedVLLM.tok_encode(proxy, prompt)
            if len(tokens) + 2048 > 32768:
                raise ValueError('Context limit exceeded; no truncation')
            prompts[str(item['question_id'])] = hashlib.sha256(
                json.dumps(tokens).encode(),
            ).hexdigest()
    return tasks, prompts


def main():
    from baseline import (
        BASE_REVISION,
        HARNESS_REVISION,
        MODEL_ID,
        MODEL_REVISION,
        AuditedVLLM,
        freeze_json,
        run_eval,
    )
    from evaluate import evaluate, summarize
    from huggingface_hub import hf_hub_download
    from lm_eval.tasks import TaskManager
    from transformers import AutoTokenizer

    root = Path(__file__).parent
    results = Path(sys.argv[1])
    deadline = float(sys.argv[2])
    selection = json.loads((root / 'selection.json').read_text())
    freeze_json(results / 'selection.json', selection)
    if summarize(results, selection) is not None:
        print('All 560 answers already completed.', flush=True)
        return
    provenance = json.loads(importlib.metadata.distribution('lm-eval')
                            .read_text('direct_url.json'))
    if provenance.get('vcs_info', {}).get('commit_id') != HARNESS_REVISION:
        raise RuntimeError('Harness revision mismatch')
    model_sources = {}
    for repo, revision in [('Qwen/Qwen3.8-27B', BASE_REVISION),
                           (MODEL_ID, MODEL_REVISION)]:
        files = {}
        for name in ('config.json', 'tokenizer.json',
                     'tokenizer_config.json', 'chat_template.jinja'):
            path = Path(hf_hub_download(repo, name, revision=revision))
            files[name] = hashlib.sha256(path.read_bytes()).hexdigest()
        model_sources[repo] = files
    expected = json.loads((root / 'model_sources.json').read_text())
    if model_sources != expected:
        raise RuntimeError('Model metadata differs from reviewed sources')
    freeze_json(results / 'model_sources.json', model_sources)
    for name, expected_hash in selection['dataset_files'].items():
        path = Path(hf_hub_download(
            selection['dataset'], name, repo_type='dataset',
            revision=selection['dataset_revision'],
        ))
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected_hash:
            raise RuntimeError('Dataset file checksum mismatch')
    tokenizer = AutoTokenizer.from_pretrained(
        'Qwen/Qwen3.8-27B', revision=BASE_REVISION,
    )
    manager = TaskManager(include_path=str(root / 'lm_eval_tasks'))
    tasks, prompts = prepare(manager, selection, tokenizer)
    freeze_json(results / 'prompt_manifest.json', prompts)
    if time.monotonic() >= deadline - 300:
        raise TimeoutError('Insufficient budget before model loading')
    print('Prompts verified. Loading FP8 once; A100 W8A16 Marlin.', flush=True)
    lm = AuditedVLLM(
        pretrained=MODEL_ID, revision=MODEL_REVISION,
        tokenizer='Qwen/Qwen3.8-27B', tokenizer_revision=BASE_REVISION,
        dtype='bfloat16', quantization='fp8', batch_size=1,
        max_model_len=32768, max_gen_toks=2048, max_num_seqs=1,
        max_num_batched_tokens=2048, gpu_memory_utilization=0.90,
        enforce_eager=True, enable_prefix_caching=False,
        language_model_only=True, seed=1234, enable_thinking=False,
    )
    smoke = next(r for r in selection['calibration']
                 if r['subject'] == 'biology')
    smoke_result = run_eval(
        lm, manager, results / 'preflight', 'calibration_' + uuid4().hex[:8],
        'mmlu_prox_ru_chat_biology', smoke['index'], smoke['sha256'],
        prepared_task=tasks['biology'],
    )
    sample = next(iter(smoke_result['samples'].values()))[0]
    if sample['filtered_resps'] == ['[invalid]']:
        raise RuntimeError('Calibration produced no extractable answer')
    evaluate(lm, manager, tasks, results, selection, prompts, deadline)
    summary = summarize(results, selection)
    if summary is None:
        raise RuntimeError('Incomplete run; no final accuracy')
    print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)


if __name__ == '__main__':
    main()
''',
    'model_sources.json': r'''{
  "Qwen/Qwen3.8-27B": {
    "config.json": "191e0af232104ed8b65258cf3fb2b842e288008baca7633c11b82a1ac7203aab",
    "tokenizer.json": "0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3",
    "tokenizer_config.json": "b11349aafa7cdc6a320767cf7ceb29ed82f7eda5d65e8e0819e76f0ce947bf27",
    "chat_template.jinja": "c3cf9e34abf4f9e36c2d72165aa9c132d3e2a725b6c2586aaa3a8af9d7a81041"
  },
  "Qwen/Qwen3.8-27B-FP8": {
    "config.json": "74227dd615bf1ea975aa676bdf355a0379858c12f394b5365cd9dfa5fc2c70bc",
    "tokenizer.json": "0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3",
    "tokenizer_config.json": "b11349aafa7cdc6a320767cf7ceb29ed82f7eda5d65e8e0819e76f0ce947bf27",
    "chat_template.jinja": "c3cf9e34abf4f9e36c2d72165aa9c132d3e2a725b6c2586aaa3a8af9d7a81041"
  }
}
''',
    'protocol.md': r'''# Расширенная оценка FP8: chat 5-shot

ID: chat-5shot-fp8-expanded-v1. Дата: 07.10.2026.
Протокол и выборка зафиксированы до генерации ответов.
Состояние исполнения — в [STATUS](../STATUS.md).

## Объём и бюджет

560 вопросов: по 40 на каждый из 14 предметов, без старых 28 вопросов.
Пользователь выделил до 12 A100-часов для этого одного режима,
включая установку, загрузку, calibration, ожидание и экспорт.
Это не бюджет шести режимов или обучения Franken.

Основание размера: завершённый FP8 прогон, 1042.821348 с / 28 =
37.244 с на вопрос evaluation wall. 560 вопросов по этому среднему —
5.79 ч. Коэффициент запаса 1.5 и 0.75 ч на подготовку/calibration/экспорт
дают 9.44 ч, оставляя 2.56 ч до предела. Это сценарная оценка по малой
выборке, не верхняя граница: новые вопросы могут требовать больше токенов.
Размер 40 на предмет выбран до новых ответов; по score его не менять.

## Фиксация данных

[Manifest](../../experiments/chat-5shot-fp8-expanded-v1/selection.json)
содержит ID, индекс внутри предмета, SHA документа, порядок, исключения,
70 validation demonstrations, 14 calibration ID, ревизии и хеши источников.
SHA-256 manifest:
`ea2fa68075e80de45e42e286e99c07dabf8c81a6b31fd851306d23db5cbdd988`.

Источник: li-lab/MMLU-ProX, ru, revision
`8e6106a6c6ce1c5027e66cc338143cf997b2aa09`, test = 11759 строк.
Скрипт [fixed_sample.py](../../eval_support/fixed_sample.py) воспроизводит
выборку; существующий manifest он не перезаписывает. Входные Parquet
скачиваются с этой ревизии; их SHA записаны в manifest.

Правила исключения:

- Весь прежний Lite test: 588 строк, 586 различных нормализованных текстов.
  ID Lite и полного набора несовместимы; сопоставление только по тексту.
  Из полного test исключены 604 соответствующие строки, включая повторы.
- Вопросы с сохранёнными ответами во всех десяти перечисленных старых
  harness ZIP; 31 соответствующая test-строка, включая прежние 28.
- Все 14 ранее отведённых calibration-вопросов.
- Совпадения с validation и все группы одинаковых текстов в test
  (103 строки с повторяющимся текстом). Не выбирать один удобный дубль.

Нормализация: Unicode NFKC, схлопывание пробелов, casefold.
Причины пересекаются: всего исключены 725 строк, осталось 11034.
Старые списки всех кандидатов без ответов не считаются экспозицией.
Нормализованное совпадение не обнаруживает все перефразированные дубли;
это контроль известной проектной экспозиции, не аудит pretraining.

В каждом предмете сортировка по SHA-256 канонического JSON
`seed=20261007, subject, question_id`, затем первые 40; score и длина
ответа не участвуют. Порядок исполнения: первый выбранный вопрос каждого
предмета по алфавиту, затем второй и так далее, 40 сбалансированных кругов.
Замены после ошибок/invalid, подбора seed и отбора по качеству нет.

## Неизменные настройки

Qwen/Qwen3.8-27B-FP8 revision
`017b9c7af6b5689d5dd426a76e0bc077eb5ca20a`; исходный tokenizer revision
`1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0`.
A100 40 GB, W8A16 Marlin/BF16, vLLM 0.20.0+cu129, harness revision
`d6de81643928d653435c431bae19945d41d32520`.
Зависимости и task/template sources привязаны к успешному FP8 ZIP.

Chat 5-shot, native thinking off; пять validation user/assistant пар
своего предмета в исходном порядке, без CoT в demonstrations.
Greedy, temperature 0, batch/concurrency 1, max output 2048,
context 32768 без обрезания, prefix cache off, eager, seed 1234.
Seeds harness: Python 0, NumPy/Torch/fewshot 1234.
Штатные stop sequences и regex → exact_match сохраняются.
Один прежний calibration biology 3273 до оценки, не в score;
успех проверки — извлечённая буква, правильность не условие запуска.

## Метрики и остановка

Основная метрика: macro accuracy по 14 предметам; при равных 40 вопросах
она равна 560-question accuracy. Отдельно accuracy каждого предмета,
число invalid и ответов на лимите. Invalid считается неверным.
Вторичная оценка: сумма subject accuracy с весами eligible_count / 11034;
она относится к оставшемуся пулу, не ко всему официальному test.

95% интервал основной метрики: стратифицированный percentile bootstrap,
10000 повторов, seed 20261008; в каждом предмете 40 ответов с возвращением,
затем среднее 14 accuracies. Это приближённый интервал, без поправки на
конечную популяцию. Сохранить раздельно старые 28 и новые 560 результатов.
NF4 на новых вопросах не измерен; разность с его 21/28 не считать парной.
После получения ответов набор становится просмотренным evaluation-набором;
по нему не выбирать обучающие гиперпараметры или лучший checkpoint.

Evaluation wall и generation wall описательные: без заявления speedup.
Это один проход качества, не парное latency-измерение с повторами.
Записывать длины, время подготовки/calibration и общий расход сессий;
неизмеренный peak GPU/TTFT не восстанавливать по косвенным логам.

По исчерпании бюджета — остановка и сохранение частичного результата.
Незавершённый прогон не объявлять оценкой всех 560 вопросов; не уменьшать
знаменатель до удобного завершённого префикса и не заменять вопросы.
Перед GPU-запуском нужен notebook с продолжением и сохранением вне
/content; суммарный бюджет продолжений также не выше 12 часов.
Текущий успешный notebook на 28 вопросов не является этим запуском.

## Проверка

Проверить повторное построение manifest, равные страты, отсутствие
пересечений и SHA источников. На CPU построить все 560 промптов штатным
harness и настоящим методом VLLM с закреплённым tokenizer/template:
пять пар, thinking off, input + 2048 <= 32768. Это не GPU-прогон.
`make check`, `make artifacts`. Коммит и пуш только с одобрения автора.

## Исполнение в Colab

Отдельный [ноутбук](../../notebooks/baseline-mmluprox-ru-chat-5shot-fp8-560.ipynb)
содержит весь код и неизменный manifest. Старый FP8 notebook сохранён.
До загрузки весов проверяются SHA данных, 70 demonstrations и 560 prompts;
задачи каждого предмета загружаются один раз. Параметры модели прежние.

Результаты, исходники, логи сессий и учёт времени пишутся в
`MyDrive/LaPA/fp8-chat5-560-ea2fa680`. Каждый завершённый ответ проверяется
по manifest; после перезапуска такие ответы не генерируются повторно.
JSON результата публикуется переименованием временного файла; незавершённая
попытка не считается completed. Ошибка checksum останавливает продолжение.
Одновременный запуск двух сред не поддерживается.

Бюджет общий для продолжений. Живой процесс обновляет время каждые
20 секунд, на сессию резервируются 120 секунд для экспорта. При аварии
без записи о завершении весь интервал до восстановления консервативно
считается израсходованным; долгое отсутствие может исчерпать лимит.
Это ограничение учтённого времени notebook, не измерение счёта Colab.
После финального экспорта пользователь отключает GPU.

ZIP с результатами скачивается и копируется в Drive с проверкой SHA.
Подключение Drive требует входа пользователя; доступны
[официальные инструкции Colab](https://colab.research.google.com/notebooks/io.ipynb).
Наличие файлов в смонтированном Drive не даёт гарантии мгновенной облачной
синхронизации при аварии: продолжение принимает только целые сохранённые
результаты. Drive и GPU проверяются пользователем в Colab, локальные
проверки покрывают вычислительную логику и файловую систему компьютера.

## Повторное ревью перед длинным запуском

Watchdog дополнен отдельным таймером: остановка группы процесса по
deadline не зависит от зависшей записи heartbeat или чтения лога Drive.
Таймер отменяется при завершении команды. Это защита процесса вычислений,
не автоматическое отключение выделенной Colab VM.

Готовность определяется final_manifest.json, а не наличием summary.json.
Он связывает итог и файлы всех завершённых ответов; перед пропуском
установки/оценки проверяются hashes. Незавершённое создание итогового
manifest восстанавливается из ответов без повторной генерации.

Локально воспроизведено зависание записи состояния: прежний watchdog
не останавливал процесс до возврата записи, новый таймер останавливает.
Проверено также прерывание второго вопроса после успешного первого:
нет полной метрики, затем продолжается только незавершённый вопрос.
Ограничения доступности среды и Drive описаны в
[официальном FAQ Colab](https://research.google.com/colaboratory/faq.html).
Фиксированная выборка, параметры модели и критерии оценки не менялись.
''',
    'evaluate.py': r'''"""Evaluate the frozen sample and summarize only a complete run."""

import hashlib
import json
import time
from pathlib import Path
from uuid import uuid4

import numpy as np
from baseline import finished_attempt, freeze_json, run_eval, verify_run


def evaluate(lm, manager, tasks, results, selection, prompts, deadline):
    mode = 'chat_5shot_direct'
    for number, item in enumerate(selection['items'], 1):
        completed = finished_attempt(results, 'baseline', mode, item)
        if completed is not None:
            continue
        if time.monotonic() >= deadline - 300:
            raise TimeoutError('Budget exhausted; completed answers saved')
        print(f"{number}/{len(selection['items'])}: {item['subject']}",
              flush=True)
        directory = Path(results) / 'baseline' / mode / (
            item['subject'] + '_' + str(item['index'])
        )
        name = 'attempt_' + uuid4().hex[:12]
        lm.expected_input_hash = prompts[str(item['question_id'])]
        try:
            run_eval(
                lm, manager, directory, name,
                'mmlu_prox_ru_chat_' + item['subject'],
                item['index'], item['sha256'],
                prepared_task=tasks[item['subject']],
            )
        finally:
            lm.expected_input_hash = None
        if not verify_run(directory / name):
            raise RuntimeError('Saved answer verification failed')


def summarize(results, selection):
    """Use saved harness scores; never grade or remove invalid answers."""
    by_subject = {s: [] for s in selection['strata']}
    checksums = {}
    invalid, capped, inputs, outputs, generation, wall = [], [], [], [], [], []
    for item in selection['items']:
        directory = finished_attempt(
            results, 'baseline', 'chat_5shot_direct', item,
        )
        if directory is None:
            return None
        relative = directory.relative_to(results)
        seal = json.loads((directory / 'manifest.json').read_text())
        for name, digest in seal.items():
            checksums[str(relative / name)] = digest
        for name in ('manifest.json', 'completed.json'):
            checksums[str(relative / name)] = hashlib.sha256(
                (directory / name).read_bytes(),
            ).hexdigest()
        result = json.loads((directory / (directory.name + '.json'))
                            .read_text())
        samples = next(iter(result['samples'].values()))
        if len(samples) != 1:
            raise ValueError('Expected one scored question')
        sample = samples[0]
        score = sample['exact_match']
        if score not in (0, 1):
            raise ValueError('Expected binary exact_match')
        by_subject[item['subject']].append(score)
        raw = [json.loads(line) for line in
               (directory / 'raw_generations.jsonl').read_text().splitlines()]
        if len(raw) != 1:
            raise ValueError('Expected one raw generation')
        raw = raw[0]
        invalid.append(sample['filtered_resps'] == ['[invalid]'])
        capped.append(len(raw['generated_ids']) >= 2048)
        inputs.append(len(raw['input_ids']))
        outputs.append(len(raw['generated_ids']))
        generation.append(raw['generation_seconds'])
        wall.append(json.loads((directory / 'timing.json').read_text())[
            'wall_seconds'])
    rng = np.random.default_rng(20261008)
    bootstrap = np.zeros(10000)
    subjects = {}
    for subject, scores in by_subject.items():
        count = selection['strata'][subject]['selected_count']
        if len(scores) != count:
            raise ValueError('Incomplete stratum')
        values = np.asarray(scores)
        bootstrap += rng.choice(values, (10000, count)).mean(axis=1)
        subjects[subject] = {'questions': count, 'correct': int(sum(scores)),
                             'accuracy': float(values.mean())}
    bootstrap /= len(subjects)
    total = sum(s['eligible_count'] for s in selection['strata'].values())
    summary = {
        'protocol': selection['id'], 'status': 'completed',
        'questions': len(invalid),
        'correct': sum(s['correct'] for s in subjects.values()),
        'macro_accuracy': float(np.mean([s['accuracy'] for s in
                                        subjects.values()])),
        'macro_accuracy_ci95': np.quantile(bootstrap, [0.025, 0.975]).tolist(),
        'eligible_pool_weighted_accuracy': sum(
            subjects[s]['accuracy'] * info['eligible_count'] / total
            for s, info in selection['strata'].items()
        ),
        'subjects': subjects, 'invalid_answers': sum(invalid),
        'outputs_at_token_limit': sum(capped),
        'evaluation_wall_seconds': sum(wall),
        'generation_wall_seconds': sum(generation),
        'input_tokens': sum(inputs), 'output_tokens': sum(outputs),
        'generation_p50_seconds': float(np.median(generation)),
        'generation_p95_seconds': float(np.quantile(generation, 0.95)),
        'note': 'Eligible-pool evaluation, not full MMLU-ProX or speedup',
    }
    summary_path = Path(results) / 'summary.json'
    freeze_json(summary_path, summary)
    checksums['summary.json'] = hashlib.sha256(
        summary_path.read_bytes(),
    ).hexdigest()
    freeze_json(Path(results) / 'final_manifest.json', checksums)
    return summary
''',
    'session.py': r'''"""Account for notebook time across resumptions; not Colab billing."""

import json
import os
import time
from pathlib import Path
from uuid import uuid4

LIMIT_SECONDS = 12 * 3600
RECOVERY_RESERVE = 120


def write_state(path, value):
    """Replace operational state; experiment results remain immutable."""
    path = Path(path)
    temporary = path.with_suffix('.tmp')
    temporary.write_text(json.dumps(value, indent=2), encoding='utf-8')
    os.replace(temporary, path)


class Budget:
    def __init__(self, directory, started):
        directory = Path(directory)
        directory.mkdir(parents=True, exist_ok=True)
        self.started = started
        self.started_at = time.time() - (time.monotonic() - started)
        self.previous = 0.0
        for path in directory.glob('*.json'):
            record = json.loads(path.read_text())
            elapsed = record['elapsed_seconds']
            if not record['closed']:
                if time.time() - record['updated_at'] < RECOVERY_RESERVE:
                    raise RuntimeError('Another session may still be active')
                elapsed = max(elapsed, time.time() - record['started_at'])
                record.update(elapsed_seconds=elapsed, closed=True,
                              charged_seconds=elapsed + RECOVERY_RESERVE,
                              recovered_at=time.time(),
                              recovery_estimated=True)
                write_state(path, record)
            self.previous += elapsed + RECOVERY_RESERVE
        self.deadline = started + LIMIT_SECONDS - self.previous
        self.path = directory / (uuid4().hex + '.json')
        self.closed = False
        self.tick()

    def tick(self, closed=False):
        if self.closed:
            raise RuntimeError('Session closed; restart Python and Run all')
        elapsed = max(0.0, time.monotonic() - self.started)
        write_state(self.path, {
            'elapsed_seconds': elapsed, 'updated_at': time.time(),
            'started_at': self.started_at,
            'charged_seconds': elapsed + RECOVERY_RESERVE,
            'closed': closed, 'prior_charged_seconds': self.previous,
            'recovery_reserve_seconds': RECOVERY_RESERVE,
            'note': 'Notebook wall time, not provider GPU billing',
        })
        self.closed = closed
        return self.deadline - time.monotonic()
''',
    'selection.json': r'''{
  "strata": {
    "biology": {
      "test_count": 717,
      "eligible_count": 622,
      "selected_count": 40
    },
    "business": {
      "test_count": 789,
      "eligible_count": 741,
      "selected_count": 40
    },
    "chemistry": {
      "test_count": 1132,
      "eligible_count": 1073,
      "selected_count": 40
    },
    "computer_science": {
      "test_count": 410,
      "eligible_count": 385,
      "selected_count": 40
    },
    "economics": {
      "test_count": 844,
      "eligible_count": 793,
      "selected_count": 40
    },
    "engineering": {
      "test_count": 969,
      "eligible_count": 916,
      "selected_count": 40
    },
    "health": {
      "test_count": 687,
      "eligible_count": 646,
      "selected_count": 40
    },
    "history": {
      "test_count": 381,
      "eligible_count": 359,
      "selected_count": 40
    },
    "law": {
      "test_count": 959,
      "eligible_count": 906,
      "selected_count": 40
    },
    "math": {
      "test_count": 1351,
      "eligible_count": 1274,
      "selected_count": 40
    },
    "other": {
      "test_count": 924,
      "eligible_count": 875,
      "selected_count": 40
    },
    "philosophy": {
      "test_count": 499,
      "eligible_count": 471,
      "selected_count": 40
    },
    "physics": {
      "test_count": 1299,
      "eligible_count": 1218,
      "selected_count": 40
    },
    "psychology": {
      "test_count": 798,
      "eligible_count": 755,
      "selected_count": 40
    }
  },
  "items": [
    {
      "subject": "biology",
      "index": 320,
      "question_id": 2936,
      "sha256": "0b25fab74fa41d1c4ef73813f8aa2229ee002e923dfa7e9ff735b92c7ce5ddd2",
      "rank_sha256": "0014a3931a94e15532b33c30dedcebbe2c0164a853e3cf9de12a03eead7459c0"
    },
    {
      "subject": "business",
      "index": 49,
      "question_id": 119,
      "sha256": "673c93439dc26eda456990d5621320faa463e0d21cddfd73089a7622743ec8c7",
      "rank_sha256": "008b8eedf6e1aa80ea6292939ff95c97f25ce00ac7307caa17692f2661f5b158"
    },
    {
      "subject": "chemistry",
      "index": 90,
      "question_id": 3423,
      "sha256": "2f682d9ef75ebe335c09517f0cbd68b28ecaafb33284994424bdc8c3e37d8ed8",
      "rank_sha256": "0063424209739cbc6ea4c737263aeeffceeb788a225cadc28126965cf8d5bcf7"
    },
    {
      "subject": "computer_science",
      "index": 320,
      "question_id": 10271,
      "sha256": "3be464ab02ec95971356cea8c88e11d05374380d296bd4e103bcb471069dd6a2",
      "rank_sha256": "01a8f115ab3669b9d210903b4131680b039bfd2453699a8ef6ad5173734b1cb1"
    },
    {
      "subject": "economics",
      "index": 320,
      "question_id": 6777,
      "sha256": "2011f1e7ed606aba32b1a62e16cd8803ad42ecab46a091a11ce05bd7a4a14adc",
      "rank_sha256": "00bcf0ecde377c57643ca95a000e13dfd4966632863283383e7d476c87069ecf"
    },
    {
      "subject": "engineering",
      "index": 897,
      "question_id": 11757,
      "sha256": "ed6915a9e614e17ffe10c9b63050f6933a8c484868e3f41b61ca368f8a91be87",
      "rank_sha256": "0033ef4aa90252267cd281642e09e60348c2804b7db4a7f035cc60d8010b0b5c"
    },
    {
      "subject": "health",
      "index": 145,
      "question_id": 5915,
      "sha256": "eda580f0315fe6973207107998f425dcb56f062d4d2e3de37d3425f194527ed0",
      "rank_sha256": "00190f9a686b2bd4937aa27aa54a855e1b2165cac908e0b81b7338e76964ce4d"
    },
    {
      "subject": "history",
      "index": 252,
      "question_id": 4717,
      "sha256": "3898b5f3c10fe72eebdf64f934577b09981984a182da82a0d5d91d74a75d65b9",
      "rank_sha256": "00687ac0f0ed24a49e7904a5b19a0649f35ebcca9c331b27287d25724e1c1009"
    },
    {
      "subject": "law",
      "index": 94,
      "question_id": 953,
      "sha256": "6c0d614b3dbd9c2d0e71e3278fa1e3b40eff64d1d65a0bcba1cbfc5f1d86fdaa",
      "rank_sha256": "000634ec3ba11cceccd6c527664b5c762aa9f2635a722ec95f6423b4e418bb54"
    },
    {
      "subject": "math",
      "index": 615,
      "question_id": 7916,
      "sha256": "42fe12962c422588822b4fdf43218e391c769980fe3a66345aee3c43f448ed73",
      "rank_sha256": "0031f7cdae24b6fdeb26a6d82ba112938894f23d393f56b31f3b7c471b894ed8"
    },
    {
      "subject": "other",
      "index": 551,
      "question_id": 5397,
      "sha256": "d51c74aae70df065b76d060bb7f579a1249e732b5e63190d5875a94a8e5fed87",
      "rank_sha256": "003ec1530456f932ab37bd4931113ea193f57afc329a99eb62730f171c7a55db"
    },
    {
      "subject": "philosophy",
      "index": 14,
      "question_id": 10375,
      "sha256": "d96b625851ae60455ef81c69c18f120e739ed0f36a7bec01b5cbe150cf6dd17d",
      "rank_sha256": "011f167c90693407d642464a784da8087691b33223244466a581fb337e3d54c2"
    },
    {
      "subject": "physics",
      "index": 1,
      "question_id": 8653,
      "sha256": "e2d32258d07321a2fff97f75a11cbb736cf22920263e3a91e390bb1fd1253528",
      "rank_sha256": "006c230684bab23b8c14abb62c9ed25fbd5f6ef722387454f26965b9465c2397"
    },
    {
      "subject": "psychology",
      "index": 387,
      "question_id": 2205,
      "sha256": "fe4d68a78f711a1e3ce6be98c3d35b48bd77aecd9721bd07072aa6996da77fde",
      "rank_sha256": "006af2592d3821fd1fefa1449dc2a278d792d42a3e81743787e520e167646fdb"
    },
    {
      "subject": "biology",
      "index": 332,
      "question_id": 2948,
      "sha256": "d406a6523f2535636dd93971ba9bae2084bb2f971a5f18bd7261aba41879784e",
      "rank_sha256": "004414cf669817519f8f785299f96c6f6788551495cfc83ee5f7f604951cbc71"
    },
    {
      "subject": "business",
      "index": 256,
      "question_id": 326,
      "sha256": "44253618f9517cb05f1629066dabe6b4f070bedcb466d1977a557a2d95bfee5b",
      "rank_sha256": "00b2eb4d2189ee575a12642b3604a7c4e6076e0833e48da07a1e3757d3b112b7"
    },
    {
      "subject": "chemistry",
      "index": 228,
      "question_id": 3561,
      "sha256": "d303458899d83041e4b1d0b97f8126734e07580182b6e6b111ab9c0ef94a8b1f",
      "rank_sha256": "008ae32c29e165d90062bf0d0713b9af63c8421354fd8fb6a30c650d1597f0ee"
    },
    {
      "subject": "computer_science",
      "index": 382,
      "question_id": 10333,
      "sha256": "63550fda47f931cbe98789cdcf2b4ad6f90570a42c42e29efd4664c974b06ecb",
      "rank_sha256": "01ecd3f70484efac577164188268ccbdcd28ed1e5d29636ea407d54fbaa8d20d"
    },
    {
      "subject": "economics",
      "index": 16,
      "question_id": 6473,
      "sha256": "3a87f1cffa40f8e702d6b8b14719ede47724c7fbdc516b98965c89d1f76a3d1d",
      "rank_sha256": "00ee5af492b4a37a1b68075c9d1db7b020592bf6d33b5a223bbbaf0fde10eaaf"
    },
    {
      "subject": "engineering",
      "index": 953,
      "question_id": 11813,
      "sha256": "cc9e90a997a735aaec1c7fdaa93fe21d65ae70445442e4cb1913f54511a66487",
      "rank_sha256": "003d088347cadb7215ea302cf21ccd8ecc6e04d9827ae28ea5ccda14143886d6"
    },
    {
      "subject": "health",
      "index": 646,
      "question_id": 6416,
      "sha256": "3f479081e551544bcbc475bcac9e15bcb419568e090aa363b261a5dbfde1d7ac",
      "rank_sha256": "00b82ac30cb3a61fc523b645ce2d497b105605a0c578a89e21dec13b1e570077"
    },
    {
      "subject": "history",
      "index": 332,
      "question_id": 4797,
      "sha256": "8fb358315d6741f5e99722afafe548e7354eb127e834aca5ec72edf8302d3549",
      "rank_sha256": "00914258adc6f0f383c91ad651f8653f76c8cb34bc2a5f769c0a16dfd7f20478"
    },
    {
      "subject": "law",
      "index": 520,
      "question_id": 1379,
      "sha256": "754c028b86599d5591eef15aacb09b58dc0b77ff1f8ed2820a7d3feac971ac07",
      "rank_sha256": "0021dc5fe6d7a81021c23e0ac95d077edb9ccef641d6b51f035f20c797e92626"
    },
    {
      "subject": "math",
      "index": 1083,
      "question_id": 8384,
      "sha256": "6df208bbd253f62099fbb1ccd79eecfbebdd046122f1d43d74a51e83560b2c89",
      "rank_sha256": "0053ab7486e424241668eae2525124af8ab9183fc9eb8943962809c1f1a7085a"
    },
    {
      "subject": "other",
      "index": 617,
      "question_id": 5463,
      "sha256": "6c262d8a5cd532f8dab6b70da065cb1e8193d98d9160684100d8707a7cc80552",
      "rank_sha256": "005c8d71e65ec1078ad90f1e1056a06cd4a2b6659af1314f4f3297c994d35f32"
    },
    {
      "subject": "philosophy",
      "index": 278,
      "question_id": 10639,
      "sha256": "725f1196d53fdaf1229452acdb25122ac2b1176d70a4baeb79bac0121cbd2066",
      "rank_sha256": "037181afc7fa7137bd1f746edc53fe6c0c108a7898b82e9fec96dd5b3542eafe"
    },
    {
      "subject": "physics",
      "index": 80,
      "question_id": 8732,
      "sha256": "f3c109939527e01f5ca8037de2e26cf9e3f24c18a34553c9ee0325557ba2f799",
      "rank_sha256": "007d4a827019fc9bf49ca018825eb6a9a2dcf1e885bcf3787fe96bab01fbad88"
    },
    {
      "subject": "psychology",
      "index": 678,
      "question_id": 2496,
      "sha256": "9a6e26a57c217bdbb2bcd7a4894fdd53022ebd8d0a47262f3b9717b6086f61a6",
      "rank_sha256": "00a4d86896b61e2de43468666839c2040eac670df41d8d17a4f3289d2c5f5b3c"
    },
    {
      "subject": "biology",
      "index": 445,
      "question_id": 3061,
      "sha256": "5cb2cb7309688769c5aca85fe535e6e47f3f4a8f570863039e6a5279eaaf6f5f",
      "rank_sha256": "01b54c4cbdf6e6cf22bdbe57e69523a0d3548543d68817d0d25dcb58ee725677"
    },
    {
      "subject": "business",
      "index": 755,
      "question_id": 825,
      "sha256": "f4757ada738abea08425496d1310ca0efd68ef7673cf973af1270e65e62c6127",
      "rank_sha256": "01381c78330f728fa21e2faf8e4b0cde28d1d8257e1db52473a034b227860bbd"
    },
    {
      "subject": "chemistry",
      "index": 282,
      "question_id": 3615,
      "sha256": "98e1a63904d038c26b625b2a6ca5bb504f363f7fe74c3ad9513b15dc1182dd20",
      "rank_sha256": "00dfcb60d1d5a49c6634f94de63b4a4174db3d4e9f23ab6e47dae6b8574df1df"
    },
    {
      "subject": "computer_science",
      "index": 233,
      "question_id": 10184,
      "sha256": "904686418e90cbe4f58f2b411f2892737ea250651ad65eaaa1a0f1805d6e9120",
      "rank_sha256": "029de0a179cb7f02f7588cb663a1a063f65b7f24609b7dbb17d6e726549d5eb9"
    },
    {
      "subject": "economics",
      "index": 728,
      "question_id": 7185,
      "sha256": "70445dbf10fd6e9ca0e059cdd27d599f6e93485af8a20a6ad57e53ab027055c2",
      "rank_sha256": "00fe638cb1dd1fae8d6398d6f4356c8619d02b212d7adc9b26fb1f80e2a0476d"
    },
    {
      "subject": "engineering",
      "index": 744,
      "question_id": 11604,
      "sha256": "d2a6ed1b0abe93ca70803979280d6c753ad75d71ca68893eedcf8bbe56fc8f4f",
      "rank_sha256": "00a6320ff47ba16247b822ff1cd2139da3ab237013b74782d8f60cfb96e67033"
    },
    {
      "subject": "health",
      "index": 141,
      "question_id": 5911,
      "sha256": "b179abf0c85d49f43508c5b33cd893e1a82c06ed6b69d8cf3dbebc29ff27bb49",
      "rank_sha256": "03417016cd8d5d99c6667156241a59b948aa6fde008d9d72b2af8bf12541a273"
    },
    {
      "subject": "history",
      "index": 246,
      "question_id": 4711,
      "sha256": "14ea55cfeada295abc39853166156c830bdae0484cf41aeeb170f28962fcf18a",
      "rank_sha256": "0271feb6b5a8d8905422949ef38b28e6910b34e4467bde880abbf0cf75aba070"
    },
    {
      "subject": "law",
      "index": 702,
      "question_id": 1561,
      "sha256": "cab557c3b89f4c715cbfe73dc780c7d454350e35d2df59a3b573a4568078cf35",
      "rank_sha256": "0063e80e060f8bbd042bc30d87e0d66cf64daea951903f9f3cf4f9a5e1bdfe3f"
    },
    {
      "subject": "math",
      "index": 173,
      "question_id": 7474,
      "sha256": "28a203436040a24a9c4debf69d521b37b1e213f9ef903ac4c50f4b844efc2dbd",
      "rank_sha256": "0056fc3f9888b5ea04a66e26036ea9fa77751fba94d49fa1b7ea57f0809815c6"
    },
    {
      "subject": "other",
      "index": 729,
      "question_id": 5575,
      "sha256": "4a73945cab29bdb504bd91e0dea286901dff501674110699fb0283163b46679c",
      "rank_sha256": "006c43a096ed4e8efd55a2147dd9cf2945e0c677294801e3bc9627c756bdd8ab"
    },
    {
      "subject": "philosophy",
      "index": 275,
      "question_id": 10636,
      "sha256": "4ebeb2cd809a98d01e9c935809609808e10f1ca46b31dd771995f9ca80adb43b",
      "rank_sha256": "03cbac49edfb550840a3006395a85e61fddc533dc184cf030ee5d0f72863bc67"
    },
    {
      "subject": "physics",
      "index": 1073,
      "question_id": 9725,
      "sha256": "0a5895d81d4a6f3917793bfca97080264405dbb1a551fafe16cb094639e4e311",
      "rank_sha256": "00e0fbf2ccb467285822b261b1179281a2a46d81130866c51f1bdadaf3a5585f"
    },
    {
      "subject": "psychology",
      "index": 460,
      "question_id": 2278,
      "sha256": "109bcf3021de87e8d1d9f348b7198f03cbbf180b75182e4f91bbf1af096ed4b4",
      "rank_sha256": "00e24b8657168c0e79c1b609d6c3c0c0ad3868a7321baabc9aa7ef9f6122b5e7"
    },
    {
      "subject": "biology",
      "index": 160,
      "question_id": 2776,
      "sha256": "65f63d9e777354781b95790e7d681e135d00ee7c640321c818f059b30dde9a58",
      "rank_sha256": "01d6a6ab0d0ea6246f8e3cf6a2bb6f46d0de7edf51a2d6c80c2e5e00b81d3f33"
    },
    {
      "subject": "business",
      "index": 377,
      "question_id": 447,
      "sha256": "92d53f1c345149c405e4697336d4a55bdb4353ad641c668bdfc708ade91d9d03",
      "rank_sha256": "01414cbf92d007379579d49b25a8fda1f96ee6ceff75097fb51fdc6d41dd81fb"
    },
    {
      "subject": "chemistry",
      "index": 175,
      "question_id": 3508,
      "sha256": "f3a9952ec62cc465d1f57d25920be339702975a1c077e5e8a8c10b8284ced1ea",
      "rank_sha256": "01234e17a006afc646b0b5fdaab54895bb70922eefda371e29567885eaaad27e"
    },
    {
      "subject": "computer_science",
      "index": 288,
      "question_id": 10239,
      "sha256": "14b447ad869d7826ff2a7a6c3335d6e8f2d92f7798640d87a5300051f03d2483",
      "rank_sha256": "02c218c2540abf46b8ba8bc3fb8193d06e43c178a8b6d3ff6a3fbc3e3cb93516"
    },
    {
      "subject": "economics",
      "index": 102,
      "question_id": 6559,
      "sha256": "8f734d10e7625017fb0926aa0255e0310b84e2d68e5b9144ea8ed36e144e8bf5",
      "rank_sha256": "01346889545f623be3fb43491824985e88d447c029fec8cd3164f0429b85984a"
    },
    {
      "subject": "engineering",
      "index": 771,
      "question_id": 11631,
      "sha256": "3b32019c6d88eec0c4a41b0f4c78945668973ce265acea5fedc0aaaef4fc30f7",
      "rank_sha256": "010fc0a1293949848d1c9b19938a836c610e1fcb7a9740d03057286fa96f7809"
    },
    {
      "subject": "health",
      "index": 339,
      "question_id": 6109,
      "sha256": "61c0bf81030db09fdbca6886859927162c3c8c0de26f52d362581d4527be2a69",
      "rank_sha256": "0397a6c95db5831b292f5f3bd873940a54092bafbc85a2691e94bc78089b1217"
    },
    {
      "subject": "history",
      "index": 229,
      "question_id": 4694,
      "sha256": "3194b6084170ddf06d82126f21af60af5a0a03b3622fab010c2250a7849a151f",
      "rank_sha256": "027bf9d708ed8935e437d1cfa762a7ae6ce01ddc1eb6be60c5ceed2a2a323ab1"
    },
    {
      "subject": "law",
      "index": 834,
      "question_id": 1693,
      "sha256": "3c0c79a93c33a6fd372a903e5b47f36a26b30ddfd587e3cbec8832261e602d3a",
      "rank_sha256": "00d64c5840435770279b7fb396186fab79b905845a3cad288290151b769e5d7e"
    },
    {
      "subject": "math",
      "index": 1181,
      "question_id": 8482,
      "sha256": "0c374b9f81b2a8d19d147e5661a26b947b5b7f92cf72f3a377cab994b1c31718",
      "rank_sha256": "0058e5f21ce818fb38bff0dc36803978fbd15f3f8f7cba664090673c8b91d6f1"
    },
    {
      "subject": "other",
      "index": 182,
      "question_id": 5028,
      "sha256": "ea901947d0ffe8110c691e6e7c69fafd7b498f3cb9b09033df7ee9182fc2377b",
      "rank_sha256": "0122e8566af1372492964c37d13c1233af677ca57c2287a7c8cf74321863d65a"
    },
    {
      "subject": "philosophy",
      "index": 101,
      "question_id": 10462,
      "sha256": "5d9c53bf1c4339c7a5649639f3fa583801ce915cb6f6e345f336da8be60a962b",
      "rank_sha256": "0426ce5a872dee8c3c84d65aa8e7ff861cda430f1c193b605850038d8f226b36"
    },
    {
      "subject": "physics",
      "index": 437,
      "question_id": 9089,
      "sha256": "7c67f4ac9adc4cf0f5a37a468da63442bb6c2d6ee90fc333ef67d4e2a0774671",
      "rank_sha256": "00fdb3c23ea416324deca9a0e5b75eea195fdce5216b79fc3ba31aa1050d7575"
    },
    {
      "subject": "psychology",
      "index": 383,
      "question_id": 2201,
      "sha256": "867c56afc0e9717378ec997947e5062a962804d6646ea689fc7337219e6b5396",
      "rank_sha256": "01e00e5e51d2b51804175d2989799f99b1af6b7334f4cea9793103b72ec8e057"
    },
    {
      "subject": "biology",
      "index": 83,
      "question_id": 2699,
      "sha256": "e68f95ad516e7c653f5f07b19d0126662487bed427945bbc960579c2ac21cf8f",
      "rank_sha256": "022900c6964d2ba0ffd1ae68a07477a5dc51afbb190e99d55d5505beffc4fe80"
    },
    {
      "subject": "business",
      "index": 117,
      "question_id": 187,
      "sha256": "67e94fd877373e69c91e5af0a376ace7f801f924ae85e823db1b7005ec1a46cf",
      "rank_sha256": "01588ae25a870fe0691380f0ac8e608bf263b5bc8d3444548acb9f8d87e37d38"
    },
    {
      "subject": "chemistry",
      "index": 538,
      "question_id": 3871,
      "sha256": "cd2ffb002dc1fcc3e0a62153b45b5b0b5640e779ea2d0f67acf00ee2cb50f7d9",
      "rank_sha256": "015ebe85975ccd36f4f51d5328883d259c15410a0990b924703be90e7178da5f"
    },
    {
      "subject": "computer_science",
      "index": 151,
      "question_id": 10102,
      "sha256": "9c10d0abce7acb96d48e69933100331e176490d62b2b0e0b559087848e1cd79d",
      "rank_sha256": "02dbcf3b09f30db7b6623bbc636417d304993b6af1efc44f676e3b8790bb2537"
    },
    {
      "subject": "economics",
      "index": 703,
      "question_id": 7160,
      "sha256": "30a7ef7f970f9a14dceb8feed4953730750663aa4513312a4e3b2e3a24223a10",
      "rank_sha256": "01577662cc49553ba3ef1238ee9cb07d90e689e7a7705fbdc4befdf56f6b6eba"
    },
    {
      "subject": "engineering",
      "index": 553,
      "question_id": 11413,
      "sha256": "b7b4629abf17683ec3c1446601899bbf99685b9c0d4ae035314435fec2ba59a1",
      "rank_sha256": "019377b237c27da6d241f2d731baeb1b1fff56b0d236452953072e92b6574f02"
    },
    {
      "subject": "health",
      "index": 25,
      "question_id": 5795,
      "sha256": "b7b96f6ec60af8ca9719828c1e1a7d11452348eaf99e49426e5ea187fa70d013",
      "rank_sha256": "03eae4dbe47806e076d549fb76d2a87c97b8d07b91cd767caa8e73ddcee86db8"
    },
    {
      "subject": "history",
      "index": 46,
      "question_id": 4511,
      "sha256": "f2fedfe0d53af2dd1b2c09dd144fed1d6e0c5aafc8c2d2befde9b8d15635e684",
      "rank_sha256": "0314d42e80d59d06154dd3ebb0ffb9e0865e7f723b75d63473aae100201a2202"
    },
    {
      "subject": "law",
      "index": 252,
      "question_id": 1111,
      "sha256": "13ba5128ce2c866adb292849178e23505f3570e576c2f7377927e315541b3b5b",
      "rank_sha256": "00f48de4d0be03303f0c415dacca867647e005782c3be84818ffeadd93d22a3a"
    },
    {
      "subject": "math",
      "index": 887,
      "question_id": 8188,
      "sha256": "28429b5288396579940960b51c173a8ce04880155216518b90af5b2b231b2bff",
      "rank_sha256": "007991e4bcf0d61bd682e40054cd721965307c612e1097be092d703944efe2d8"
    },
    {
      "subject": "other",
      "index": 49,
      "question_id": 4895,
      "sha256": "8406962b07739e6a644bf9554ae749eb858ff2fa21391a2c32f2a914aa33ea9e",
      "rank_sha256": "01271a8687d5384aed5a7edc6526c5edc8c104b45ff5cd670b197d7e36cfb8e5"
    },
    {
      "subject": "philosophy",
      "index": 216,
      "question_id": 10577,
      "sha256": "e274d8ee04bab9b2be829e95a05104fa6c833920cecb052b827e6e675f0dafdc",
      "rank_sha256": "044fd0bc42795dad85f5ad2944a3add515c75da83a5b9d31e56e7951ff473164"
    },
    {
      "subject": "physics",
      "index": 224,
      "question_id": 8876,
      "sha256": "c925f80e5a60dcb7e61b1a56f10fe88fd10503dc23b665ed51c59566c90921c1",
      "rank_sha256": "013c99d34a7b3ba81b97243c6645114eb51f550ab3268b99790757e7e67ec699"
    },
    {
      "subject": "psychology",
      "index": 721,
      "question_id": 2539,
      "sha256": "10160cfc6f233725fc84546db78c8f67736a00064001c7a271eb632dbb8ca047",
      "rank_sha256": "0211bbb631ce5ee9b94ef81718370b51b6a249230989f73c948f27d8e85aeec2"
    },
    {
      "subject": "biology",
      "index": 660,
      "question_id": 3276,
      "sha256": "31d4cc918b55ebe7d6247b12ca4e8cb6a0bbcbb6544f815c8c60c54d529ad05d",
      "rank_sha256": "0269735cb73b34dc3c4711dcfbf72e63501c0149d532a145721ab8876bba7ccb"
    },
    {
      "subject": "business",
      "index": 674,
      "question_id": 744,
      "sha256": "604b1567a78a4d385ecf5e6fe32fb71a9598d625cf3438ae748e0cfe70c02653",
      "rank_sha256": "0167bd777557d5450c0c681f553f9f52f138838d45ad16df68793d40bbffb8b4"
    },
    {
      "subject": "chemistry",
      "index": 512,
      "question_id": 3845,
      "sha256": "23f34bb1c04082dddba68f6e572add285c539a7351016740a1c747f8275f197e",
      "rank_sha256": "01edb020dc9e6dcfe328cc3ab09093bd9f351f792aa5d20bceb975ec066cee1d"
    },
    {
      "subject": "computer_science",
      "index": 70,
      "question_id": 10021,
      "sha256": "49363c08baf1715f0f26d54afaf00c8f69524264b94aba1caad251e748c2328d",
      "rank_sha256": "0339fd0ac00613de8264d82265371513a787030fb24cdc94f1a17d693f43e0cd"
    },
    {
      "subject": "economics",
      "index": 155,
      "question_id": 6612,
      "sha256": "a1aa2eb8ba72a1b11fc27ab3b657be58736169d9cc54c325b6f5a197d75bb0ea",
      "rank_sha256": "0158fc197d0f762eb98487d7c02fe351636fb51931f49627a49bae10dda4b3dd"
    },
    {
      "subject": "engineering",
      "index": 457,
      "question_id": 11317,
      "sha256": "cef23c00576c43b78f53940ab400f19fb37e88ce9fd7634102b5715ee60e2c01",
      "rank_sha256": "01c30620a58a4cf3d3794ca9def42c526ecb63de741fc25e6a9368da6c93cfe8"
    },
    {
      "subject": "health",
      "index": 613,
      "question_id": 6383,
      "sha256": "48ba3b987c3c67e35eb710dbfb8b8c82f38bdb300317e8574b35af2957428966",
      "rank_sha256": "040992f40e36e64a8117dad7fdb97210a820e291ab26a0b44d65639060801b69"
    },
    {
      "subject": "history",
      "index": 354,
      "question_id": 4819,
      "sha256": "540cc66550d8de58b51f59402f413a54503b7e4e0d428ae752ab67c0b72a2b68",
      "rank_sha256": "03aed59587a02dba406281a80200273f9afa5745615ad311dcfcc0445a52b21e"
    },
    {
      "subject": "law",
      "index": 949,
      "question_id": 1808,
      "sha256": "4e87121c1e9230431fc72ff03cfe94198a202d5cd11f480028cb3ed10a4a0e53",
      "rank_sha256": "01e5f10a3012847e54b18fa54f1f1e5b81b96032796c5dbeff18cb0fa4278b6d"
    },
    {
      "subject": "math",
      "index": 776,
      "question_id": 8077,
      "sha256": "cc9d5990065d8e28a13cacd4e57b7088eef096d9aab8cc27d510fe2cd60a175a",
      "rank_sha256": "00a40185c3bf17d40b0c2567e9a06c093ed08e0d68f6fcb833451e144875befb"
    },
    {
      "subject": "other",
      "index": 172,
      "question_id": 5018,
      "sha256": "13074c13031ae68448a7e249aefdc69c4efea7d67c2ea4c6b17a4550cc600727",
      "rank_sha256": "013caad02d681cd96d166b02244b24c9be541151a71fb252ff51d47a2709a4a7"
    },
    {
      "subject": "philosophy",
      "index": 35,
      "question_id": 10396,
      "sha256": "a2bf0f21eb3f79f6490a04ed375f176fc95a69e04276b2c43bd371843eea743a",
      "rank_sha256": "04898653eafda9b44704492a0f3861b85c7f6d06412f444fd68c42e91baaad11"
    },
    {
      "subject": "physics",
      "index": 39,
      "question_id": 8691,
      "sha256": "737df1731029a6c069085da1cb354dd9bad5229ea5b41a3628cb9f174527eeb0",
      "rank_sha256": "015445ae735c087a424fa16771009ba2f48ba16ba730da2f493f5b001d764f0e"
    },
    {
      "subject": "psychology",
      "index": 263,
      "question_id": 2081,
      "sha256": "e8a2645fdf23430c06eb3a538bf0654b921c91e2aeb034885847c3d50a4a20e2",
      "rank_sha256": "021e678dc793f4f31c0c58aedfa751a74aa984282af0d8ad9702002c6a8c4b89"
    },
    {
      "subject": "biology",
      "index": 88,
      "question_id": 2704,
      "sha256": "88dbeff2ca3ca56406d01f3d6ab3c539072d265e6783cf6c5a5adb7541cf0053",
      "rank_sha256": "026e1a0aa1da49700010bbb52337572391ce5df7fb0a3caad84d84bd2b91faaa"
    },
    {
      "subject": "business",
      "index": 33,
      "question_id": 103,
      "sha256": "a035b800421eeaf6bd5c6dd96941a0ba4e95ec80a9bc850138be56cc81668857",
      "rank_sha256": "0184fe8f7ec19e42dc4fb1a07d40cf5d1b694cee225209517f35272a8e60dff3"
    },
    {
      "subject": "chemistry",
      "index": 913,
      "question_id": 4246,
      "sha256": "81717287223cb11bb92218eba00f97ebbbfe227e29139410ddb28e6678be7f38",
      "rank_sha256": "01fd8734e53a380c19fc4118341b01c22ad9ccfb273ab29982ff6991803d9d30"
    },
    {
      "subject": "computer_science",
      "index": 311,
      "question_id": 10262,
      "sha256": "737c128a4d4e76a380b4acbba3aabf84d4ea8829335738a2dcd006e35655dd58",
      "rank_sha256": "04565521a1e7d18dd6e280ed175cf9e15680aa00b59e00c268d60fe434796662"
    },
    {
      "subject": "economics",
      "index": 564,
      "question_id": 7021,
      "sha256": "d3561c845b308a8bb1be21f7bebea0bf5a355882ac940fe91f4a27baad1bfe5b",
      "rank_sha256": "0173ac6335abcf499ebc008c1b003d7257ce15fb5824f03b27dbff3da72e2b1e"
    },
    {
      "subject": "engineering",
      "index": 716,
      "question_id": 11576,
      "sha256": "e718a890f64dcd6ee9d99f4d1c46fc184a66eb964a65dd9bdd329b2a966c2e91",
      "rank_sha256": "022970649804b8c60b5415939d2698a41a74341ac29bb93fe70951c97625602e"
    },
    {
      "subject": "health",
      "index": 548,
      "question_id": 6318,
      "sha256": "dfc3f4e8283d7a62baa2e7eba2b5d0bed5e3f37ca80c0e8706d93dddcdefc218",
      "rank_sha256": "042fb852f8893932b20d53921868fcd64212a9a4c8b6fdd08d4a72e6cec038ad"
    },
    {
      "subject": "history",
      "index": 61,
      "question_id": 4526,
      "sha256": "c29ab254ccbdf5e29fb1257c11c64258ae90b5e3e057a078e3e1a9b5c76a496e",
      "rank_sha256": "04dc94b97c06c26803a38a894de45f782935932117fd63086e7525d64d1682a8"
    },
    {
      "subject": "law",
      "index": 466,
      "question_id": 1325,
      "sha256": "d28572f21c5192cd3a4255342f8e1faed103213052411456b0af7cd78f918865",
      "rank_sha256": "020554cb973a59142b426a3260f99d4aaacbb92879326e8f036dbfe450f1e8ca"
    },
    {
      "subject": "math",
      "index": 597,
      "question_id": 7898,
      "sha256": "c4ccce8a7e42191cdce450445511f1ba49338e15c7884660d0622a57574a42c7",
      "rank_sha256": "00ad455ed193c6d417affc3467d9b0337e5a9cae0985fa06cae701c902306e76"
    },
    {
      "subject": "other",
      "index": 796,
      "question_id": 5642,
      "sha256": "b258867187d072bfb018f59351f8cd2f9752c024acb08c8a9582d6aa59978425",
      "rank_sha256": "01cd28428bb61df4bd876e233e8cdb0f0197192cea728a59ddd81ad5f15efc4a"
    },
    {
      "subject": "philosophy",
      "index": 382,
      "question_id": 10743,
      "sha256": "ca0d3b9003e3fb894b2b022d72e5fa25cede98df7b12f2919dd0d9e7d8c07395",
      "rank_sha256": "04dbb8e4746b756600b961a296bdc5c72af5ad0d6d76765c94e7680288443643"
    },
    {
      "subject": "physics",
      "index": 959,
      "question_id": 9611,
      "sha256": "cdae8fdaaf35986e9fa5287df8034a07c28e62bdcee7eac3568591cb1907238a",
      "rank_sha256": "0198ea6e6529f95dd597ceaa3bd8c56b58a49764363d2ded466e0e600556800d"
    },
    {
      "subject": "psychology",
      "index": 195,
      "question_id": 2013,
      "sha256": "13e4d1711ba55bcb80c7bde74125c005045b05661a532daa412bcc37af67036b",
      "rank_sha256": "032897ef68b6d5012b9dda8f3d1aa572b0608213c885b785fb4e3af20343b674"
    },
    {
      "subject": "biology",
      "index": 25,
      "question_id": 2641,
      "sha256": "afae9df0f5f20cb6d4bbd42a5acc58e27d1b083ecc2cdb986d0d144f08953f24",
      "rank_sha256": "02f842de59139a0981578a850f45affa5a8f4187553e026732d56c4193f080f9"
    },
    {
      "subject": "business",
      "index": 78,
      "question_id": 148,
      "sha256": "c375c06f7029abc83a8cf41c9ee1837522938045bc95c3bc3ba4a3201efbe2d8",
      "rank_sha256": "0246fba0629c4701e532e18911494794996184d7e46818753aa9e00d8dd2ed95"
    },
    {
      "subject": "chemistry",
      "index": 482,
      "question_id": 3815,
      "sha256": "cc44acd12a6e3dc777171f528614603f746c011c25eb3c515e73cb39e0e13431",
      "rank_sha256": "02402aa7b9f6bbab6a0093a68045ce3d2a4dddfed7e2a367a21dc3d9c1d8f5f1"
    },
    {
      "subject": "computer_science",
      "index": 2,
      "question_id": 9953,
      "sha256": "156cb55645cef4950cd6564ebcdf4c8065472fde10370f26062ef1b47c0f9e3d",
      "rank_sha256": "0501f2d78ab5ac20697caebfcb6607ece46244b7b6507052f6ff0e308fab5333"
    },
    {
      "subject": "economics",
      "index": 434,
      "question_id": 6891,
      "sha256": "cdcc1b5990826a9ca7f2ff933e0815cf1a9c8e0d1a27788a36c2d56796ec2352",
      "rank_sha256": "01beeccb3db9a53d302c73369319de4dc21f397a827d663b2c660e404317bc30"
    },
    {
      "subject": "engineering",
      "index": 885,
      "question_id": 11745,
      "sha256": "55da7df0857d676d7eb57980ae982a1b6dc8ff096111b489b554c59d3d0aa87c",
      "rank_sha256": "022a5dbff12c5be45bad6e53fbe3a1895200ae217b48199954feaa6fddf2f599"
    },
    {
      "subject": "health",
      "index": 482,
      "question_id": 6252,
      "sha256": "354e265052e01e0eafd0cc91795978bffbac850435833cdcac4c1336ce2f2fed",
      "rank_sha256": "04dbad98c891831783ffb291ac29c1451a4bbd0addcacdf34e9f0812720031ad"
    },
    {
      "subject": "history",
      "index": 256,
      "question_id": 4721,
      "sha256": "7436d5949aa677dd8f8a09a90c6b92c2477355b97dd254697e308b236f1d7b57",
      "rank_sha256": "05b3aa8ab80f923176fbd27a23402e1872b38b2a1cc3acb427a5c45bbf9d67d6"
    },
    {
      "subject": "law",
      "index": 342,
      "question_id": 1201,
      "sha256": "1f16cc4d06e0b9ee223bb1c707ebbca01e914f16363cf1bc53e4a9f6a82351b6",
      "rank_sha256": "023cb9885bbb87bb603a275b83f3c99c4f969f730612100e4cecfab30184eb2b"
    },
    {
      "subject": "math",
      "index": 1061,
      "question_id": 8362,
      "sha256": "bc9218ab24337325316407e0c10648876060fdaa2d4273a1ba551b067afd99f9",
      "rank_sha256": "01929ca7681a4b85a766b53436310c78304c4fa064b2bd9001d21937708247a6"
    },
    {
      "subject": "other",
      "index": 793,
      "question_id": 5639,
      "sha256": "09f984a5d1067c768653d05c35d602307b5ecafb64731f3cf1cc1be63dba93d0",
      "rank_sha256": "01d4756daa69595cfdc5729570438d81a0ebe440f41418cfc48502c2d5bcb914"
    },
    {
      "subject": "philosophy",
      "index": 155,
      "question_id": 10516,
      "sha256": "592cfc2aff054658a7ce37b335188c3fd311717ef2ef6d7b6e45e11d61b143b9",
      "rank_sha256": "05c29ed5e85e9540e0180edcb07cef059e0a35ee0365bad596b36984b6195501"
    },
    {
      "subject": "physics",
      "index": 587,
      "question_id": 9239,
      "sha256": "8906980cdb9b56b9621731b213d8ae3193c5f5ae57bd6441de4f05e87c8bd91b",
      "rank_sha256": "01aefe7b233205af5e6a75c111099f3ea05b38c270e95b3118eea7d1109cc26f"
    },
    {
      "subject": "psychology",
      "index": 551,
      "question_id": 2369,
      "sha256": "d33cae5f899adca3d49bc4fbdb2a0dc945873d6b8843bc56964ca7b4461f096a",
      "rank_sha256": "0375cf086c1da8d45971b1dd09f212d0e0e737520bf6a88670f1bf239245d183"
    },
    {
      "subject": "biology",
      "index": 257,
      "question_id": 2873,
      "sha256": "835c80a5f2b71be514b4f2d1664ba5684c5657e79dd1d2bed33ee4490a50f261",
      "rank_sha256": "04a3ada8986f1e4835b1c1d7ec5ab78976f148315ba0a31c7abe82b25eb3567b"
    },
    {
      "subject": "business",
      "index": 236,
      "question_id": 306,
      "sha256": "7721429442b849ae5ef13ab131fc62dba19562d7159deb56c4b64f728ae0d55c",
      "rank_sha256": "025038e80188acaedc43843e758477c8a0fa9e1a8637f3b2807243b3517bd5ab"
    },
    {
      "subject": "chemistry",
      "index": 315,
      "question_id": 3648,
      "sha256": "9473f23b5508e1b36fa7b5a4be946918251905c9cb77d38f4834358bd0cdba24",
      "rank_sha256": "02ae6ff8f18a407163b393831de6e84cb89ff6428e47acf4007f224b7b63ea08"
    },
    {
      "subject": "computer_science",
      "index": 216,
      "question_id": 10167,
      "sha256": "cfe9915dfa84fc02bec0d70865247eea4be87d305ad293e05a32caebb0f6254a",
      "rank_sha256": "05282a1e2a5d95a7fd280684b1b8ba8e9b6caefb19ae2313e3f58f2222e9f26e"
    },
    {
      "subject": "economics",
      "index": 525,
      "question_id": 6982,
      "sha256": "b5c97134b77d35cb907ec31b37b734ea8315276bd01e692acdeff645ee125600",
      "rank_sha256": "02b2cb692760e98012247ce0a852785c171119b841eb898cfd905bb431b025e3"
    },
    {
      "subject": "engineering",
      "index": 404,
      "question_id": 11264,
      "sha256": "2565dad4092f44bd041d47f8033e83dc87d051999b3dc93a445059b2bbe37388",
      "rank_sha256": "022ae4fcf711771b55dc1d9d9431b29d639a6407cc5aaf30a6b0e958e54808a1"
    },
    {
      "subject": "health",
      "index": 408,
      "question_id": 6178,
      "sha256": "60b60aed9a1a6a87e2c88ff5d8c65debbddd0fcac242373d42f71fed5528bc79",
      "rank_sha256": "04dda9f5d3998626eb3a4855db30c813c82351849fa461cb5519c9cd20579d86"
    },
    {
      "subject": "history",
      "index": 360,
      "question_id": 4825,
      "sha256": "7a53ef569ee7362b70171b49bba1c89cae2bec96022db69fa42237775473b39d",
      "rank_sha256": "0660974c905524fd4e78b38f853970d353fe26086e0a2e635c9abfd72610c40c"
    },
    {
      "subject": "law",
      "index": 223,
      "question_id": 1082,
      "sha256": "26d53ba6953857b504ecb3be77b31b7c6522cf9107c302d4d964a733240b2e1e",
      "rank_sha256": "02af91a33d9c5e6f006c165918c967e6cf60f033d8f109dd1dda3be8c8bf25ed"
    },
    {
      "subject": "math",
      "index": 642,
      "question_id": 7943,
      "sha256": "45b2d3cc987748fa7bb7f6f2ec9d095b4efabcc8538c1072c7b3eca2091c7efa",
      "rank_sha256": "01cd8b6cac32d45e7bb9adc619e1b573c716153c7a497b68d17e721284ed50cb"
    },
    {
      "subject": "other",
      "index": 294,
      "question_id": 5140,
      "sha256": "6bac66fe88f1af12615f7e346818fec1477bcafaa7dcc54fea3f8721e9863263",
      "rank_sha256": "01e31228e9294d5637199fb1ee08fbb5e4cf9f9e7eb262ac77ef2fdb1f29b1fe"
    },
    {
      "subject": "philosophy",
      "index": 219,
      "question_id": 10580,
      "sha256": "f1a7e5a4fca1190169133220daa29725f9a951187fba516e88bc532e328f2bd4",
      "rank_sha256": "05e2cb057b80da0c16038f7ace4a227945ec26fa5c0b2e9650679f188f679158"
    },
    {
      "subject": "physics",
      "index": 924,
      "question_id": 9576,
      "sha256": "2a139068a7b0bfad232f2efe279e08c72d99cf82ff69d97d02b009fdc8175ffa",
      "rank_sha256": "01b3d0b634a60a6dec3d3b0fcb5a75b9dd72825eb8bd43091f31423278e563c6"
    },
    {
      "subject": "psychology",
      "index": 600,
      "question_id": 2418,
      "sha256": "e6e0ff01d09fcfbf637282d4fdfc751a62e78a5ce52681733eabb6eb59de6946",
      "rank_sha256": "040c1cc669b71b33f00028e75b497a361cbbb808fcc78ace8f4c599feb21c2ce"
    },
    {
      "subject": "biology",
      "index": 166,
      "question_id": 2782,
      "sha256": "d02ac6faa69a4854d687924a7f5ebd7f0754f98a44c4284ace4a829a1c4c3fb6",
      "rank_sha256": "056c00d3ad432237e7cbd8f230e5cd609e051572e8f9b9692b502ead573a0f11"
    },
    {
      "subject": "business",
      "index": 399,
      "question_id": 469,
      "sha256": "d66848f4185bbc31e0301fb44fe5d759844d514ef57ab2ab7e9ec138bb3fcee4",
      "rank_sha256": "027b8dbd06c9ea03a62f0d2029258f53dab00461a482d024cf6ad543ebd8f72d"
    },
    {
      "subject": "chemistry",
      "index": 255,
      "question_id": 3588,
      "sha256": "861bffa7e75308dbde19609f414bf0e2fdc6a157e9109db4ce3dc5aac33aef56",
      "rank_sha256": "02db7a998450584a3f9db896b96d5e2ec7428291c5bca67775cf4afc3cfe0882"
    },
    {
      "subject": "computer_science",
      "index": 225,
      "question_id": 10176,
      "sha256": "01e6b08bb0ebd80be6268f6f2dd648e9e491c543933578b2d7c180a08191e83d",
      "rank_sha256": "05740e56cbdc341e31a9d18d81b30af0bdc9287eda3818a400d414f216cdc165"
    },
    {
      "subject": "economics",
      "index": 408,
      "question_id": 6865,
      "sha256": "e9a1aa275cc2a9217457ad84a3baec1f4a9c51bd83a5d1ad87e03dbec2837836",
      "rank_sha256": "02b7d7385b8869f01aa313837c2ca7a6689ae30a3f17a95ba33f14089b6f1ac3"
    },
    {
      "subject": "engineering",
      "index": 282,
      "question_id": 11142,
      "sha256": "47caf33608df5e62cad3b2e7988f52d2639b629aaa46bfbf6992a463fde666f6",
      "rank_sha256": "0269826172766e91bd85184d304021777c111ad8f193cc8620e9834986b98cd7"
    },
    {
      "subject": "health",
      "index": 586,
      "question_id": 6356,
      "sha256": "2a4fe14fdfdc58bc35c08528a9838be093d2933da2f3782d655f503223bb9204",
      "rank_sha256": "04df9ba6b3678ab2813960d44c2152999c17f96766ada33f13196e88e72c44f8"
    },
    {
      "subject": "history",
      "index": 371,
      "question_id": 4836,
      "sha256": "74f184032fb0114178f80291c06504b1c74f8e4f9597ed766ca6235b7c6ad836",
      "rank_sha256": "06f92df2d7402468bfe15ead936cb90809c62645d59b8823da84fab3b13f4a25"
    },
    {
      "subject": "law",
      "index": 898,
      "question_id": 1757,
      "sha256": "0d0e31b928da5115a2a6f956d14b5aab9708be29f88fdab4257fe6f1633c3d9a",
      "rank_sha256": "0358bb67e7a8f453dab567d0a5477ce2b074fb5ec0676a1831951351d7c59c54"
    },
    {
      "subject": "math",
      "index": 275,
      "question_id": 7576,
      "sha256": "598947d4f97ef5dbbce71f6c0c7063e64831d24013eff3d387b240fb17334ce1",
      "rank_sha256": "01f46d0e20df2cc45c65a77596213f7064e97de8b42444a2a5c9b4c709b7e5df"
    },
    {
      "subject": "other",
      "index": 353,
      "question_id": 5199,
      "sha256": "ee8adddfca0a87e1b528c634ecaee133d10c29fe911cb7119a11c2b78d38e748",
      "rank_sha256": "02f1390eaf4cdcf026a56f21719d85cff301d2a5334668ba8f7e8ae0c4afde21"
    },
    {
      "subject": "philosophy",
      "index": 55,
      "question_id": 10416,
      "sha256": "378a464adf34e136445e57d0fe292b38bb53cb11f0ef33227ba8bbce038af5f2",
      "rank_sha256": "0685e18d138bb32213b08df2e09526b1a90bbc5123d5e16bdb3d45b333ddf02a"
    },
    {
      "subject": "physics",
      "index": 268,
      "question_id": 8920,
      "sha256": "d8112ce681f1bf852fdcc59c403a80f0050a4719a2b1bf23825d09db459cd020",
      "rank_sha256": "0264a2cfa4ff6d339a5eff0307c0a1de51d76fba49422eeddfea908e055036b9"
    },
    {
      "subject": "psychology",
      "index": 414,
      "question_id": 2232,
      "sha256": "144d7d44cd6ae6aa2923738e2db7ecfeba7237e98e2d8f03e03220d3da820491",
      "rank_sha256": "052cab2f19c0e886207d89bcec7ffc9a6b9345c32870c098b6c51b9a76f63c67"
    },
    {
      "subject": "biology",
      "index": 337,
      "question_id": 2953,
      "sha256": "758b3b315c89cb87cbddffa68863b84cdcb5874fdfcae3dd38add5e40c78e43e",
      "rank_sha256": "05908478d00a1ed25bc951faab86563d76c4e380cf2e88ae63a77ef5272d0e8d"
    },
    {
      "subject": "business",
      "index": 472,
      "question_id": 542,
      "sha256": "57c846db562cb3208b300b070c2a4d781c5a2275132b94efebb1ec4b88005d21",
      "rank_sha256": "03460048e491b7111c31fafcbe12dcf2ad5d540697f225150650791c223e257c"
    },
    {
      "subject": "chemistry",
      "index": 376,
      "question_id": 3709,
      "sha256": "b006ca8c1886f853e1269c67cd5a91806463d57ea24fcf1343b79a5371bdb04a",
      "rank_sha256": "02fea08a41c89b5a930e652b8222c8ba06a9a967d915aff88f3d934a2ff1e95c"
    },
    {
      "subject": "computer_science",
      "index": 264,
      "question_id": 10215,
      "sha256": "fb1e26f7d4048a09d6497238f84da256a1c00329b558ce4f39960fb3d53f1903",
      "rank_sha256": "05a174e53917910bec30671dd73480452c1db891c9ee3372c850f9b376d24323"
    },
    {
      "subject": "economics",
      "index": 809,
      "question_id": 7266,
      "sha256": "028e69aeaf64ec223c83158a2646f8e36f5573e14973aff02b429bd084e1695f",
      "rank_sha256": "02ee4a8d1f632dc4f40c7a4c1d99f9b5f640d9d61582894b3c97b9c8e27a30af"
    },
    {
      "subject": "engineering",
      "index": 618,
      "question_id": 11478,
      "sha256": "069db700bc512813fc482b60cd6e8d3a9676481113e69666eb0826dedc7b48bc",
      "rank_sha256": "0271b5afe11331d9c6e6515d84520e232b564ec1de6ad8a77f9b5a83e8c8c399"
    },
    {
      "subject": "health",
      "index": 638,
      "question_id": 6408,
      "sha256": "ac8b581d67e8de6f344ccfafb5cedf71312115bdd2c8e52a5288e8c62277ba5c",
      "rank_sha256": "05a6eb9a35fad5da4d3c57f456570368ec2f93058dce4a88e05f9b755c24336a"
    },
    {
      "subject": "history",
      "index": 66,
      "question_id": 4531,
      "sha256": "979d419ee15422ae42e949ab884c00e50e7081adcb67bd9cc09e3a5127d959a0",
      "rank_sha256": "071b6611c77e3e67085fe59139a85975188ec13a53adc999432272dc1416350c"
    },
    {
      "subject": "law",
      "index": 395,
      "question_id": 1254,
      "sha256": "0dbf3db18979d045c2fc9c9f36304615872996fc04e14568421f7d0593c9b4c4",
      "rank_sha256": "038d088968aed742854596039b9f992a62e3c4ce1f2a24ecd33f9d698acadbe9"
    },
    {
      "subject": "math",
      "index": 888,
      "question_id": 8189,
      "sha256": "29173bcdd06e85030f5862c7994022eb08ce1f61002cbbe590c9fdacd9141b69",
      "rank_sha256": "0211d9421b8ee92cd1ab3a0058b0c95444bad0a4530de3d93f99ddde1afef65c"
    },
    {
      "subject": "other",
      "index": 226,
      "question_id": 5072,
      "sha256": "5910c0b76969dc8e27a359cc3c1ff723a528a9eeb19dc6960e72cee9926126eb",
      "rank_sha256": "0310b9c62c4fb661f8c650e2249c2ce38e1466c619baa15a4bb7b4d3e128754d"
    },
    {
      "subject": "philosophy",
      "index": 437,
      "question_id": 10798,
      "sha256": "9a48e8035de94e2e47cab4a06bec28b01b3d03d5cc24fb4f8c773cfba0177b52",
      "rank_sha256": "07bc6132894ce6f462e7584e1c27406f6c548846f15d4f0197371ea39a91e945"
    },
    {
      "subject": "physics",
      "index": 376,
      "question_id": 9028,
      "sha256": "e43d96582235ff312da2f1d866fa1d957c4012adc652b7ba5141a1ec63a1321b",
      "rank_sha256": "0289620ae237c3d56c03464095aeaab23e89189e6f74a71043666a9a16669f7c"
    },
    {
      "subject": "psychology",
      "index": 741,
      "question_id": 2559,
      "sha256": "8f2430d92558a0b735eabf597165408a9392a4b1bfb15d18227662d57abb97c9",
      "rank_sha256": "05346d273a60e700e1a9ae8cc73e24a65d28b14941f18ed9f5bbb342b4893efe"
    },
    {
      "subject": "biology",
      "index": 301,
      "question_id": 2917,
      "sha256": "160771eede43a795347b75287c5ae5b96eea96604f2b0cd48cba0ecdf0b7c3d8",
      "rank_sha256": "05a7fa64e0348cfa03012f1ae3b0d2ee68198c7837744707b1cc67ebb5b91eb7"
    },
    {
      "subject": "business",
      "index": 97,
      "question_id": 167,
      "sha256": "9ab25beeb56f95bc0e65d49a2b63b9c5a784fe9f4bd30a5e19f7d150d50dbac2",
      "rank_sha256": "03c1699bf9ab73df0c107b0b238c75b6f61881143ca137dd92b7fd98e5934da0"
    },
    {
      "subject": "chemistry",
      "index": 1027,
      "question_id": 4360,
      "sha256": "455ecd2ea81e3d336f5a701c0993eb670006e64748c558d6221f32e312135c2e",
      "rank_sha256": "031b3e1f8fb1fc8f9aff3079f25c370cb244ffccab6590995083f4d43ce47844"
    },
    {
      "subject": "computer_science",
      "index": 138,
      "question_id": 10089,
      "sha256": "8864e3f04774cd2e473773b9ac697db74c49344518f1e6a23997fd7e942d4eef",
      "rank_sha256": "05b649af83abfeb9a962ef8dc64d0a20bbbecdcd82d1b153e8556dbf9faf4d5d"
    },
    {
      "subject": "economics",
      "index": 10,
      "question_id": 6467,
      "sha256": "d5fdb737fc0931497fb66b5c45ff663e053bbdd3acd815f322a6bcfc75f0ecdf",
      "rank_sha256": "03412aeb025491fd8609f8879ed4b98c13ba71b7be3140b2c17a50f01e9036a5"
    },
    {
      "subject": "engineering",
      "index": 711,
      "question_id": 11571,
      "sha256": "3a6b3f7d6c278b8deea4c532108e2a8614ea92f8841fcb35214fd2f7dc823a25",
      "rank_sha256": "0309668100ae3f7f9ebac712d78d6aefbe4ef6967d0ce498ddc8894438173401"
    },
    {
      "subject": "health",
      "index": 458,
      "question_id": 6228,
      "sha256": "c55f95fad6bc9108dddfd1a596f195c4754f41e7c90e1dfd4186d265d71068de",
      "rank_sha256": "05c6f1da3da8778551742c80658e13f16d288bc67af7678ab79a29700cdd802d"
    },
    {
      "subject": "history",
      "index": 22,
      "question_id": 4487,
      "sha256": "d21925d67a343cd5fe1469d5e3e153f11ad53f4a1cfdbba2c42c6fbc70720d55",
      "rank_sha256": "07e290288def06deff429b5f5629c40dbe55a8ef6b4f063dbb5ccc7780451eae"
    },
    {
      "subject": "law",
      "index": 181,
      "question_id": 1040,
      "sha256": "160e683e5478ec2bed02a8bde1bc92d64610986a2d5f09e8c9ad0025838b9ca9",
      "rank_sha256": "03edc58a86bf7a8d922b715be07be85b142ceca66a7d0f1f54b4eec9dcfc9d05"
    },
    {
      "subject": "math",
      "index": 1098,
      "question_id": 8399,
      "sha256": "d5edbc3bf9236785070a4e4bc163031d84adfd0bf1e845f17926b14a9d234a33",
      "rank_sha256": "0241057cb1dbc57a8ae29fbdd264e2897d5112d7c0d3da4ec95f18829ba5a2a3"
    },
    {
      "subject": "other",
      "index": 710,
      "question_id": 5556,
      "sha256": "68e543369404ee13631b8e6e3831db3f4ac0bafe89ea23935beab3bda5557476",
      "rank_sha256": "03347695857669a418eff8eab9dd1f6e29b4a1c5290c7e4ed9ac45c05822c056"
    },
    {
      "subject": "philosophy",
      "index": 401,
      "question_id": 10762,
      "sha256": "c853cf44d8e4c412a1f8c8df345502ab09e2eaf03b1dd7bc04ee10e53f03157a",
      "rank_sha256": "08473726950019b8053c31680386e3b99643153b829357aae429b01f4ffd4926"
    },
    {
      "subject": "physics",
      "index": 1067,
      "question_id": 9719,
      "sha256": "9106cbe2a91215c08253c8d8e5b9a80b530eef1802e0fddcc68d6e5109fa8ec1",
      "rank_sha256": "02a105f94743f73e1c54fa42d67829383973120326fe91e86975d5e02fcd6818"
    },
    {
      "subject": "psychology",
      "index": 543,
      "question_id": 2361,
      "sha256": "3aad58e3e5d78390ad54c458b42907e354683720d47652f9b9d796530690fe6c",
      "rank_sha256": "0554a848a08eb5b832b37a2a47bdb5a66fdaac1197603cc723c0491a623965d1"
    },
    {
      "subject": "biology",
      "index": 302,
      "question_id": 2918,
      "sha256": "33ec53973c36d739a898abecd4dbe121e558c5b93e3b24711385df02e2e8696e",
      "rank_sha256": "0744c8c1aaad8192ecebd3a07ca97a7b4cedc066325ed356eb4250e1072f01a0"
    },
    {
      "subject": "business",
      "index": 759,
      "question_id": 829,
      "sha256": "45ab796eb6c75d2e028b6a4fe9575919db555f8675be1ed0dd77915699abda36",
      "rank_sha256": "03c8b98743c8a35fbbef25368d57a82b2029bc82c32492e286752f1d9e0bcc62"
    },
    {
      "subject": "chemistry",
      "index": 242,
      "question_id": 3575,
      "sha256": "ef89187b2f90260ef6c4c5d02d8e8ca259f9c983a538a8992f49e3be5194bd4e",
      "rank_sha256": "03320df323061e729c83ca90a1e965290efc0b60f66e699ae6d33dea2fefb66a"
    },
    {
      "subject": "computer_science",
      "index": 91,
      "question_id": 10042,
      "sha256": "32ba7373ff2545182e870962bbd7f05ae7987804b75ef4ffdb0880d817b3c662",
      "rank_sha256": "09702cabf630a1106d23ca377c62a62051f2e9efa62a4ce3f6e20883658124ac"
    },
    {
      "subject": "economics",
      "index": 776,
      "question_id": 7233,
      "sha256": "2ee117a3844748b62c5fc85c8ce7ed5152a4964524e8357b958c1bd3d30bef8e",
      "rank_sha256": "03c04d1fa47b7ccf3b77db7a0afa59235b9312f82c2622209ebce23e2c92dff5"
    },
    {
      "subject": "engineering",
      "index": 89,
      "question_id": 10949,
      "sha256": "6d5fba2a93287c268c0b8d128fc002985ea3310bbe7d47cac70d3b34fb18ae45",
      "rank_sha256": "03ba6173d566bd3548aa5a90977f79fd856b5b2995e07690bc94fc9f7296c44c"
    },
    {
      "subject": "health",
      "index": 635,
      "question_id": 6405,
      "sha256": "81471f79fbdbacde252023dfd5817834a0f71f706c3f2a21698136e19b89ade4",
      "rank_sha256": "0629275a62184962c35034ab3dfe1424710254999791e6e54c92c3da374f9438"
    },
    {
      "subject": "history",
      "index": 14,
      "question_id": 4479,
      "sha256": "c0e1e9225bc26df1804e9f8b75cbabe2ca3b3e898c84cbc5d6c654ee18067168",
      "rank_sha256": "082f6db9114fe84157e7350530e457cd3ae64d89f7cc13d7568082376524fe6d"
    },
    {
      "subject": "law",
      "index": 257,
      "question_id": 1116,
      "sha256": "5f8712a0d6d469fa3222754346c4f90b84b8b476ac457610f4873ff8871b3b5a",
      "rank_sha256": "0421236726e6ad53f0c353c7babf629ed21550e6b4fbb4813706c8a90f8f2e5a"
    },
    {
      "subject": "math",
      "index": 733,
      "question_id": 8034,
      "sha256": "f1f23701b9d16bc247573c5e49c0efde2d3ba476980606e3829c9792e577227a",
      "rank_sha256": "024281a6ab43f4f67822d8cd4444fd89adb102a3904cfed8e4dfb4664ad0fe02"
    },
    {
      "subject": "other",
      "index": 614,
      "question_id": 5460,
      "sha256": "1ade964545e49e36ffa253822cf0ebc144602861a087c775f45cc30276c8b6ba",
      "rank_sha256": "0338daadd575b5eac94244744315b98bfde34e623a70f5d6b7d341be0fb21577"
    },
    {
      "subject": "philosophy",
      "index": 497,
      "question_id": 10858,
      "sha256": "467eb01db086fb58bbf11196705e9c8916cc65fe7a6462d19756c5dbfb2932aa",
      "rank_sha256": "08af02348684d3aa261ecddf8bdca2bd638d12688f9fdb254c1b7a43e64377bf"
    },
    {
      "subject": "physics",
      "index": 710,
      "question_id": 9362,
      "sha256": "10ef74abf7f149db45f8b4aab806c2dead91162abcfba90e720d078514f0a313",
      "rank_sha256": "02b1a645e72c3269b1be76636f054643c82861374776ca1776ee1f0c0615d5f9"
    },
    {
      "subject": "psychology",
      "index": 357,
      "question_id": 2175,
      "sha256": "710297f48ecbba5c84a9366c0bcde135fddc8185393fef05a9c9f452428b15ab",
      "rank_sha256": "05a6f34c639d5fa2d2dc848650522bdd061e3eebe500e139e4b4702dfd52b071"
    },
    {
      "subject": "biology",
      "index": 125,
      "question_id": 2741,
      "sha256": "a65b257a6bda56bd5347a041965da1c5f5a9e3efaa61f5249736d9608d4319ea",
      "rank_sha256": "077e470c4db2bd2677fdc0f2c9b63d60c477aaa78665fff31c5fbe765ad8fb28"
    },
    {
      "subject": "business",
      "index": 98,
      "question_id": 168,
      "sha256": "074583d7f71187fb773c43b835d3f71f714d225079f9132e3f0af9b85cb2de17",
      "rank_sha256": "03d89b8c0b980fc9f1d5caaf73df2ed0e61596b9b8665a5a4a32eafa80130b51"
    },
    {
      "subject": "chemistry",
      "index": 54,
      "question_id": 3387,
      "sha256": "fd4438224f7b0dd109712db84a34f08b5275715e1bfc69efa93c5f9e4b342185",
      "rank_sha256": "0336b5dc70deb02c25a759c3828cb4ee68faf09eab31959da12ad3cafd5e5f0e"
    },
    {
      "subject": "computer_science",
      "index": 368,
      "question_id": 10319,
      "sha256": "dc2e8debbad724581dfa86212cf675ca09b19e04813001095681237c772ececa",
      "rank_sha256": "0991ad826ee8f4e58a47cbf00a356b5094e0eff333622aa4d2d5a35998e2e888"
    },
    {
      "subject": "economics",
      "index": 568,
      "question_id": 7025,
      "sha256": "2d90b03db1f9bb6023f470939adcf2bdfa0f688f57a27929b68a56734b4f6486",
      "rank_sha256": "03f27e60f4aa38547bfe428a550086878d4887ced12f230e1bdce95820cd3f5e"
    },
    {
      "subject": "engineering",
      "index": 224,
      "question_id": 11084,
      "sha256": "e9dc9c0ecb038a18a1dbb9b9b43eba52f4fbb5952d57378b25ebfcd1e3b0dd59",
      "rank_sha256": "03fb62a73949f58c1f8c8c8f633fa2f831b22523b9605d96b7ddf4eda7163754"
    },
    {
      "subject": "health",
      "index": 47,
      "question_id": 5817,
      "sha256": "b2dfe5fdd68b93d8da5373cdad1f16c103e0c1297547a3b448c9a9c5421640f7",
      "rank_sha256": "06ef47b7b078709b94a12534384b3af1b38dd42d8cff661174da5b9d6fc7cec3"
    },
    {
      "subject": "history",
      "index": 291,
      "question_id": 4756,
      "sha256": "e56d8c7b9cad0407644aba5e7c76333673d575b7a99160821aeb64dc62132dfd",
      "rank_sha256": "08e096e359311b4b3b86dba5bfa157471beb92969f29d723a51e39576eca6491"
    },
    {
      "subject": "law",
      "index": 323,
      "question_id": 1182,
      "sha256": "d2666d9f370178e2da388d023770d77ebadc20cfd63cdaa8910f51c54dac3420",
      "rank_sha256": "04686926b7072c355ed71729685581452838dc01fe1d70a069b380344544bfa3"
    },
    {
      "subject": "math",
      "index": 523,
      "question_id": 7824,
      "sha256": "b5a63bea302c197cf8e8b8f02d9eb5e35631bad4794e7a895dce99319f22e067",
      "rank_sha256": "024942126a2f15a4c6bbe897883f64b1ea625bb119ec089b0baa9e7878982006"
    },
    {
      "subject": "other",
      "index": 349,
      "question_id": 5195,
      "sha256": "2b7e9f9f61acf67d23e36dc42dce0e5071a0c8d8a36808c9060f6261d318de89",
      "rank_sha256": "03b8f537ee51dc1411b35269c35dd72b663db4eb403bfe6a4c6dc6ee43bc1167"
    },
    {
      "subject": "philosophy",
      "index": 381,
      "question_id": 10742,
      "sha256": "77fe38755b84c0d0d8e53407b9b79046a1d3e8a962d345fe9c5e1612c9337043",
      "rank_sha256": "0abf0a518a3a2a18b90bf2fe6749a67f6d1d7c5194f9f345ca5c9bebd4f1e9a8"
    },
    {
      "subject": "physics",
      "index": 963,
      "question_id": 9615,
      "sha256": "a639766177f63e907111d88e5e098569e357e6db113bed8d1d054f78ae802d2f",
      "rank_sha256": "02b8c85cef556f0de6d76564e7ad1bd830bb033c23d16edd55194741c0b48878"
    },
    {
      "subject": "psychology",
      "index": 560,
      "question_id": 2378,
      "sha256": "1feb677545b9dce5479d09ddc540049968127a92c4dfccfac9c4313b3dcd04ff",
      "rank_sha256": "05aea2b711d1e8a487e656eddb0c07ae6cd51e3025dbf94971bef5f42096ec61"
    },
    {
      "subject": "biology",
      "index": 527,
      "question_id": 3143,
      "sha256": "7b2e7e0e735342329f1cac283ef1f6ac9100535487388bc96c12672ee25abbff",
      "rank_sha256": "07dde9ae4aab9e8eedb3dad9e7225212b827d244d70d3f7f23b9e91ab6aace3b"
    },
    {
      "subject": "business",
      "index": 335,
      "question_id": 405,
      "sha256": "63ae309fd919437c56f6d692a1a50bffc1bed5a9adb37121a2a927a73a10dbda",
      "rank_sha256": "03e40f130b4572784242c49a269c472c04c97e470a013376e4eb7555fab235bf"
    },
    {
      "subject": "chemistry",
      "index": 316,
      "question_id": 3649,
      "sha256": "653da470c53032945bc79979eafdbbd1a86fa273a414fa27e46fb2eb6cea47e5",
      "rank_sha256": "036ba555cceae514f8c52af7e5e9de002745200285683550096f1191c31c3810"
    },
    {
      "subject": "computer_science",
      "index": 88,
      "question_id": 10039,
      "sha256": "9a7e55719a053ce9e996a93aa0ac896e7ff76b59198cabed6e0898f7e295ffa1",
      "rank_sha256": "0ab2e07cca6dd9cc83bcc8ec7a7bf237034caff4a77cab20780b88d72bddb83d"
    },
    {
      "subject": "economics",
      "index": 435,
      "question_id": 6892,
      "sha256": "05e17d1f20ae0b3e6f32f0ce7347225f3e8f5236cf6f7bdd25cb7bbc78260b34",
      "rank_sha256": "040b7ecf7dbb021a640d3168537ac4ce80003a52ce7428f32b1fb1404f3e93d9"
    },
    {
      "subject": "engineering",
      "index": 606,
      "question_id": 11466,
      "sha256": "a584ed5431812acdf171eaee8818b41a22cf58c1d34f48d8e2c251be3d412020",
      "rank_sha256": "040a0dd17bfb029936b302fbf167e1c1b4bd7a3b2877d07243bd3ed96b63b3a7"
    },
    {
      "subject": "health",
      "index": 65,
      "question_id": 5835,
      "sha256": "b6ed4daa687cbfd3a7d3c507e3a43a0720453b3eb1c2c99cdf429133cfc58151",
      "rank_sha256": "082b4f5478e7d3b26dd4602fc0051c3140eb0f0fedf4119481edb5451ad3cf3e"
    },
    {
      "subject": "history",
      "index": 72,
      "question_id": 4537,
      "sha256": "be60ad09b78dc0b6dae32b2906ecad57e46e4fd34c4df6a74d98a5ae0ad4874b",
      "rank_sha256": "09549389360c7f018e0d57a7c241c73209ea4931092b485dd897e054a3d138e4"
    },
    {
      "subject": "law",
      "index": 78,
      "question_id": 937,
      "sha256": "357f78e022bce8b68c781d94d295862af18136e23c123d75bf6d5972d1767a5c",
      "rank_sha256": "046dfaff2aa2a68c3454a4350c67a96db7690b952ffdb14b508970db4402eb5b"
    },
    {
      "subject": "math",
      "index": 1026,
      "question_id": 8327,
      "sha256": "6007f1bc11980b2e7d5501270304bd8675a093d95a226999e2951b59f499cb2f",
      "rank_sha256": "024c2805a73e46f9fce2cea5590b4b05468539b5bb7bd1d6af2284eefb204706"
    },
    {
      "subject": "other",
      "index": 886,
      "question_id": 5732,
      "sha256": "fd66e938cfaea18b34963ee9eafcefe05ffcf365f500cdfda35996042eca5fe1",
      "rank_sha256": "04b7816d996d68c4019123b75d3b54c5e5b76301bf5395f5db9e65b150bfb673"
    },
    {
      "subject": "philosophy",
      "index": 76,
      "question_id": 10437,
      "sha256": "e1b2cd033e24d88466f5204eb09f46f1d7b16a739d840882c28af586f0c92e2f",
      "rank_sha256": "0b0efd195de5a3f2cd53af196de5568a545ed4551f630183a2938e767cbb6832"
    },
    {
      "subject": "physics",
      "index": 1043,
      "question_id": 9695,
      "sha256": "b015175545b37c35efc8f13b7441f7602114bd58e905867e49ed3a53a6e077ff",
      "rank_sha256": "02bb609e736d9efb98821ecfae6595516aef6997974462ab05502879588fba18"
    },
    {
      "subject": "psychology",
      "index": 517,
      "question_id": 2335,
      "sha256": "4fbd1d2a529a459f3191085f37cf1a800f2199b4924c519247d6641de573d6ac",
      "rank_sha256": "05d12f138072616b076ee53df2ec9305afa0d9bf75e53d2ef608a1d9734bc3a7"
    },
    {
      "subject": "biology",
      "index": 224,
      "question_id": 2840,
      "sha256": "c1359486b39d342f6d70a9255b83bf6ff395d70cdd0928266cab9038f7bab452",
      "rank_sha256": "08c0d1fd211116a955a1c4214a29e14835f7f822e306c788e2abae1689447f44"
    },
    {
      "subject": "business",
      "index": 241,
      "question_id": 311,
      "sha256": "7d71f8260d947be55fa58e5f272c0676370632d5418a078c8403e78ffaaf55dd",
      "rank_sha256": "03fd28e4f6c56778ab6c6906f2c3515f3e0834bc6a77830790390eaf6b83b16f"
    },
    {
      "subject": "chemistry",
      "index": 816,
      "question_id": 4149,
      "sha256": "019c8c872a25d7c4616f04caa0d497afb31fcbcefb80b07b2f9af1423d654e54",
      "rank_sha256": "037492118f1f330a3fa09d7235e363c9d334453c59aa2afe1c8c4683d58cf040"
    },
    {
      "subject": "computer_science",
      "index": 27,
      "question_id": 9978,
      "sha256": "dd7dbd40595d6141b6e451e7f21f2ae4fbba8dc34f11182ccfcc89a1b547fd16",
      "rank_sha256": "0be0c749a3a46c4a48c0cec491ccc5b0305ad3bdf5b40b85c6254b5576d3cc1d"
    },
    {
      "subject": "economics",
      "index": 771,
      "question_id": 7228,
      "sha256": "12aface70c59713c0d9e81cbbf078157a2eda69a83fd08770ee840dbf26aa45c",
      "rank_sha256": "045b43d93a029f79c4d9e943bd630796c3e9dea90ac3d64f2d3bdcb214fff8bd"
    },
    {
      "subject": "engineering",
      "index": 82,
      "question_id": 10942,
      "sha256": "409f00785e9c2a3d3b4d5bf12c816c72ce3ba4a56d0a50ce4ade2a8fd1fc631b",
      "rank_sha256": "0442381e981392cf6d1ea86619048facae2c0970243939478ceff40b9af41dda"
    },
    {
      "subject": "health",
      "index": 283,
      "question_id": 6053,
      "sha256": "b792e42cf94116307429b78d7bca48f579ceeecc71ff2c7fc8ee4195e214c9b5",
      "rank_sha256": "0897a235f8768d97dc15a4f320a8926fe5026bd3c840ba0c097af31aa0c5947c"
    },
    {
      "subject": "history",
      "index": 92,
      "question_id": 4557,
      "sha256": "b741f67ab7f3cc43389488e3e99f020d3b78b297998071aed6a4acb1015377d4",
      "rank_sha256": "09c2dbcc05616c72f7fd917f3ed1d9fb8f3e77e98deb54d034fbe7426e697bf9"
    },
    {
      "subject": "law",
      "index": 423,
      "question_id": 1282,
      "sha256": "cb6f955dfa3f15f3d282bbd6297a6653e2ed59851bc8337a465be3329b302422",
      "rank_sha256": "048791ab83e6580c903fdf837e1d8dc99f0f81c25de6b06ee46683caf67a9dfe"
    },
    {
      "subject": "math",
      "index": 23,
      "question_id": 7324,
      "sha256": "9d157ce45093c83f763ed5f08e45c2d62cafe2089767dc119d118355e77bda4d",
      "rank_sha256": "02569216b4408e0da2419aacfc0ba656593ab7bdf2959ccf701315798351dcbe"
    },
    {
      "subject": "other",
      "index": 827,
      "question_id": 5673,
      "sha256": "17851c3fa8c1e39f90cef39b2cafe687044839ed665ca117ee1701cbf3b46f25",
      "rank_sha256": "04ddf74c162652f38b6f559d045c82f0b06ffa7592ed24821db9455222ab7c1b"
    },
    {
      "subject": "philosophy",
      "index": 476,
      "question_id": 10837,
      "sha256": "47d9e3e54f1afe12170bebb21d812216c77b814837c9ab55d17264990ab7be1f",
      "rank_sha256": "0b573688c089fa874cdba9f3f293ec2fa0b7fc3d34b13dc75c2456ee3a76c73c"
    },
    {
      "subject": "physics",
      "index": 293,
      "question_id": 8945,
      "sha256": "8a9c5b05bc24383aa4253ab4c011a059236885f8d31528c17992f8ffa5393c60",
      "rank_sha256": "030bded84961cb957412b1e7fd1e51972535cb720ab414c692a411814ddaabee"
    },
    {
      "subject": "psychology",
      "index": 91,
      "question_id": 1909,
      "sha256": "f8df938860bf8cdcc49ad4cb62c1dc337ebff33ea0e0915caf89555dbe7b3d94",
      "rank_sha256": "06582ec110e68198fd52e13750eca2160ff579522d60184570b8d262e5fac5a7"
    },
    {
      "subject": "biology",
      "index": 195,
      "question_id": 2811,
      "sha256": "3ca21fb8c93592d7e869ab9b6137b96bec2c808b5f38c0751b0eb0528aeff594",
      "rank_sha256": "08c608458f8086b287a104391039c2e58bb111572f719ba05ee5a3396949766e"
    },
    {
      "subject": "business",
      "index": 189,
      "question_id": 259,
      "sha256": "0427fc14bbb8d5f80209627f84e849073aa1c6f27ab1d11401024377b7bd7524",
      "rank_sha256": "042f51812b818a90fe67db6e9674f22e6478b82455353b21f26aa84a2d98fa5a"
    },
    {
      "subject": "chemistry",
      "index": 440,
      "question_id": 3773,
      "sha256": "0089580a43e60a72b7e526bebe4a6bbacb8f29353ba4fdf5956ac637c3b319d4",
      "rank_sha256": "043434405c946a07434c03476e5f431a9e87f178db0b30335da907b2d39b1914"
    },
    {
      "subject": "computer_science",
      "index": 351,
      "question_id": 10302,
      "sha256": "46d7da6cc02b63adbb64f66e63713de719247e2a4f2cd4cdfeab796b9a0aa40a",
      "rank_sha256": "0c606c43ef4612904ddefab1003f30178b32c090c9395237194f8f060177b1ab"
    },
    {
      "subject": "economics",
      "index": 721,
      "question_id": 7178,
      "sha256": "18189411ae31d5f53e585e9d6378d6e4849e0dc33d8348c8485e88935ca53be2",
      "rank_sha256": "04a6bfeb0006813f59bdfc5f9cbf9574f49be230d5f8a95ee04250198b8b9237"
    },
    {
      "subject": "engineering",
      "index": 80,
      "question_id": 10940,
      "sha256": "26808146e7280525d4fce6e9ce49f0ca75122b4b25625b22d3582e5179185d31",
      "rank_sha256": "047464913b061ae82de6a1205180ac53d979107035288b63622d35f5f40047e6"
    },
    {
      "subject": "health",
      "index": 8,
      "question_id": 5778,
      "sha256": "0245238709f78ba963c48e23a92428645a77967a36d8a6365548a4a11812a781",
      "rank_sha256": "09aa82c672886168e6319f74fe308ceac23279f618ae9c12717332be3c2f4148"
    },
    {
      "subject": "history",
      "index": 313,
      "question_id": 4778,
      "sha256": "be3a05e0f0029b3ad33fd5fa9f9984576187f82ee773e5883b6bb4b64938a1c3",
      "rank_sha256": "0a28b2dcec8a33610a0e22b7ab0f6ff39bdcdab9c53ba96a84f5fc52d08eaa23"
    },
    {
      "subject": "law",
      "index": 856,
      "question_id": 1715,
      "sha256": "916a4c753d5c684a88cf41c45ad3a13b7c6d136902ba68dc48e10516067f9661",
      "rank_sha256": "048822d5bf8dae8ba19fe9db99d13c7fe6891f80a142b0a82d47497ed6141388"
    },
    {
      "subject": "math",
      "index": 16,
      "question_id": 7317,
      "sha256": "5b746047b1bd8e435eb8575d941d8c826c3cbc3fd6073ffbbb2a61b9573fb302",
      "rank_sha256": "025abbf795c6d6a655eff02438183356b7cfaadae517fbd76616dc6fb3416894"
    },
    {
      "subject": "other",
      "index": 173,
      "question_id": 5019,
      "sha256": "7ac5e2c5f4f11ba80efd3b7b95e0a2db5a47fa62fb2251d003f4689fe9e4a327",
      "rank_sha256": "05324603d17f20c990eb325cc0478a847c79ff3c251865a65c84e3344aca95d5"
    },
    {
      "subject": "philosophy",
      "index": 364,
      "question_id": 10725,
      "sha256": "f846e2b7c93a9750c3312cdf27d4757e37dc79dbbe0a9b0177d5670940830f84",
      "rank_sha256": "0b7f2bac1319be4d2eea3bb39fc22d50de3da39ba0d313e77e390543b31e1a81"
    },
    {
      "subject": "physics",
      "index": 1195,
      "question_id": 9847,
      "sha256": "9d59480277aa40d1ad31e8134ce6d245a55a6ad9b0f592dd4f237cc1d0298eae",
      "rank_sha256": "03921be71cdc9989a892ec1dac957d5c350ac261803c0fd7d63b4924ea18647f"
    },
    {
      "subject": "psychology",
      "index": 24,
      "question_id": 1842,
      "sha256": "2e8aea82ae7d46a68ee979799e75569e8854973881cbc13ec7cd06da0fa89c0f",
      "rank_sha256": "068c4023e3419c66da4a59274607c2861ecfabbf928561e46fae25357f0266be"
    },
    {
      "subject": "biology",
      "index": 662,
      "question_id": 3278,
      "sha256": "8d1e4cd9f739d65a2298cd237963dc6310bcb93900dc04537850cc6c4a51bba9",
      "rank_sha256": "0a422845d82e9c4f0f283a2d032bc56de95b1e9a1a6a22763728c741e722f326"
    },
    {
      "subject": "business",
      "index": 68,
      "question_id": 138,
      "sha256": "33edfb6a858ebecee568a254b7dc8785c6bc40ce6d30de70b2ad2fecbf20599b",
      "rank_sha256": "044927214d889bc416226240dc73af03875f444e29f443cdfe199bf615bb11ed"
    },
    {
      "subject": "chemistry",
      "index": 1048,
      "question_id": 4381,
      "sha256": "2857f8664105ab02470db96ea1fa64ca6355a41680cfab43cacf0ed9a75e79b1",
      "rank_sha256": "0443b76087547d268359475c6869f2401817d09e6e8bf71fdbd8dfc840f21f73"
    },
    {
      "subject": "computer_science",
      "index": 364,
      "question_id": 10315,
      "sha256": "ed71bf6041c87f91ba8271ee34fa3f037c70a08d0e43b0c391d0478ddaae4e03",
      "rank_sha256": "0d05f34cde6ff71cfd5b1099eba70d175ecd59e5d93c07cbda276fab16af34fd"
    },
    {
      "subject": "economics",
      "index": 59,
      "question_id": 6516,
      "sha256": "8c4679046388f1a31fa387fcb36d86fb475739c5d02e415e8c5032086e63eed9",
      "rank_sha256": "04e5162049bff9deccdb842cf4c00e3a08f696f800e92793433b7563d682674b"
    },
    {
      "subject": "engineering",
      "index": 5,
      "question_id": 10865,
      "sha256": "635768a1b3dc5648b5f15bd63fa3cc03814c15257aaed816d40c9ae627f967f6",
      "rank_sha256": "0492013b32d58cb5295547be0df05be8b41d0c0704f5d452b021534c583eedb0"
    },
    {
      "subject": "health",
      "index": 664,
      "question_id": 6434,
      "sha256": "3fbc7b185c558d61bb8386dd1ef151da417c0c14a925e6a3464bfcf6eef528ed",
      "rank_sha256": "09f8de24e7e01ccd825467faf4347bca486499e89a22279ff5bda71ee3a9b254"
    },
    {
      "subject": "history",
      "index": 101,
      "question_id": 4566,
      "sha256": "0b9335178efea68388ff6b7ec4328f3fecf5f10608a802f9f52e85ec993748ca",
      "rank_sha256": "0a8e3f4a53165b2a2c7df60442f0d711d2ba0322857612e64c6a10ee233d3485"
    },
    {
      "subject": "law",
      "index": 165,
      "question_id": 1024,
      "sha256": "4f98ec5ff3b6ca97007c1471db14c3f146f3a1daf83d3e817f7d222d2a426213",
      "rank_sha256": "048a04f0bdc70e2d5532f3b801357cf9449782f188804ada98cdfe3b7951c0bc"
    },
    {
      "subject": "math",
      "index": 950,
      "question_id": 8251,
      "sha256": "7ac8cb255867d5f363713271c60e53de80d93fa41fd1993f77734de012ae5944",
      "rank_sha256": "026e82adbf665f47efe994ef2c690599ca170538ec6801e88c17d60b298523e4"
    },
    {
      "subject": "other",
      "index": 434,
      "question_id": 5280,
      "sha256": "b13663ec6397ca64c752bfd24e0ce726327a281608680b49f438b8cd8877223c",
      "rank_sha256": "053c26b922907fe89dd07d434749ce699ddfdf0b08c0c37bc33eff2dc9ef583e"
    },
    {
      "subject": "philosophy",
      "index": 288,
      "question_id": 10649,
      "sha256": "90c9e96aa24fbd84c1e3d87fb871d901314c42988a788719b8e3a1d9e8a15d76",
      "rank_sha256": "0c6dc8c35990632c070288d2c06445129375ec0f9b65c7dc22c687c78816a01b"
    },
    {
      "subject": "physics",
      "index": 597,
      "question_id": 9249,
      "sha256": "f0fe4402742dd04d6410346b1fa19d6d837915d5683e61a254e414dcb3ff9c8f",
      "rank_sha256": "0396497ead690df424641974915ec54f687bf89627e4a34c0dbf7655899fb2b5"
    },
    {
      "subject": "psychology",
      "index": 494,
      "question_id": 2312,
      "sha256": "dc3ee83a6e2ad3bd2911bf1d2243386bbd24a60232ad201bf957fe6bcce19c4f",
      "rank_sha256": "06d7887c69e5b51dd9ba235b8fd442bf7bdfc82ffa5f60d729231138061c82bc"
    },
    {
      "subject": "biology",
      "index": 233,
      "question_id": 2849,
      "sha256": "0a7a07464605ac3b2444be9866437d3ec78ad2334ecb9d65bf3e3ed78b0acc91",
      "rank_sha256": "0a8c4aff4fc6007aa5a0556bb647166f06146c17aa95c9cca5ff09c60472596c"
    },
    {
      "subject": "business",
      "index": 691,
      "question_id": 761,
      "sha256": "810a8f019ecee8469f899c223381e14cdb86b0c329fe9decb51595a8cf73b21b",
      "rank_sha256": "044f73bfb9fdc5b59f6e9604fd99882f89d7c83862f3aefc5c897619669bf39a"
    },
    {
      "subject": "chemistry",
      "index": 893,
      "question_id": 4226,
      "sha256": "b1ffbe56cf84145d07da5d307c614c4b6e5ff5b0f671165018677ebdb40cf857",
      "rank_sha256": "0487d2ca5edd3644167d0738b2f51baa61f978341eda94a1b1c7b823f1de12af"
    },
    {
      "subject": "computer_science",
      "index": 155,
      "question_id": 10106,
      "sha256": "b3cd3019f2b2b116759aaca6bfbcdbf06c65b678806681cfe19e425d3cb50843",
      "rank_sha256": "0f92ba1e39c1d6c2e9059a06742691b2c0d7b761f794f725573b9cfc7bc30bef"
    },
    {
      "subject": "economics",
      "index": 50,
      "question_id": 6507,
      "sha256": "3a5294ffdeea46c5702fb61e09b9485e9f824b31bf564d417b5ea3633c1e4562",
      "rank_sha256": "04f1563fd32d7ebb70c37247160fdeba73d9d400f088c4f4619c2333f0c65613"
    },
    {
      "subject": "engineering",
      "index": 724,
      "question_id": 11584,
      "sha256": "532ca9e6590c73b561c3237467242b109469cf4c89a1580dc994a5eb6c040ca7",
      "rank_sha256": "04a72ad006405bb82ff240ef50269c2798903a6dbf1623f1158415e730cd3b66"
    },
    {
      "subject": "health",
      "index": 147,
      "question_id": 5917,
      "sha256": "c858c4d5b6953d71c9328b7deed9473c412d148a8f2736808dfcffabb9b3ccbf",
      "rank_sha256": "0a6895b0545afd2fbc6be81d245c2a9939aa5525ffdd2349025ae8672c6f1885"
    },
    {
      "subject": "history",
      "index": 88,
      "question_id": 4553,
      "sha256": "bdd55c727fa77278286bfd2b9d786fd216ed6803e209b7817ed6132cf2ac90bb",
      "rank_sha256": "0dbc6bdf59bc3fdd8e298b8b097203931eb76b17277ef86fea12c9e4811d39c4"
    },
    {
      "subject": "law",
      "index": 340,
      "question_id": 1199,
      "sha256": "b80d4c6eb01ab403f40eb8aa8eda9167a67fe5014cc89a66a4cadea6e2f619dc",
      "rank_sha256": "052aec410ce26219ba23b3e734eaefa1cae99aab3c7207bbb94be0af707e0454"
    },
    {
      "subject": "math",
      "index": 84,
      "question_id": 7385,
      "sha256": "6dfc116251fe421fef9d7aa5add87ffa9809ff9cf64864ac54104e06c79a1852",
      "rank_sha256": "027e4fe4454a48f761bc3d3654ad74993d6a356ab1e2aebeeb060a48e6b2e45a"
    },
    {
      "subject": "other",
      "index": 657,
      "question_id": 5503,
      "sha256": "80eb1fd003cec10d0c085c4774062f93733893f8f350276edfbb0864680a8ae6",
      "rank_sha256": "057edc26cc32947618361ccf0ca0bc0db393abfb4df822622ed8764a7097d26f"
    },
    {
      "subject": "philosophy",
      "index": 418,
      "question_id": 10779,
      "sha256": "f6df4eaa143b7620fe21fcb42e2a02c74ce0b86cfd556018491e40e1a3296700",
      "rank_sha256": "0cc3dfa75f2314b3f0d2a7e527419d27046ebd450c2550b0aa9595ef0ce3b7fc"
    },
    {
      "subject": "physics",
      "index": 201,
      "question_id": 8853,
      "sha256": "3586a952419a694b35515db1ecaa87e5207fad0602b5fd8866b45210767a8637",
      "rank_sha256": "0442d16a75c8dc9d0c58363973eb93560726c9e0a19bc3b32900918c5ff9825e"
    },
    {
      "subject": "psychology",
      "index": 778,
      "question_id": 2596,
      "sha256": "32d878e02820dae8d384b88e9d2daa481c45ace86eb12c57eec3b6ec45fa22d4",
      "rank_sha256": "06fd82ab41aa0cca063ed92c6c5798cc5901d33123db3e58ef05d4cdcebabef0"
    },
    {
      "subject": "biology",
      "index": 458,
      "question_id": 3074,
      "sha256": "66b5102b4eba069cca1707201e0d705265714b92bfa0961dfe62e035e94349ad",
      "rank_sha256": "0ae0159fc010d403c255b47e63c81cc8d0e35ca21916ed606f87e60f7b402946"
    },
    {
      "subject": "business",
      "index": 254,
      "question_id": 324,
      "sha256": "863e9fd646f7b1dcaabc86b3b8712632d4ffc6445cdeda734797ba6bbf27c75d",
      "rank_sha256": "045e5a78bfbc6f5cfbb650e3b0db8e95403c918a8b7dbda9fea1012f86009795"
    },
    {
      "subject": "chemistry",
      "index": 973,
      "question_id": 4306,
      "sha256": "6ec3d87bf1a81a717468133f4742371919190d3cf531036eddbb82541996f702",
      "rank_sha256": "0529e091753de56c7ba1164415d4edf8088a5dfcbf57f16ed0ef69b32b0e4f16"
    },
    {
      "subject": "computer_science",
      "index": 317,
      "question_id": 10268,
      "sha256": "b8009effe061475f9d5d72261e60c394187eaa68aa12d132d8efc40ddfa5a4cc",
      "rank_sha256": "0fbae1be01c00f5953475f70f3bd5159d57bf10656e41030f984a1416f84f0c6"
    },
    {
      "subject": "economics",
      "index": 474,
      "question_id": 6931,
      "sha256": "6935032f8e753631976e5d56ce4a25383e96e729607e87b13410a60e651c2420",
      "rank_sha256": "04fa4ce47e969827ae742410b45be624817199270a3b28c7f5b0596eab4fa8a5"
    },
    {
      "subject": "engineering",
      "index": 160,
      "question_id": 11020,
      "sha256": "a97a122d1aad12d049c765073231be512d67287c241a14479c12e94448d74d8c",
      "rank_sha256": "04bb37e9cc67f9e64ea56339208f66ab8f6a39f8dfff4409cde979355f92c7b7"
    },
    {
      "subject": "health",
      "index": 411,
      "question_id": 6181,
      "sha256": "069e7f0732408c6ef51aa9a2a2da5373e2bc35a9722760c4c13c1c9ccb5e3334",
      "rank_sha256": "0a714c1a2cdb93c3ce4624657bb1521140222fa9866370f2e819f816f9cd94ca"
    },
    {
      "subject": "history",
      "index": 2,
      "question_id": 4467,
      "sha256": "36d92a12475c003248fd0e5a0fc9b0ec3d8e5d13ab78d92903aafdf6128d047b",
      "rank_sha256": "0e42d22f0add65ded006c852f34a58f65b2e3c137bb08cd8256164f9bb904888"
    },
    {
      "subject": "law",
      "index": 798,
      "question_id": 1657,
      "sha256": "52fe855f28b3d90dcbad3f4d58d435128edb260306d423c26b57ed44664d805a",
      "rank_sha256": "061aefdd68b84f19c6315f696fadc8299f405db335c3db7966a28a3c21683804"
    },
    {
      "subject": "math",
      "index": 955,
      "question_id": 8256,
      "sha256": "e13bf64fbe586afd19692ecebfe725ecdbbe380df4402ac31e30f93d0ea22371",
      "rank_sha256": "02801158f048839f512e3a10b70347c6a58db11179da48e8e338b4283f7706dd"
    },
    {
      "subject": "other",
      "index": 10,
      "question_id": 4856,
      "sha256": "896e424c4ae70fa9fb79a139f0a635f4a4f2c1b42e8181ddcb995935d76f520a",
      "rank_sha256": "061736d1691ab6b65773baf322d951eb36d695958abd480413b4a69007564f9e"
    },
    {
      "subject": "philosophy",
      "index": 301,
      "question_id": 10662,
      "sha256": "a24e2fae65d5aece01f1628cd2577aff1d4bd4bd835d4cb1e7c09f5579ad11f5",
      "rank_sha256": "0cec002e35fdb7eb31855a896461cbb316e8bea7ed20d57e3be2e7fb381ab574"
    },
    {
      "subject": "physics",
      "index": 359,
      "question_id": 9011,
      "sha256": "60011254c82ec23824579d80cd02b2ea6cf599c11aee98e64629bb3f5f32841c",
      "rank_sha256": "04954975e5b83f13e603d5765ee07096e180cf5c85a8052446aa4d3d8a5a903b"
    },
    {
      "subject": "psychology",
      "index": 707,
      "question_id": 2525,
      "sha256": "c1ed85c272ceb5e84466ef8d1da726b88196bbdeaaa584370c7af07273428a04",
      "rank_sha256": "0777040ba1d2ccc8492da8755b4c834b0328c685a91bf1ba097f0589689b806f"
    },
    {
      "subject": "biology",
      "index": 264,
      "question_id": 2880,
      "sha256": "fd2f64f325f2ded5afe8bf54734a350ccbda75698b25fbe33c485c4e79a3327a",
      "rank_sha256": "0bcdca9622e8a53021162eef36f514b8304d39f8d91e1c2f5d2fa30e12567241"
    },
    {
      "subject": "business",
      "index": 577,
      "question_id": 647,
      "sha256": "8c2d5f1935ff174f61cf3d865f3405579192e6b9952dec771f9e3cdb3baede67",
      "rank_sha256": "04cb893cedd2a78fcf88fa210e10d0e89a1c39bbe8b6a5b8c9204d640d190620"
    },
    {
      "subject": "chemistry",
      "index": 4,
      "question_id": 3337,
      "sha256": "826ba8ccddf51747a686f92ac2430b5958a322c79f6db474d7de7ec7f993b726",
      "rank_sha256": "05b26d06c6c65623b98fd78bb2cd67b687262ba3b216ee1af323a6562620a659"
    },
    {
      "subject": "computer_science",
      "index": 262,
      "question_id": 10213,
      "sha256": "bc9b4d1e557ad1f0d18f60dec490bb854f2fcf2bf21b043955464804e95fa64d",
      "rank_sha256": "0fbc83218003783c62cdfbfdb2757d5e90d40d4983b58008fa55d1032ff38f15"
    },
    {
      "subject": "economics",
      "index": 287,
      "question_id": 6744,
      "sha256": "42d845dd2a8cf76930d2c6807b4676c91d4eafec4ec26d07666aeaf892b78851",
      "rank_sha256": "05046a3ce402dc020cbedee6b6978ba3b6f2721b933895048069b254a7c7edb7"
    },
    {
      "subject": "engineering",
      "index": 659,
      "question_id": 11519,
      "sha256": "b1392de093bcb4a0e68a572f5b38130a2316533dc0ae77f2cc9211cb6d3179ee",
      "rank_sha256": "0549726d391f46ca0d9d6b6144911cf50c5740b1b8890f169743d6e092fb2d5e"
    },
    {
      "subject": "health",
      "index": 506,
      "question_id": 6276,
      "sha256": "8bce8ff83d05e7c17e7838fec276ee1f94b54d08db274bc98ece49078e699280",
      "rank_sha256": "0af667285114b86b2cf60518cf7aef2df6afb8cbfc55b4df20f080a0f7d0bffd"
    },
    {
      "subject": "history",
      "index": 166,
      "question_id": 4631,
      "sha256": "9af47003069d7e163188e374c9249007aab96dc5c47a26a5c5c74aaec693211a",
      "rank_sha256": "0e452ab3bb182580cf91896e5ad85e1c32686061b9e720e43e03f888da25c59f"
    },
    {
      "subject": "law",
      "index": 314,
      "question_id": 1173,
      "sha256": "22ae761cf622b11e592d57d87d6c4d5115f338d8edfdb3f930f8dea7648e3777",
      "rank_sha256": "067298145571b4fbedf9a781aa50f0f9f0220e50704d78576954b8208c22e971"
    },
    {
      "subject": "math",
      "index": 511,
      "question_id": 7812,
      "sha256": "942da03df20ee712937079a4f08e3dd82e3e6a69a60c5652957dda8c830c4fde",
      "rank_sha256": "02c0b4547911493966e9c9802282defbade7e134af658927bec38f4dc6bb591f"
    },
    {
      "subject": "other",
      "index": 698,
      "question_id": 5544,
      "sha256": "d69a4260800e08e150a6731bfa1447295cb42f356a269cc0cb2bb0668772d1f6",
      "rank_sha256": "06c46ea5263faf7f19e9cb2fda43322eeaecf0f7c931d79947f69cacc76c3d3d"
    },
    {
      "subject": "philosophy",
      "index": 257,
      "question_id": 10618,
      "sha256": "64e16744415c80ae2f2e2bf917ba2fde158882388c237a0957b1e39df792164c",
      "rank_sha256": "0dd3737751e2a2e39704a13287bca6573da9e4c2783cb6ab590de5ff2e3cfe28"
    },
    {
      "subject": "physics",
      "index": 1288,
      "question_id": 9940,
      "sha256": "348d3ff5bd5089ca7c5aab1cf080f84bf6ad02b48857d9d4556762adb79e536b",
      "rank_sha256": "04ad2205cd81eeda9894270898a329b9e045874f7b841659b414f04609feb7c0"
    },
    {
      "subject": "psychology",
      "index": 575,
      "question_id": 2393,
      "sha256": "aa1d311e2055933ebdb6a4eb7b634bb3e6c67186bbaca0767ec0c39c742f17bc",
      "rank_sha256": "081276bbc9cde61b95e0daf877f5d4225f096b1e607eca6ff0eda5aaa8e1079e"
    },
    {
      "subject": "biology",
      "index": 133,
      "question_id": 2749,
      "sha256": "3010baf3c6fae75cac85a8c85afea2912cb0e8fc56e5739f5fb2aeb63f7d9264",
      "rank_sha256": "0c62dda6bb853a921a8c000de8b5545c663f64bede65c12ba7b7cccca71909a4"
    },
    {
      "subject": "business",
      "index": 4,
      "question_id": 74,
      "sha256": "52695735891b0ae88722a00df2e2dc5dec70875d9e68af3ec82a99c00eaccb81",
      "rank_sha256": "059e0b8a81f51d0d56565c18feda0d1f51d3f3b852176e818a5c9d8169fb3b32"
    },
    {
      "subject": "chemistry",
      "index": 565,
      "question_id": 3898,
      "sha256": "86ff9ae944d904b0394c3cb17e686b18aa1592def832edc4418d56bc93a81d8c",
      "rank_sha256": "05ebbffa62992df68eed830e0693702427fb10b9d7334bc8886a004b1a1bda56"
    },
    {
      "subject": "computer_science",
      "index": 248,
      "question_id": 10199,
      "sha256": "05a968108681896e1f7dff53a55a33088a7aa346afc87f6de1b0217520292623",
      "rank_sha256": "1103701fbab895c4edce9e7716b0b641fe1b88012ac4aea66049af3b4eeb34d1"
    },
    {
      "subject": "economics",
      "index": 483,
      "question_id": 6940,
      "sha256": "68d5da2c215491d44a4ab6f4ec308f16b91bf6083c6ece15c434f973ccba5469",
      "rank_sha256": "0512096550af3fe80c42a2ce15c603bbadb6baf2e06296f1e7d930cd513eee24"
    },
    {
      "subject": "engineering",
      "index": 145,
      "question_id": 11005,
      "sha256": "fd9cd928f603c502f2ecc4e180513436812b672f95505b504603f504b56e3ea2",
      "rank_sha256": "05af8a819d972258daef49dd703e31edded815ccd470f1240a6cf743223521d8"
    },
    {
      "subject": "health",
      "index": 315,
      "question_id": 6085,
      "sha256": "93d1a6690af96b0d8b027f8d1ed25b6524288fec7129e888b8b5ffba1d7caf44",
      "rank_sha256": "0c413c96bb1f9b15590f40174e965e65cd6322937884ccbdfc87e8f1eeeca458"
    },
    {
      "subject": "history",
      "index": 207,
      "question_id": 4672,
      "sha256": "a268344f16a78decf816502afb489104b28f48de8a491c43131a6a4aa99762c0",
      "rank_sha256": "0ed81aa2a8accd9beab780743e29deadefa7fe3c053b35a0f6441047385b2ef6"
    },
    {
      "subject": "law",
      "index": 130,
      "question_id": 989,
      "sha256": "0addd37e2142570310f7f16a740ec85834640c0d9696cfa01d98320321bce09f",
      "rank_sha256": "07093c03ce08c5dd693f0af15387c4ae7c75840db6a1cb77c2058d30d767cb45"
    },
    {
      "subject": "math",
      "index": 822,
      "question_id": 8123,
      "sha256": "f2322b86e3401eeab06027d352d0e2acfdc4161eaf6fc880140ecb05b08d8445",
      "rank_sha256": "033446f835edb2299b092b19fbb5cf32a53ccdb173c8bcd8b0f8794d5ac6c12c"
    },
    {
      "subject": "other",
      "index": 901,
      "question_id": 5747,
      "sha256": "989b20d17f973b294ec4a68c6d40907f393c526c4db107866be9ca3e076817d4",
      "rank_sha256": "06f02841fc197db7abd964862d2177df2dcad037361a497d8c4d1eb96c3f9cbe"
    },
    {
      "subject": "philosophy",
      "index": 457,
      "question_id": 10818,
      "sha256": "1d11ee7c8dd8e9c7891a01379cb16d90961327f014200d23e931d822c86b32d4",
      "rank_sha256": "0deba07e9b5d87c5340bc82bc427cc6cb5c33d3d91e593ed4cc4af002f7a38aa"
    },
    {
      "subject": "physics",
      "index": 324,
      "question_id": 8976,
      "sha256": "c2f6c756d84c08c784db0a255a8d1b77e7820174eeb57f854da33907674586f2",
      "rank_sha256": "04b43e1a0d53cb07b20aad5c84c48fa3fbf8f87d90b46b7693cd98b26804ff24"
    },
    {
      "subject": "psychology",
      "index": 294,
      "question_id": 2112,
      "sha256": "0ced46af3ae489c5c73923a38fa453dbf12d055af548e9642bba4843b96fef84",
      "rank_sha256": "0841c6776db869c20f2ee979c0f022fc1d128bc6e98fcd84f73d909e4253367c"
    },
    {
      "subject": "biology",
      "index": 57,
      "question_id": 2673,
      "sha256": "d7404c56ecd1b0cb1a84c9858c772663620bfce6e7473c50d8fe34aab1883182",
      "rank_sha256": "0c77245092a2bc3036d134669a1c5f367bf321adaba674e4403fe6f9c3346d30"
    },
    {
      "subject": "business",
      "index": 145,
      "question_id": 215,
      "sha256": "193ae754d9eef736efd84f11fd72ee723aa2d7f322846c69c2c52509f57b464a",
      "rank_sha256": "05f0db18ef0571a180809a0ec98d8072c7dd00679d0c454dfe905f36e3a06910"
    },
    {
      "subject": "chemistry",
      "index": 554,
      "question_id": 3887,
      "sha256": "c6bde3098c53a46b0583f78f79fa997841f876fc33defff6115242c1db61d492",
      "rank_sha256": "0637fe0099865753db3035521d8be627a43f82d942dc4d7cc28d9810698282a6"
    },
    {
      "subject": "computer_science",
      "index": 328,
      "question_id": 10279,
      "sha256": "5b50ad726dc1350564a543fc684df4bd4305001e5fad223b295bc5c2f3314569",
      "rank_sha256": "1111cc691862c461359b6d6c7412789d0289619d9bef7cd6949ed5a41a3576af"
    },
    {
      "subject": "economics",
      "index": 356,
      "question_id": 6813,
      "sha256": "fc2ea1f9bc691e067116ac63bb30c2188e0b7cf317811cb168f224ae0516f5bd",
      "rank_sha256": "059d706cfb1adcf0a5a5bfb9ee31d99320ab49584f2e09f9d02e05e53ccaae2a"
    },
    {
      "subject": "engineering",
      "index": 53,
      "question_id": 10913,
      "sha256": "8e92d6d2061a657f85aa780441ec5a889d77e83d0ed74cf6efc219038a1eaf4b",
      "rank_sha256": "05f5500c5f96ab0e3b9dd25bd51aeba731fa8150aba8394df2a10f252aa2a7e2"
    },
    {
      "subject": "health",
      "index": 319,
      "question_id": 6089,
      "sha256": "c9fee71eb4489981d34bcaedd1842ac054c2290ce59d48e9e1e31095dbddaee4",
      "rank_sha256": "0c459178018223c8fb95a67cdd6f7f0f45555757760c9fad1876d96674598f85"
    },
    {
      "subject": "history",
      "index": 238,
      "question_id": 4703,
      "sha256": "eb6ef0f2594215d074882193632c5b90fa7f02c3eb50f67fe57f77e486340fe8",
      "rank_sha256": "0f906b4e5d403784f4a0c10ef899fff94aaa9a9c45e84b552f15d010733a107e"
    },
    {
      "subject": "law",
      "index": 30,
      "question_id": 889,
      "sha256": "42a1b67d3841675bbcbb0012ed48034729edf467ba15195002147427efdf730f",
      "rank_sha256": "075bb4e634fb4918caf216db4d07b04cc6ba569480fc48470a7a97a5db5462ff"
    },
    {
      "subject": "math",
      "index": 1197,
      "question_id": 8498,
      "sha256": "2aa3121775090dc7b12cb9e958aed4686c2192ac632fa16dedf281d5e6156554",
      "rank_sha256": "03365b2e444e615f6ef10093f1911e0a5907e4ad6c0823e5a43862cc6ebacf65"
    },
    {
      "subject": "other",
      "index": 301,
      "question_id": 5147,
      "sha256": "be107ac28f7d72639fb96c83ce565ddb3dd5928a253671bebfc17dc0c18c2546",
      "rank_sha256": "0791feb3f605636a124d5baa10e85481f541cdd72d6b3a095eba2a1952810fa8"
    },
    {
      "subject": "philosophy",
      "index": 465,
      "question_id": 10826,
      "sha256": "63665a864895cea0b6006c3a4a0cfd7e3f2d641a5ffcd25d585944e37f6b6675",
      "rank_sha256": "0e189b45deae84b249d196ccde7cc1521f9d6a0d2442e2eff467721ab52132f7"
    },
    {
      "subject": "physics",
      "index": 763,
      "question_id": 9415,
      "sha256": "b0555f3ae518f08ab8142f63454c84652bd974c121e410f48301cd46bb1b7610",
      "rank_sha256": "04cec586ec8c2c8b76d8ef0d8db50e0ce46aaf9ac790f6ebed67a0da0bc32b37"
    },
    {
      "subject": "psychology",
      "index": 179,
      "question_id": 1997,
      "sha256": "9b5fad403ee86e20071cf9f4ae207828db6076c18199027a4d1514377c6c11d3",
      "rank_sha256": "0883ce132e19fafd11eac4f6a87ceb8fcc5f6a36b86bd3a26568be2085597e0f"
    },
    {
      "subject": "biology",
      "index": 606,
      "question_id": 3222,
      "sha256": "769e1af1062ee08d649577d2dba71569c77e7172421490df703ad11676082943",
      "rank_sha256": "0c9c8e5d77dd281417176018ceb7dbc302e6c879f7672088fe15bbf1f4b716cb"
    },
    {
      "subject": "business",
      "index": 124,
      "question_id": 194,
      "sha256": "63a23c3513b35c92ab0d76cb7800cf7eac4de03f27293822d08fe82b211b3ada",
      "rank_sha256": "064a5711521411b44cf087408a811dc8427f16758074e2b53e2f6740f3d8217c"
    },
    {
      "subject": "chemistry",
      "index": 352,
      "question_id": 3685,
      "sha256": "f3dd59c50ae3584b15180c4ed0a5bdfc050b822e05e2c5545be984dbf3038a6d",
      "rank_sha256": "06aeebcda281f5b59b86a035bf52c2c8d9d49a047d97b8bb4e59f68cec03d991"
    },
    {
      "subject": "computer_science",
      "index": 392,
      "question_id": 10343,
      "sha256": "451756142fd80d0029aa4be8a64a491994cf25dfc20f6f25e0b10cec9d32d7cc",
      "rank_sha256": "127a50f18686b4e1c72fdaffef7c9e8fb15a301b0814005a30428479c7b22e3f"
    },
    {
      "subject": "economics",
      "index": 527,
      "question_id": 6984,
      "sha256": "ab4998fef6643ac14d72994075d8d9413c9c5d21403b6c5894ac8f9bcb2c490f",
      "rank_sha256": "05aaf4a9f5bb5185d3045e27ecb9c15f1193e40ef1b6520ae66080d6de86b21e"
    },
    {
      "subject": "engineering",
      "index": 155,
      "question_id": 11015,
      "sha256": "6eb5b2cbc4536c146519e606e4e3c03373acf2b851e488a4080893c94a7f8245",
      "rank_sha256": "05ffe4e82adfd096d2be6ab1b9cf59f771a0cbcd837f2ff7e31691351daf9036"
    },
    {
      "subject": "health",
      "index": 513,
      "question_id": 6283,
      "sha256": "f4421fc27962c373e7ba03c230045543d8111adda9365cd6c7cd86a2849aab35",
      "rank_sha256": "0c721ec9aeb3f5da367ef36d865cff476f599b345ed1ea3b59b751d385763844"
    },
    {
      "subject": "history",
      "index": 76,
      "question_id": 4541,
      "sha256": "cb7cd3a0512869ec538a614b8de013862aaeef7f5bc75bd7c1730b34018d400c",
      "rank_sha256": "105a8dcc2604ee6bcb9f09e537fe4ab4718e571698df79d2dba347c935f30499"
    },
    {
      "subject": "law",
      "index": 360,
      "question_id": 1219,
      "sha256": "8ac8675ef6ef2abc7f47fd4bb0e705e5c3121f74558e522d5fcda89ecc1ef023",
      "rank_sha256": "07677f60d0e207bf73a9a147e69d12b886bd9f448c8f120651b2a3765b116ef5"
    },
    {
      "subject": "math",
      "index": 301,
      "question_id": 7602,
      "sha256": "52f29e8e7c600f20adfe380cb5be9b149a0b1a10371b9713bcc71d46092007e0",
      "rank_sha256": "036983832169c939b86d1c6f10343b569cd28f9823b6bb336549d5c7d128cbad"
    },
    {
      "subject": "other",
      "index": 597,
      "question_id": 5443,
      "sha256": "e8780fa955138e5992e6719009c97b375f0f9cd6e02b51cc1ca2105576e86189",
      "rank_sha256": "079dd3b837f23f8be854b714bfc6022f2efd806fbc9c03bad96e395b84216a42"
    },
    {
      "subject": "philosophy",
      "index": 4,
      "question_id": 10365,
      "sha256": "e35132ea1f6fded95f0eb14f47c33cd47614d854971171c5f81e12e8bb1dc2f3",
      "rank_sha256": "0f27ecabbedcb1344195b5698895dac98670a6f0eca85f2e01b523f97fbb3357"
    },
    {
      "subject": "physics",
      "index": 794,
      "question_id": 9446,
      "sha256": "d49bff40eeb7627bc6f898596a9b98c84eaaa8bf211ed1c3923f963e335fdffa",
      "rank_sha256": "04d7e445146ee1d0d77db3300fd2277d15d278479401d5b7096e7c9c2bcd645a"
    },
    {
      "subject": "psychology",
      "index": 416,
      "question_id": 2234,
      "sha256": "6828c828bae4f24f8b47c37ba685b296b1f4328578be8d75a10c230f0383095b",
      "rank_sha256": "08b7194b8b31a435b64b79e28d0e7731be399772fa908d634cdb1287981d4995"
    },
    {
      "subject": "biology",
      "index": 317,
      "question_id": 2933,
      "sha256": "bf72607bb3c6c12c90dee4e6dd63cab29b33769f6a67a7fdefa7fdaa9acc3b7c",
      "rank_sha256": "0ca49e571f718c6cb355a115f02d26f012a15cca7823681e819571a5bca9ec85"
    },
    {
      "subject": "business",
      "index": 398,
      "question_id": 468,
      "sha256": "b9a8996377a6fab3ec18bd2536112c2facb64db4892bb85b9e0f58057afef155",
      "rank_sha256": "06694221d139bae906e7d200e54b7c4fbddc8d1745edef5f8c9fc58d287a91d0"
    },
    {
      "subject": "chemistry",
      "index": 444,
      "question_id": 3777,
      "sha256": "d5468b05756b75a073871e0e835e5ef991a062723e057331dbd9a17fcaae79e7",
      "rank_sha256": "06f321b1642188827d2a142acbe786c0f3b00137133001230be98619d7bf699c"
    },
    {
      "subject": "computer_science",
      "index": 251,
      "question_id": 10202,
      "sha256": "85337ff321bf1f7e2a9e9f3774f13e547aac05e1a0563f9797b8bfb2a4b54e6f",
      "rank_sha256": "1451c417030750f35a58d279a53512bedeb2623524f59e5ba62a8e4ea01fda20"
    },
    {
      "subject": "economics",
      "index": 194,
      "question_id": 6651,
      "sha256": "049fe8d71743e630e324818ce94ea54e063f49c856f60d09ffaa2afd3c24ddde",
      "rank_sha256": "05ce350b466c490fc0dc8e346cd9a760d6de544f79e904db70de4daf1b78b523"
    },
    {
      "subject": "engineering",
      "index": 951,
      "question_id": 11811,
      "sha256": "cd00925a89525bb6ced435ab550507e366d5a7876c9b5f288d80cb0a85a84af5",
      "rank_sha256": "0609b658468b2420d049c1d4fff56992ea8e28f210597052940c3509e91dfcb1"
    },
    {
      "subject": "health",
      "index": 661,
      "question_id": 6431,
      "sha256": "414297d723e1f4cbb17d42c09b0b3295150cb21ed0ce1de451e9be5d0fbc2f6b",
      "rank_sha256": "0ca1a1211ec77570ed44c5677d762cb5a42e8aae8603f85e39443ca112371847"
    },
    {
      "subject": "history",
      "index": 290,
      "question_id": 4755,
      "sha256": "279a0a2dead28795c0e51813b82d694688224ee51114dc900e714e4f80e9309e",
      "rank_sha256": "10fcbc9441d4a176dfb7f9faddb06319ee2d37d1a73e435a911247c3c429b3ea"
    },
    {
      "subject": "law",
      "index": 542,
      "question_id": 1401,
      "sha256": "8832a06b8afc1734e7787971ff95fb8f82392cb1808c78031d3aae53a89e27de",
      "rank_sha256": "07cdfe527099eac6c6e315e92a9507e8960bf17214bc5a8884dee77413411b59"
    },
    {
      "subject": "math",
      "index": 996,
      "question_id": 8297,
      "sha256": "409a166e92f3001364bec531429d3cc0fe73ed9188fa6218c2250f7ec3c05d8f",
      "rank_sha256": "038098054e556efeb5510f5da422370561365aefd2f838f03af4824a1e9aa435"
    },
    {
      "subject": "other",
      "index": 766,
      "question_id": 5612,
      "sha256": "ac529bb2341a32a438a1c04a76f0098bdd8f62c01d55a2101a400c5b0e04caeb",
      "rank_sha256": "07c15060820d73c8734dee521dbf521731b5fc92ca0ad1ff17e14531d17c932c"
    },
    {
      "subject": "philosophy",
      "index": 294,
      "question_id": 10655,
      "sha256": "57950d8e3e6cfe40d1c45b761f3571b4cb238e1b3c9dcf38c3c5063bfd0d0099",
      "rank_sha256": "0f5caae48e0ec6ec45c04c7aa35a74af199a760699eef3b1dab464a139d2756a"
    },
    {
      "subject": "physics",
      "index": 1214,
      "question_id": 9866,
      "sha256": "be64bd9c1721d8e26fa6dfd1af9bfec9bef6b2d807c685c54e86a6104ce64568",
      "rank_sha256": "052939146ee79ae8b1ca63244954bcf9593cea77dd191dec39ffd75676664cd4"
    },
    {
      "subject": "psychology",
      "index": 33,
      "question_id": 1851,
      "sha256": "c61a3bde8d90db5f119e51641378706fbf5573246c3d79c5408b1f78642378cf",
      "rank_sha256": "08f47f3467b34f9fd29ddc773f7060aa90c52648cd30e7d1b26f4f5a3b563266"
    },
    {
      "subject": "biology",
      "index": 610,
      "question_id": 3226,
      "sha256": "455d0de0b711d8390b9759a40d4d64d091d891294948b167e6941537c14deb65",
      "rank_sha256": "0caedc97ee78b14b0101bf7f6da6717bd3802649324403efec6ae3d33826752d"
    },
    {
      "subject": "business",
      "index": 770,
      "question_id": 840,
      "sha256": "c859ed029c364c59c213025a5f3463498e0f0c30703f1c6defc3cea3f3e07afb",
      "rank_sha256": "06859a633adb78d75c65d2eafecd12f875327511282c42840d461c20d64a281d"
    },
    {
      "subject": "chemistry",
      "index": 834,
      "question_id": 4167,
      "sha256": "2150d833efe04fa752964b4bc9860e00e31bb744ffdb42238bfd41cea48a1aa1",
      "rank_sha256": "07065cefff954a26c7a19039facdfda5c465c5652164f50ddb9f2afe8ec53626"
    },
    {
      "subject": "computer_science",
      "index": 26,
      "question_id": 9977,
      "sha256": "33d6bd2f791d7d1411ee522d91ffdca64faaa12f5fc323f77b5ac1c0b390309a",
      "rank_sha256": "14adc11d2f03f8f785af83c2c54fd5a30ea3aed8a18c4e2e55ed0d5d5438ecfa"
    },
    {
      "subject": "economics",
      "index": 388,
      "question_id": 6845,
      "sha256": "26b5af59114a1b162849a2f9a01e30ceb150998ca09abcd556cfbb90dc87c1b7",
      "rank_sha256": "05f073462a738aad66de12282d161a5b301cb2abd01517fece510897fa18ab22"
    },
    {
      "subject": "engineering",
      "index": 534,
      "question_id": 11394,
      "sha256": "13f261fafa1df53607d3519bd3bd051a822c28c4a623dce45786bc7f6b9147b7",
      "rank_sha256": "0635ece30f86581c9ea21bd9a9f509f17f1d7c0fef19018219eb91cce3ab685a"
    },
    {
      "subject": "health",
      "index": 518,
      "question_id": 6288,
      "sha256": "9b14b6907750b5bdd85192ba93f943f232c8165a78fcc217b33e5771cf1060cd",
      "rank_sha256": "0d0c1645c9e3c848168a31a13360d6f82f965ca40d97226176c98c93e663b33c"
    },
    {
      "subject": "history",
      "index": 153,
      "question_id": 4618,
      "sha256": "851c2487affe52e59931e6f5d31aef39dd124f6a147a8ea9b5de2743a5a0efce",
      "rank_sha256": "115865d1296c3c0b4eb3c28ecd7816672836933a016f85599a2316d8f67e8960"
    },
    {
      "subject": "law",
      "index": 768,
      "question_id": 1627,
      "sha256": "d72a76ce7e7fb35c02c7bc4c59fe6b9ff11e9cb5bb3dcb531760430b73f55464",
      "rank_sha256": "07d48312ed8e66d74f417f6256b1c19727ca708ce61434af17cebb7b0d636b8a"
    },
    {
      "subject": "math",
      "index": 1172,
      "question_id": 8473,
      "sha256": "de007955056322cd485975d20c467251fca0613b10e62cb526dbe7bb500c1024",
      "rank_sha256": "038a5d9424dcb322a2e7f4c69dc9f7b9844bd334a239e7380ba5cd41a7c63ac3"
    },
    {
      "subject": "other",
      "index": 910,
      "question_id": 5756,
      "sha256": "85083e436f9d05a48c81efb2f3a81b8d68cc55e5d317bf3158c93b413c2213d5",
      "rank_sha256": "0805b9607d6e959f83684c9c8fa7042a67663a2387c0d7dee4d9c466d813941c"
    },
    {
      "subject": "philosophy",
      "index": 454,
      "question_id": 10815,
      "sha256": "a7c039d785b55539fdef2bdeb434c001066354507e894e803a4f865a0b991bda",
      "rank_sha256": "0f5ce002ad9aa25e3ff63afd02582d6d32f30761e55b01f743fe714bd80e5a67"
    },
    {
      "subject": "physics",
      "index": 629,
      "question_id": 9281,
      "sha256": "019cc715f8884c064c745121e46e1749eaead467fb8a28292a6154f9d5483e24",
      "rank_sha256": "0564ab1aac1a1509b9b95096829677d4adca6cffef21fa4ea15094ab65e9fcb4"
    },
    {
      "subject": "psychology",
      "index": 533,
      "question_id": 2351,
      "sha256": "4b6b2e1e25a35da1208dadf1a00f3d8c79c224be085659d0b437f47885b5061c",
      "rank_sha256": "0a5c46f8a3e37403549381df8aaa86ec85d74de667e8f3c1e6a5f238dbc8a66e"
    },
    {
      "subject": "biology",
      "index": 549,
      "question_id": 3165,
      "sha256": "5afcd44fec345e9186280c1e445989e0ec5bcd69c9afaf6cfc6eb518c581da13",
      "rank_sha256": "0d6971b35bd234432775eed4231686b99349b9def4e2da1f5fef06005571efbe"
    },
    {
      "subject": "business",
      "index": 648,
      "question_id": 718,
      "sha256": "1c68d41669e3f38924370ceb190e1faf689b3c752067115aa13c0599218c18e4",
      "rank_sha256": "06ac5ab918bb053f22d40a286a15cce0f696a9afe65c0879d9cefcd53c835fef"
    },
    {
      "subject": "chemistry",
      "index": 768,
      "question_id": 4101,
      "sha256": "cb26447a3be6d220e5a2ee4dd2f158d86138b881818cf792e9f026c4a90b225c",
      "rank_sha256": "0714ed0ee39de4f3cd48205be0685f5913d4513e8b29c325b3a508388d116cb9"
    },
    {
      "subject": "computer_science",
      "index": 177,
      "question_id": 10128,
      "sha256": "6d3280e7923d21b3927418e339f19bb1770b4da6bfa8a1e25d0e34faa72f7f1c",
      "rank_sha256": "14fc2c85edc6fcfd1594a7d8fd69be2a42a45c41ef19875515a0f94ddb70ef82"
    },
    {
      "subject": "economics",
      "index": 325,
      "question_id": 6782,
      "sha256": "dc5d4ce455e5dc7a5fa8bcfa48cdbb5c6339f309bc540cc9b195adbb4ea8876c",
      "rank_sha256": "071cce75f3b2f572b2475da49aa42db43ebde4f826116a829abb5133fbf38849"
    },
    {
      "subject": "engineering",
      "index": 480,
      "question_id": 11340,
      "sha256": "b516f8e2271afa250e1bc4bbd441a3f7dfc80826634b0ee32da9ed46aed1be91",
      "rank_sha256": "0711d10cce22b2bdcbc5e0c2c4491b3a8ee2fe0263165ae797acb01fec02dd46"
    },
    {
      "subject": "health",
      "index": 169,
      "question_id": 5939,
      "sha256": "a47a7d1f8b59d8672d9caf5803e3312c98b5a100f4673c653debbad78a1d5ac2",
      "rank_sha256": "0d2aaaf6cc1e3828d972f15ed0722ff9c39722288157f1a4e4e69fee29543d86"
    },
    {
      "subject": "history",
      "index": 44,
      "question_id": 4509,
      "sha256": "4797acc90fdaec223af1e75a5814022cdf5e5655ee606bb74b3ad248e91b6ed8",
      "rank_sha256": "1335a0437d9ef801e39ee4112b06f30cdcd66df69dec2f5e7cc45bb60c38aa28"
    },
    {
      "subject": "law",
      "index": 476,
      "question_id": 1335,
      "sha256": "fa6a26037bdb27da59dcf5f9b5dff59d5c2e0837a47544adbd4f5450ccee04bc",
      "rank_sha256": "07e10590fb7209a17b32343d40e94a8191c3dbe18a7c9cc44e2a48643e9072d3"
    },
    {
      "subject": "math",
      "index": 222,
      "question_id": 7523,
      "sha256": "1d68b75f733d5e0e8a6451cdb5f118ae7cc0d75fa1a9042976d39aed39997eeb",
      "rank_sha256": "039c80419b3fb37a21047aa9fe49d3098c06ceb924e898fad6c6e006b7895f0a"
    },
    {
      "subject": "other",
      "index": 823,
      "question_id": 5669,
      "sha256": "4bae2f539081e0307720fdf43883ac73d87df8e24e4c5800b42b95d9046d95f8",
      "rank_sha256": "08e2fa7f067a871df86265c05424f2d11c7d522cce24ff0e4a3b04cb0ae3fa26"
    },
    {
      "subject": "philosophy",
      "index": 220,
      "question_id": 10581,
      "sha256": "71ed757381a37b6b5d23108ac2713ea03235f446c3d16613978b248d1ef7b7d7",
      "rank_sha256": "0f5e4d4b69eb96c26272e1b0bc06e6e68726441637e6cd8e3ea5c25afa709bd3"
    },
    {
      "subject": "physics",
      "index": 244,
      "question_id": 8896,
      "sha256": "2d555087d5c28f07b854784d4bab90c318953f80bd553318519aee05bb678472",
      "rank_sha256": "05a835ccf17548fdce3264811a26a21abad531378f4e1584a54cc6c3c9eb6c4e"
    },
    {
      "subject": "psychology",
      "index": 677,
      "question_id": 2495,
      "sha256": "31974771d9317ec4bfd9ec2230d4b5036b857bfd137aa9bcaa8ebc45cf60fa8e",
      "rank_sha256": "0afc2eb498f31c71886915094edce7e1227a2b1dd20904c4d7afa3ed136c9a4a"
    },
    {
      "subject": "biology",
      "index": 423,
      "question_id": 3039,
      "sha256": "54aa98e7fd2a16445fe0fe591a9bff57524546719a6db178e2a5b0e8f2b98d44",
      "rank_sha256": "0d8fc06d15e2bd47dd6522d13cc220951f4761b5453e787a60c32f78b91c34ec"
    },
    {
      "subject": "business",
      "index": 767,
      "question_id": 837,
      "sha256": "eca4c42fd7308ce26cace8bdaa7e97fd546a0407b144bb5ac76fd4f3d77e9328",
      "rank_sha256": "06b79d01121085c3e8fe6dc57dec5649521beba2ea918fd5939a18d31a353c7f"
    },
    {
      "subject": "chemistry",
      "index": 930,
      "question_id": 4263,
      "sha256": "17faa39928cd70d36187431f98398f20ccfffe14daf692fecf70eb786543e11c",
      "rank_sha256": "0750de85ac7021585eda0c2b61e1306aa77418bd4a5caca1cd429a14f55386a7"
    },
    {
      "subject": "computer_science",
      "index": 298,
      "question_id": 10249,
      "sha256": "d7e42de3c033f23f90bb73144e53ad316d80f30d82d6f9b48717ff35f8fc9122",
      "rank_sha256": "15c87e02e516bf46cb913487ca54b15fd5794a51f31c0c562ccc3df73811ee94"
    },
    {
      "subject": "economics",
      "index": 592,
      "question_id": 7049,
      "sha256": "69ba49203d733aed819b0317ecb56a5ca3f7543f2a7cc07c1ace280585823938",
      "rank_sha256": "0752b38ae02ced293a6142209d50c87074d4cf26df687771ac14b026b34c2099"
    },
    {
      "subject": "engineering",
      "index": 789,
      "question_id": 11649,
      "sha256": "0925856c0a24570c6d1318a435a3129e7db37dd2e85282847296c3d695c64dcc",
      "rank_sha256": "074dbb53d1fddc7bd99d215688211f2743eda7979994580fbeebd10e026005dc"
    },
    {
      "subject": "health",
      "index": 344,
      "question_id": 6114,
      "sha256": "0e7481933843a71c2000d74f0b4b7114e3b1a477d6b70d03ae4b62a3b48fe9c5",
      "rank_sha256": "0e13eb6911d2a91f8173aece5e7bdd772d9b0bff9d1559b93580147772ff5bd6"
    },
    {
      "subject": "history",
      "index": 228,
      "question_id": 4693,
      "sha256": "908edab5d5f328cf7988d629c6001aee840bf8a49525ef29d9f1b9086ebfa418",
      "rank_sha256": "135f5423cfb810a2b9d3fa7889d4745d83539da6bce40a94ec72fac5126f9462"
    },
    {
      "subject": "law",
      "index": 239,
      "question_id": 1098,
      "sha256": "b2810ccc59e20728f8b517324b84e2c0d2ebda224d14103ccac748023228ec58",
      "rank_sha256": "0826a1786ae0a5e41d0128d2aad2ac014b1401b1818bc3448178474fee982466"
    },
    {
      "subject": "math",
      "index": 153,
      "question_id": 7454,
      "sha256": "97463e56790044ccf18a393861333d7ce3dd00b2efc13b85557ed337570bfa64",
      "rank_sha256": "042fdbbd0b7e8e22726c3670ea42e05482f35f61f81010741bac901c768d2aeb"
    },
    {
      "subject": "other",
      "index": 420,
      "question_id": 5266,
      "sha256": "4cfd178bd7a853922cd0b2395bf15f61cc0011deaf5459ef78ab5a31b04277f8",
      "rank_sha256": "09df24a32777d47d93227ce6fdd57e354c327ef56964083bfa1f3cac9c6b9f78"
    },
    {
      "subject": "philosophy",
      "index": 287,
      "question_id": 10648,
      "sha256": "9ab5c696f37a14f35217b1db8935dcb6037f58620663fd361bfd54c2c55faa47",
      "rank_sha256": "0fe5b920ec9ab06692a79e951574120e1bf94e82d47ea7c9e1c5e71f0dfe2c39"
    },
    {
      "subject": "physics",
      "index": 646,
      "question_id": 9298,
      "sha256": "8c2c00ad3148c9a64146bf20bd1828823fa59e8c97b3b3bf2e1df6fd0c8ff278",
      "rank_sha256": "05c72af5a05693df6b45ca417b769740be4e111686fa75c845532e82f1d7bc6b"
    },
    {
      "subject": "psychology",
      "index": 438,
      "question_id": 2256,
      "sha256": "76e3503ff61a75f72a74f6a746e37748af1efb41aec90b3683d1218a0f0c5aee",
      "rank_sha256": "0c71724dbd8107b045d60322cfd289cf838b4c570966399a4f3e6dc707828ba5"
    },
    {
      "subject": "biology",
      "index": 646,
      "question_id": 3262,
      "sha256": "90f5284e17763fc0bb549b5d921396e74160aa1f7d8dda992977649f9e49a57c",
      "rank_sha256": "0ed41de1740cd11e6a3ce84ccb4ba809beba3aa5c1c7d53a32212e0aca66b8e7"
    },
    {
      "subject": "business",
      "index": 391,
      "question_id": 461,
      "sha256": "d6c7d2954045ebb06398d0192ac57f0905b54c2ccbea2405f04d6e78921bf9bf",
      "rank_sha256": "06bcaa67f2503f87329114037ddfc03ea3ecfb0d9e357d68f6feb37f88e1d680"
    },
    {
      "subject": "chemistry",
      "index": 1043,
      "question_id": 4376,
      "sha256": "0402718c4cf83d9e0182641bba902f59e94ab8b66c4cf26b25b2c9623a989cb9",
      "rank_sha256": "07afb78ccab953dae638671cfed45087323b79e666b2e552adcbfc802fb191f4"
    },
    {
      "subject": "computer_science",
      "index": 396,
      "question_id": 10347,
      "sha256": "33660928188de90f4b7e7b6d1f7864bf938e74ac7c257fb04f4bd38299445415",
      "rank_sha256": "162a89c38ec682a94f1339a9d5663d87b23dc106240fe754eb09c01f63fb1eae"
    },
    {
      "subject": "economics",
      "index": 736,
      "question_id": 7193,
      "sha256": "59666b99db3e3436fde4e79e635ff793fd09506e7f77dcf0a64d1ec234be8299",
      "rank_sha256": "079f05f2f565dff223e12fc21fee37c9f20be51620a737b19a8629f4edab2464"
    },
    {
      "subject": "engineering",
      "index": 677,
      "question_id": 11537,
      "sha256": "c3d55fe2ea8ff884d7e8350f23ff8dc6754a024eab952a11ed78612c878c4d2e",
      "rank_sha256": "07cc740600ff50eb055f2a50cd07bd39297038493a287e6522bafdd17420f138"
    },
    {
      "subject": "health",
      "index": 214,
      "question_id": 5984,
      "sha256": "e18d663ade5b66ce5ab5d09fea5a6a05d6f2111fd13b9cd82bc36fd2b97478f1",
      "rank_sha256": "0e7050b4c3025d98e5086934ce5eb505f558f2e7ce56030eefd346c212354fb8"
    },
    {
      "subject": "history",
      "index": 358,
      "question_id": 4823,
      "sha256": "90c92ab8c773f551ba1c215a295af73a89e8b34782caf3e44cb53a97755b0f2b",
      "rank_sha256": "13d20331077485f12c0ba84b7a508af5385ca10d994c30db7ff9c4b55de5758a"
    },
    {
      "subject": "law",
      "index": 253,
      "question_id": 1112,
      "sha256": "fcaa22d91150ad69c8270631e2345f5420a8f5291e16569af53258015c39d92c",
      "rank_sha256": "0828434c3a502ebfd7d9360f25ceec0a801bc67cad5a6f25cd43672505535fc9"
    },
    {
      "subject": "math",
      "index": 100,
      "question_id": 7401,
      "sha256": "e3437d7fa1bc9a3437074012e34561e3da36fc2f3dba5435600834ad2393d88a",
      "rank_sha256": "046d56469bed057cf0ff4ab1b629d7542b4fd92c3f2533bdeffc16e510a5e090"
    },
    {
      "subject": "other",
      "index": 113,
      "question_id": 4959,
      "sha256": "60eb961e12ed007e8c0f0df8bda6c5cc819d9699f7f26e32a7827d5259e2a388",
      "rank_sha256": "0a682775ce92e3010090f9f791c54f9543b501301dacce54ac6534c96ad45bc5"
    },
    {
      "subject": "philosophy",
      "index": 148,
      "question_id": 10509,
      "sha256": "9f90b1e4aa9ba0158d814a383990cb9436c156e56d2455c78f725461881a47da",
      "rank_sha256": "10c5455b1b7362febdae15d93de4dc46a8056b0f7e884a2ab7c9794f5dfb9b8a"
    },
    {
      "subject": "physics",
      "index": 582,
      "question_id": 9234,
      "sha256": "5ff103a9a060c0c7d8c4a64d7cff4701c8f7809d57e97ab9590cc0c670325650",
      "rank_sha256": "05d908c2fc24a5f9a58239544e0bac9ba1895c1dad365859e41e0bc55f94bfbf"
    },
    {
      "subject": "psychology",
      "index": 365,
      "question_id": 2183,
      "sha256": "8fcb198e9e42b95bb6048ac0cee7b2fa5e01d4adc3ecd99d3c5b64195b4c31d4",
      "rank_sha256": "0d52875eecd4cf9ae9d61f5cd84bb3ef1d4dc98b696568f52868409bf0b4ed61"
    },
    {
      "subject": "biology",
      "index": 389,
      "question_id": 3005,
      "sha256": "b3564b457104813e4fbc47d0feb6e5fb99ef92a1aca53e7599b8d3ce00562f52",
      "rank_sha256": "0f5777fa05d74e5bfc1bd2967cc6ecee3818b758ee63d02a22a308f655b0c0e4"
    },
    {
      "subject": "business",
      "index": 92,
      "question_id": 162,
      "sha256": "6b62608ad10f1d898211e5e010235acdd451b07677ff678cd2435c7cb9fd3e7f",
      "rank_sha256": "06d14e5e6a1d256a2eb786476f781af11bac360bc1b7f573076d2ddc0806cd8d"
    },
    {
      "subject": "chemistry",
      "index": 1042,
      "question_id": 4375,
      "sha256": "904e5d10b5475f8095093f28657a9b20383136f5080cc25a4afbf07527d86811",
      "rank_sha256": "07ca97b4863e75f667263c92a67930776e7f57e09a6ad1e18d001bc9b6d4f150"
    },
    {
      "subject": "computer_science",
      "index": 124,
      "question_id": 10075,
      "sha256": "1a18ee0ff9f44a7619d236e15351599b59ae7a214d3cce382479e268375a4875",
      "rank_sha256": "1724780c3b3762f84f3919c527a9f7b557886d94a78063ba52dc0bf4095f6adb"
    },
    {
      "subject": "economics",
      "index": 99,
      "question_id": 6556,
      "sha256": "aae914ad86ffa9d842dcdef101bf0cbe100659079d4b1542e188ed646be18d95",
      "rank_sha256": "08974637b1b45ae183e187482f397cc73ff3b79045eaa468dd7b1001a50b50d2"
    },
    {
      "subject": "engineering",
      "index": 902,
      "question_id": 11762,
      "sha256": "57c7848c17bc14ff1cebb52bfbee7fa6499b288eaf72df5a81d0b5c7d81f4874",
      "rank_sha256": "08291f6b46a7fb9602a5ca14fb5040d658a6476b99cd95df940b8e4eea15053e"
    },
    {
      "subject": "health",
      "index": 255,
      "question_id": 6025,
      "sha256": "ec6c49791e8f56a78e721ca4cced8bc597667f9c0d1e29f91c26dee8cf9a9df2",
      "rank_sha256": "0ee535f680d4a25cb7385f86e4412c0db07a18a39f7e7fdb1c351a890b77583f"
    },
    {
      "subject": "history",
      "index": 35,
      "question_id": 4500,
      "sha256": "6a8702ef86689e43d54afc7761b127b6c981714b12eb70078db9e6cd14123cbe",
      "rank_sha256": "14b8d6ef9da625b843960d118f7594461da9457412fa3be598300c58240bf709"
    },
    {
      "subject": "law",
      "index": 925,
      "question_id": 1784,
      "sha256": "b5a685e39f3993d654ad519a1b299db86f11f1339f3fea644983f77d860722ca",
      "rank_sha256": "0835d1836501fa59aa87bdcd77807530b7a64593c2ba5e5cd3a05a9d97b465b5"
    },
    {
      "subject": "math",
      "index": 748,
      "question_id": 8049,
      "sha256": "ec555c33f2c34f65f31c62d1a3a722399ab52d22b7f3bec31a5b5e74261b8910",
      "rank_sha256": "04cd69f2f2184ef11e23fbb662cb8929a6b516cf072f3b262e8430ff3bb52f5e"
    },
    {
      "subject": "other",
      "index": 621,
      "question_id": 5467,
      "sha256": "06e4980ddced0fe483315bb0a5263d1c2a617f0bdf3bc1d3d5168c9b6a6d48b7",
      "rank_sha256": "0a995c971b701d2033831a14c89522ab9964984bd11249229f46353cac2bf988"
    },
    {
      "subject": "philosophy",
      "index": 104,
      "question_id": 10465,
      "sha256": "0cb7b18f2900af785d53eea42677514e5f7e032c14a87b30596a405fb9e1e7e3",
      "rank_sha256": "124e1a8e25d4aff97fad4e4852b7b2301292becdb14162b07524220485fd1d64"
    },
    {
      "subject": "physics",
      "index": 1005,
      "question_id": 9657,
      "sha256": "c1528ef48944f86fd3b65ddf73d262b4043daf2d718823ef5cf6a09116392c83",
      "rank_sha256": "05e2065ddaecbd61547ca38fa2a2a1b89fa5ee1390db21d0a7e14e46b48a0eda"
    },
    {
      "subject": "psychology",
      "index": 482,
      "question_id": 2300,
      "sha256": "d8f410001cd81b2f7a6253b935afacb202afccb8316e6613eacf0feb18a6cdae",
      "rank_sha256": "0d744c0eca6c8f6f074142150f00cd66d353b7e90285de1ef7ca08150c8ce964"
    },
    {
      "subject": "biology",
      "index": 52,
      "question_id": 2668,
      "sha256": "3e8bdc3576812988eb5c68f709266c2b08a14b5571175bb5b0343528ed665dba",
      "rank_sha256": "0f636cc358de6fda2fcb516568c6d424adfb9d6944e714f76ea2a15a782c9ae0"
    },
    {
      "subject": "business",
      "index": 245,
      "question_id": 315,
      "sha256": "8d24917f211a8b3f593bcbd98888f55ce85433719e41e55e28a8cef524b330c6",
      "rank_sha256": "077d3dfea99dca16a85400782d4c1a6393764c9ac649b9de42cd41b3d13bc5ee"
    },
    {
      "subject": "chemistry",
      "index": 318,
      "question_id": 3651,
      "sha256": "1d7d554999991eb7a126ed886a75eee040095f638f7f46bb70812ff047eba7bf",
      "rank_sha256": "07ff3a69aef5fe917755f4ace7d1347c0e541960d00296b6d3a69a140349f22e"
    },
    {
      "subject": "computer_science",
      "index": 240,
      "question_id": 10191,
      "sha256": "dfe320f84b2994b762d101a37334a38e00c2ffe74a94925802ce67318b5ebd4b",
      "rank_sha256": "177424cf75119d6819dbc0be61b2a9210aff9f724134d49abd7db40b04ad7809"
    },
    {
      "subject": "economics",
      "index": 650,
      "question_id": 7107,
      "sha256": "c6fa875165c9bd4803adac4c5dd2ab9793049b1b0649726ed78f7974de869100",
      "rank_sha256": "095816a3cb5a7110ba0ca27d1370c5cfb8c5806664cdab006eaf6aea266ffc0b"
    },
    {
      "subject": "engineering",
      "index": 916,
      "question_id": 11776,
      "sha256": "efce5cafc60419f13fdda4401a1ac077711249715eca96ade8637632ce8d3e19",
      "rank_sha256": "085f678475bac0b0c94814324ca7bc0eb0b6c857e417cec166b6cb4d0ab476b3"
    },
    {
      "subject": "health",
      "index": 412,
      "question_id": 6182,
      "sha256": "20fbc239f97c27a104f728141e5608cce987be3e4935194a6faa32c251ec8f66",
      "rank_sha256": "0f72fdac034bda4e48ce21ccda0ab8bcba449ceac7fc9b64e27818e54636fa7a"
    },
    {
      "subject": "history",
      "index": 41,
      "question_id": 4506,
      "sha256": "e909e92696898a8005584ddcb223aa46c6119ed70c85b34213ddd6ca52784458",
      "rank_sha256": "14e5f49873a944cd869c7f47442437734bef68d919cb061977ff676e82590642"
    },
    {
      "subject": "law",
      "index": 796,
      "question_id": 1655,
      "sha256": "21ce4b71bd7e4d8bfc06b7fcdb782e0a2e58bff4672949f62d25260aef95b76d",
      "rank_sha256": "084205c9acfeb96e3dc3e251cada749dc16f325582041c519022879379143293"
    },
    {
      "subject": "math",
      "index": 282,
      "question_id": 7583,
      "sha256": "5e2f85b5fea1299414f961668248eb24dd185ba1f26f37ec2a5c8a97403bc045",
      "rank_sha256": "04f189a6dffcfff94febee4a83898ec5cc7de311b3e532a3c3d402fe47ba5256"
    },
    {
      "subject": "other",
      "index": 336,
      "question_id": 5182,
      "sha256": "415fc3e5c21af7b0c01efbfb83ed8dced9adce6abf2003f7c8f4a43bd9b7a863",
      "rank_sha256": "0ab26bdfc1c1207da0fdfebe450403b3d4a2696c2cd920a23ad0a9d3443ba702"
    },
    {
      "subject": "philosophy",
      "index": 107,
      "question_id": 10468,
      "sha256": "27b956b549f34319350e0b6894f90ab4e892e598c35a40b6ce785b0948bacd1f",
      "rank_sha256": "127c1738d50b0ed44eb17c8ac9e1bf336e49e5e3606410ed666e0e929ad32b8c"
    },
    {
      "subject": "physics",
      "index": 686,
      "question_id": 9338,
      "sha256": "0f58c8d3072b4e6e307c92acdcdf039f287fb98daeb5671be0250d34b7ba279a",
      "rank_sha256": "0637c766e984789e617da7b81c090319c1855371c8283fd228eea94595d8e1f7"
    },
    {
      "subject": "psychology",
      "index": 186,
      "question_id": 2004,
      "sha256": "c61e560c4ba61fefa8e8b6b2a3c679c05ac471de04725cff7cbc0222f5f07b56",
      "rank_sha256": "0d78e7b8e1b402912e6f87351ff7c919eb5ce70ed540797fe63f9ae97340c0db"
    },
    {
      "subject": "biology",
      "index": 629,
      "question_id": 3245,
      "sha256": "07515ecf2bc7cd658de12a30ea7a86c988694666cbf9fd7a5891c7da938b0c08",
      "rank_sha256": "0fbc51b1f3532996a7639e4972c7f228efde4018c30d5910450e683dbc7db867"
    },
    {
      "subject": "business",
      "index": 219,
      "question_id": 289,
      "sha256": "63b17c56ced5f1609bc6545570a760922f624c49c16dc33f08587f305ca90424",
      "rank_sha256": "0841d1c704d4aa17257b7226b8717f0ade69314ea7b01f2d05c024f3064b6eac"
    },
    {
      "subject": "chemistry",
      "index": 1130,
      "question_id": 4463,
      "sha256": "6fbe380a9c809dc31f50996f07ec8809161c2e11646d55592df6b0d4c7752e5c",
      "rank_sha256": "0819874bcab66b7364c279e69661b7227622ec08ce6e34a41c4e33746cb64930"
    },
    {
      "subject": "computer_science",
      "index": 110,
      "question_id": 10061,
      "sha256": "dec78ad53439ddaa0088e7746fd5e55615ad166fd5ec0e602ec86df5f6cdb0d5",
      "rank_sha256": "1808962677899fb77ce40738d5aee60e23f7b93a2ecf232f6db6d9748f1da494"
    },
    {
      "subject": "economics",
      "index": 712,
      "question_id": 7169,
      "sha256": "97f67e75ccd912046a1f06ec8c264a49406f8ac14075d4fbfc23abe13d8f726d",
      "rank_sha256": "0970560e90c163fdc11ae05b32f2ac951266d8411596723fbbac81a5b45769c3"
    },
    {
      "subject": "engineering",
      "index": 623,
      "question_id": 11483,
      "sha256": "a497fef7791d80713726b7c8b2d6f4612c7d3d4b55c4a3651bf4ad8c39db163b",
      "rank_sha256": "08aea81b797095cd0eaa1f1e70fd571c6d819e0caa99f6dbcd021760492393a8"
    },
    {
      "subject": "health",
      "index": 326,
      "question_id": 6096,
      "sha256": "022d22124649e108b658b7a78030021c7cd535752832f6e051de7f092c6d18cb",
      "rank_sha256": "0fc7afd2c085f7ca5223f58aee159bb94f5c5b58c237283b3eaf5e37846cba2d"
    },
    {
      "subject": "history",
      "index": 63,
      "question_id": 4528,
      "sha256": "82943d23cd26e9674a83974a34a532c71b9fe18a6363e44204593873c98562a3",
      "rank_sha256": "15974ef6b6fb368d10b969f7a99b1631f6a420769e4f3503e192c7ee6020cb70"
    },
    {
      "subject": "law",
      "index": 767,
      "question_id": 1626,
      "sha256": "394e2d43fafe1a0cc66b8e7d33b7ec855526f75b24f8a9375614e40301442a93",
      "rank_sha256": "08c7328889ba9f3711ae8f747e376e6812c2eb6aa64cb322658887ce03c49942"
    },
    {
      "subject": "math",
      "index": 703,
      "question_id": 8004,
      "sha256": "97367599a9cf822e000640ebc7dc8d8909b302e86b20049ee65075e7e0dc481b",
      "rank_sha256": "04fd9273066183b296548ae645d4d2bd0dd5c64fa115dfe1daac0f71e2ca8350"
    },
    {
      "subject": "other",
      "index": 919,
      "question_id": 5765,
      "sha256": "db56e0767e82ef0f47a98e5aa0a3945c87c252daf502a844bb197c4679bcd788",
      "rank_sha256": "0ab3c2e8cbdbd092c7325e06c1cf5ed2eca13c2927656cd4b4a5d6ad295b0d08"
    },
    {
      "subject": "philosophy",
      "index": 292,
      "question_id": 10653,
      "sha256": "1a1b7590cef3a011eeb735f0f13b08db10df2f2507f921bd3c7a11d0886c3202",
      "rank_sha256": "12c9566735a9b8be8733f94050d3122fe52692fd5716f4cbf8fb3ffbbbb544b2"
    },
    {
      "subject": "physics",
      "index": 180,
      "question_id": 8832,
      "sha256": "ac8c02729b0ca8a17383ee756152784119ecd5c031db18b0950346ce36594292",
      "rank_sha256": "065c964222fd9ea558e578cb90d6e2d13a027e010a977657a1cbebd31b9b8200"
    },
    {
      "subject": "psychology",
      "index": 191,
      "question_id": 2009,
      "sha256": "9270eb9eb27d5ed702d921d3c7e8cc9049fe474fd22b7d426a04753772000463",
      "rank_sha256": "0d7c48cfd517f065673404ba243d9bbb7dfa43816d3bcd4ba97ce6d0c38c9510"
    },
    {
      "subject": "biology",
      "index": 661,
      "question_id": 3277,
      "sha256": "9f651dbd6e764f0c30201949b50e63cb43770a7c478174742e8f490c539e4b64",
      "rank_sha256": "100ef4430ed48f129656497e2bcb021e1eaed4c50be95159decab066145316fc"
    },
    {
      "subject": "business",
      "index": 315,
      "question_id": 385,
      "sha256": "3fd048d79773f65aeca27d8e2bb268e018a770a0e331989ad9148bfc8506d4a3",
      "rank_sha256": "0861f5cebe40405087e4a1500624d3aa1da17b01f048069373813c7bbcd2e2a9"
    },
    {
      "subject": "chemistry",
      "index": 401,
      "question_id": 3734,
      "sha256": "9e689eba81d10f640ea50d6bf683092d472bc64a28c3884a532f7ca5acb3885b",
      "rank_sha256": "08a4880c80327e749f7b485b41389f1ca3119a02095698ad8884be80e5907dac"
    },
    {
      "subject": "computer_science",
      "index": 158,
      "question_id": 10109,
      "sha256": "184303e9f7db03e0e78fdc452e5db0c4e1c652f275bc6a08c875b4b772914c3f",
      "rank_sha256": "1813a65e9ba57679b84ee066b7977293df500531d68edee456709fb8949d5fb1"
    },
    {
      "subject": "economics",
      "index": 52,
      "question_id": 6509,
      "sha256": "1d9b96ea47eedede5600f3c82fa3a693ea650e1954571fa66d8a1f79b9efcc06",
      "rank_sha256": "098b1d5a95fdebc7a0d5144fd6129bfc6ce5943a2f435664e0d48ca394e68979"
    },
    {
      "subject": "engineering",
      "index": 258,
      "question_id": 11118,
      "sha256": "fa6cd7952ae19f56f8f10b2dd6d488bcc7d79ca36ab3623ca6d42c244ea144a4",
      "rank_sha256": "08f8482c3d3ae7e71e5d44793d5d945df96edc9ae169871961301658755bb50d"
    },
    {
      "subject": "health",
      "index": 614,
      "question_id": 6384,
      "sha256": "9d6bb11d5ff5a7829afd7e619a023b5d6924f5fb69d4853115df91156257fc3a",
      "rank_sha256": "105b7b0f0d7360424b2896ca8198935327b5ea8b630ae35e0cbb581b0aa20c6c"
    },
    {
      "subject": "history",
      "index": 75,
      "question_id": 4540,
      "sha256": "3e02967d67b2c81f06a1b5011e03d5ae6f8b6ad15888e666bd392c26e9c45a0e",
      "rank_sha256": "172ac36ad87288b0d2bf13c358abe1f51ac09af638eb68ec8b915f59cc1216dd"
    },
    {
      "subject": "law",
      "index": 222,
      "question_id": 1081,
      "sha256": "0659a669702160600eb83f1e400f0fed13eb8698d2b86bd7e4925304587b5df5",
      "rank_sha256": "08d66329be164b17ab96e841643e204ebd216d072eadcd2e7620508a6078e3a0"
    },
    {
      "subject": "math",
      "index": 844,
      "question_id": 8145,
      "sha256": "7956c3587aa54880293ee9427fba31bc6d975c0489082be3eb7ff9628a918d07",
      "rank_sha256": "0520d3b89df50ad1a3ca573131beed759a03bcf347105c8c769df5d470d413d4"
    },
    {
      "subject": "other",
      "index": 200,
      "question_id": 5046,
      "sha256": "7b8ba81885b88cb7956c525d4a82f7163744440d8c58d58d90ca04dd5459f6e4",
      "rank_sha256": "0abce02dadfe6a884a916987f77d66ddc9b2698edb8d428135f2a261dbe7b07a"
    },
    {
      "subject": "philosophy",
      "index": 26,
      "question_id": 10387,
      "sha256": "64a7a4d8195adc14fba575428005e1610b4be83f0124e262c9134bdaf25a09cd",
      "rank_sha256": "12f210a95165c5d2dce87623cff6116e0f96531ca587ad0eedc37cd53a53abf4"
    },
    {
      "subject": "physics",
      "index": 932,
      "question_id": 9584,
      "sha256": "dd8c7eff35b2562da761ebcf8c844404b4fd0d42908cdbc6d667eb3da1c5bb8b",
      "rank_sha256": "06c482b840156687b495f22f8b0639ec74b9119df4f0f644152644b2fda11559"
    },
    {
      "subject": "psychology",
      "index": 116,
      "question_id": 1934,
      "sha256": "ae81d82a83dd81d381725ce13c19f6eb96379dc9d3027e1c5ed39a3421a3ff6e",
      "rank_sha256": "0d80b62651eb08ebbf0e0a35e09712d2c495dc7ac8c6d60a9175509da6ac2127"
    },
    {
      "subject": "biology",
      "index": 303,
      "question_id": 2919,
      "sha256": "ccc71c80853d79a1a0cb442b8018f4d3c84f082c473c9c7013c9de4f968332d6",
      "rank_sha256": "100f4ec9abb1396fe5a31a2ff97dee631423dcb85f4d5eabc9d5027d7ba9ac62"
    },
    {
      "subject": "business",
      "index": 471,
      "question_id": 541,
      "sha256": "5fd6ab67088c2fbbc9ff43a4b9f1e63d860b39422816a122aaefc30793a9c9ff",
      "rank_sha256": "08f49f173f18a7debec69430bb74282693442feb43d423364a7a662009e1b6af"
    },
    {
      "subject": "chemistry",
      "index": 813,
      "question_id": 4146,
      "sha256": "be196e2d7fb01efc6d437f39fc39529c79b590f229c52df82c86cf70270cbfd4",
      "rank_sha256": "08cdae064f07c0da0d092c1ce481acc286f62ca3618f8ac40433d6878f969783"
    },
    {
      "subject": "computer_science",
      "index": 24,
      "question_id": 9975,
      "sha256": "4d67f1d8921f327998eab8673df926bc68ed01637ab7b5513839f1384a6207bf",
      "rank_sha256": "181ad7c48def294b55403153e8d7a60090b3508e66172275b81a67e5ce646732"
    },
    {
      "subject": "economics",
      "index": 642,
      "question_id": 7099,
      "sha256": "be6dda3fb297e220a7c0b36c438d8a9bd83632d08c37db4c94fbc4a7666a481a",
      "rank_sha256": "09b715cc0c19897956540528cfdbeb61999e022ebc791fd243eeca237e6bbd8b"
    },
    {
      "subject": "engineering",
      "index": 49,
      "question_id": 10909,
      "sha256": "73e32a05d29f2e2508cabc16289fa37406f5a1e6ca83fbd587b9c65a53bdae22",
      "rank_sha256": "090a0107ff7643fab1204cc61f6bd1057a55f47ffdc20dc6550ec1150f519a25"
    },
    {
      "subject": "health",
      "index": 215,
      "question_id": 5985,
      "sha256": "21b32c45a775b16037d568df13d682e5e0cb220f21d7de1c2796c7754dc80c41",
      "rank_sha256": "10aac0d61bd6185c00e475b3da42d74011a8e4e02127c633e632cfb22ae734ed"
    },
    {
      "subject": "history",
      "index": 152,
      "question_id": 4617,
      "sha256": "ac23b072d22968bcc89c29356b41de65c0b18a7823a5069d8942ed3db3bff481",
      "rank_sha256": "1978a2bac4499c90854766aec2e3290aa2f429098a2f55a6592b9e81b7b407c2"
    },
    {
      "subject": "law",
      "index": 114,
      "question_id": 973,
      "sha256": "4be5371f1f890c7bd11611bdc23aae9724499c3c327da3a3da38f92931820ae4",
      "rank_sha256": "092d6bc007d82c6d3f19d6313d1db97604e96b9d1e514473eab89b4f57bbdb3c"
    },
    {
      "subject": "math",
      "index": 1015,
      "question_id": 8316,
      "sha256": "32082956dfe021656dee8afe908041d4adde66eb0e22394875fee264f875b42e",
      "rank_sha256": "05538cac17ce48850d444d7791a3167ffe0e822569b6415fb636c7bbefc67daa"
    },
    {
      "subject": "other",
      "index": 455,
      "question_id": 5301,
      "sha256": "74548a5f2bc433dbf047a7fab68872e776b4a7006ccffbd30f23e5588be15830",
      "rank_sha256": "0ae24dccfc9bf633071eaf8d249caf28f40b850e27b02723581dcfd52f157782"
    },
    {
      "subject": "philosophy",
      "index": 352,
      "question_id": 10713,
      "sha256": "b62b659ef8bda645853612ffad693403a0922c11ccb19858e1c14256403e5375",
      "rank_sha256": "1334a8db9b078c342fdffb0beb2b5bec5cb580d4e799eaa5cc0ad295f93c81a7"
    },
    {
      "subject": "physics",
      "index": 947,
      "question_id": 9599,
      "sha256": "2984c804f730ca5e97bd7281bf3e2eaf413450cc3d4a3761a0099f2850fb3dfa",
      "rank_sha256": "073ee37b8a3af7bc042f44bcac4bce631a77a3b1dc93cfd2f9e635f36013116b"
    },
    {
      "subject": "psychology",
      "index": 406,
      "question_id": 2224,
      "sha256": "f265fc800009c0d49d385444c091c8a5eda1d961a2054950f15aa319c1bc904a",
      "rank_sha256": "0d853cf6d225566a3e1c985ddd49a2f8af8b0f422a47f911ce7d5beec7434163"
    },
    {
      "subject": "biology",
      "index": 376,
      "question_id": 2992,
      "sha256": "0bc6b05b02811c64ff0615380cd1b7d4548056462536e1ac2d5f355329c44f07",
      "rank_sha256": "101488de590db9478905969aefd5ed1f7a05ab3522a0d4b26b27c2f79fac171f"
    },
    {
      "subject": "business",
      "index": 47,
      "question_id": 117,
      "sha256": "c9d04c0d060ee4d110ffcfce8de7852655e49b3075ce7a33c87ce1c272703eef",
      "rank_sha256": "09604c0537f428cc74a4ffe37ea1d17d266df3bed957ebf28f0e3169187acede"
    },
    {
      "subject": "chemistry",
      "index": 1010,
      "question_id": 4343,
      "sha256": "9eb73056d95f96b29864e4b8a008138a62d5d8420a4c275805ae7da32fa80de6",
      "rank_sha256": "08d60aa64a7772f63779322ccb3e000b0e5f272981fbd62c68fcc087e435f680"
    },
    {
      "subject": "computer_science",
      "index": 201,
      "question_id": 10152,
      "sha256": "7a2193750d4bee439664fefce296a9d3fd7162cd6a265266753da13fd61aa157",
      "rank_sha256": "191de1afdc4703283d66d4c1c2eb7647c4921ccfeedad95ee7609d3777d37bf0"
    },
    {
      "subject": "economics",
      "index": 349,
      "question_id": 6806,
      "sha256": "76771370b42c0c2d1eb37bb22a717e859160f3be66b8a3818a7018d14f3c3dfc",
      "rank_sha256": "09cdd7e0ac4f810e2829e461a496b0a1d723ed14a829e78f4cb79f88146f6154"
    },
    {
      "subject": "engineering",
      "index": 485,
      "question_id": 11345,
      "sha256": "a9f0749f8c761b3e0ce475dd9eae1da0d648aea61e0ffac53e0d4879a8e53a5a",
      "rank_sha256": "095b803ad88c863adb60c94c916b732d5af3cc496180667db879291a1a34e6b3"
    },
    {
      "subject": "health",
      "index": 195,
      "question_id": 5965,
      "sha256": "4b1118c474f3e8dfc7a92153d44fc3ff6565b9675fd14439edadcbeb6f565c72",
      "rank_sha256": "10c96cce67e98482421f443d10e1e98d50b57cf7889fdc8b7caad02a6440401a"
    },
    {
      "subject": "history",
      "index": 55,
      "question_id": 4520,
      "sha256": "29f02c7f49e736db5ec94ef82769f2ad3f8092f7e79f9ce8af06442cc0a90cff",
      "rank_sha256": "19daf131bb2312bec23381395ce8141b0bec1d0e0e64f42092da31b2852a6e42"
    },
    {
      "subject": "law",
      "index": 361,
      "question_id": 1220,
      "sha256": "44d54b9c427e317c6d47915a485e81ebc123e97256b06c6546478e87d23c3788",
      "rank_sha256": "09317e4ef767ed623821a20ca63f4f57d0657d45f958691efaabf06fca03913c"
    },
    {
      "subject": "math",
      "index": 293,
      "question_id": 7594,
      "sha256": "d70bc9a46fa518930b3fdc818cf7ec3fe7a64c8da2af7778257badcb8305f095",
      "rank_sha256": "05b9ddaa61025356dc365bb84a9c4869ba7781d31b918e19805f77ed90e2f97b"
    },
    {
      "subject": "other",
      "index": 771,
      "question_id": 5617,
      "sha256": "e7965fb34722e71aa16ccf2fd916e616de78c96565947188f3fba3b9cd4e345e",
      "rank_sha256": "0afb6ded4e560574581f3d443f4c6dbf677b259e2a4125ff8bbd9c078261a81f"
    },
    {
      "subject": "philosophy",
      "index": 246,
      "question_id": 10607,
      "sha256": "d5040c9e93db80356e0dd71ac061daebcf0f74f21d2903f3e55160e653f15943",
      "rank_sha256": "135a36369914b7ffa576bbdd3fb1fd8bf6aeb93670292bd6f3a79cac622610c3"
    },
    {
      "subject": "physics",
      "index": 610,
      "question_id": 9262,
      "sha256": "7d0ebb63db5415d4e21df01a5ab4ddb432ba228e7beadc0169392caffb956b97",
      "rank_sha256": "07b534476117a5101d75ba1bfba5006f918c6e5bc8118cd1882ef8b1d69be849"
    },
    {
      "subject": "psychology",
      "index": 794,
      "question_id": 2612,
      "sha256": "f02dc247d591155f4ff38a3a3d2765811e1e87bc9c4c24aa98c964530374518a",
      "rank_sha256": "0dc0abda4ce92622d074fb4a09357374bcfa768dd2f481740555682d08b5e630"
    },
    {
      "subject": "biology",
      "index": 69,
      "question_id": 2685,
      "sha256": "d2b9dc990b0f426e32f8024f0121743b908ab57cc32e72e910a7ecb92fa4759f",
      "rank_sha256": "106cbc5f7aaf967f8183629921288c47b15c7c080bf8ecbacc5e94ea6060f612"
    },
    {
      "subject": "business",
      "index": 215,
      "question_id": 285,
      "sha256": "ec9397fb9c1ac4da8f21cca3ca17bbe0004d7991b93de1d17fff262cc63f8eaa",
      "rank_sha256": "09b8cc4aed7e608d8f96d78bb47f2fc1b573becfc73a808ca78e66a9c8337a50"
    },
    {
      "subject": "chemistry",
      "index": 1099,
      "question_id": 4432,
      "sha256": "43cbb875a55b000c39fbfb083fb7a46de0a2797dac39a2ce952a16013b08f2e0",
      "rank_sha256": "090bb10c5eac9cd52558862ce821337f89d79225acbe20a7ac167f5a832ef992"
    },
    {
      "subject": "computer_science",
      "index": 21,
      "question_id": 9972,
      "sha256": "24806a446787d358c0f89e34dcac7498405d70a94a1dd2c2a3aeaf10f6a0520f",
      "rank_sha256": "196ad179d51c2dfa256d38802f9b221a8135f49b36239a7ec31e64b5f35e16f5"
    },
    {
      "subject": "economics",
      "index": 492,
      "question_id": 6949,
      "sha256": "18b2258c7d081141281d52fb83b197a5895d4c747a7f9350d5a7a7695cd14e2b",
      "rank_sha256": "0a94546fd98fb778c2e9ecb100ae3609131352502dd497e1d8a49d546bdd8e4f"
    },
    {
      "subject": "engineering",
      "index": 405,
      "question_id": 11265,
      "sha256": "ee780bc88e5548e62c77d47819882713a4138314b4f711461f89499aafa94644",
      "rank_sha256": "0964f968b949421d6e1cb51219994c9051401a30dd1723e5838555b2e6bf1912"
    },
    {
      "subject": "health",
      "index": 39,
      "question_id": 5809,
      "sha256": "1969a41f353f96cd9ed5a2d82286baf7e5d761d8405a1af43516ed0a6ba5ebea",
      "rank_sha256": "10f5fe13f3d475ba0ebfd4038ccd30b2f31e3546393ce6bc1ed7ae660ca2375c"
    },
    {
      "subject": "history",
      "index": 294,
      "question_id": 4759,
      "sha256": "6f39094d1660afb19ac6242038dcd761f6925fba7266ca174857edef662c9820",
      "rank_sha256": "1a5808f0488c6d7234e5d0394347f8c2e4f0f280b15a30a46a0a417d0c7cd236"
    },
    {
      "subject": "law",
      "index": 625,
      "question_id": 1484,
      "sha256": "0455c986c446a549ece99f40ff4dfc61da8ea10594743ba6da992303bf907b87",
      "rank_sha256": "0aba209e3242d2c42cb2b2a3cf71ccd6064cc4d2421ed425083f41537d11220d"
    },
    {
      "subject": "math",
      "index": 1146,
      "question_id": 8447,
      "sha256": "6355c0b6d3b481409505373cf17efb180472d7b2e4fdc875e69197575054162a",
      "rank_sha256": "05c2e19c23e739f46191197414ce1b8260b4c3413e82c10aaaa71f592444fc11"
    },
    {
      "subject": "other",
      "index": 407,
      "question_id": 5253,
      "sha256": "7c4eb08dd76cf14dded5a929f21bac30cfce511f6ab8b2d17b2cacbf1969929e",
      "rank_sha256": "0b7f8e80210d453349bf0bd53ab682bacc7b4f876615a4c06177a8a81e61840c"
    },
    {
      "subject": "philosophy",
      "index": 266,
      "question_id": 10627,
      "sha256": "7b9d3607333959d12cbfed5eb20ab350df751b9f04ff8704c5e51174a470c65c",
      "rank_sha256": "137bbbea545fa3beca4e65de9827627764a2cd420b142ecac5ca021800b5bdb4"
    },
    {
      "subject": "physics",
      "index": 151,
      "question_id": 8803,
      "sha256": "504b12515d1add2cda42e19f64c51e771786f93fa1f70b8cdf24892b00695ee8",
      "rank_sha256": "07b7ca6b36442d947e60491f8f7339b62c05f1ba32e645217a6c52a70ee6f3ab"
    },
    {
      "subject": "psychology",
      "index": 479,
      "question_id": 2297,
      "sha256": "ea9eb50ec3e29fa22873a1de15612fd2fb57156908dce86736de25173aee0599",
      "rank_sha256": "0deeeb9e5bc72622c56e5e2d0b31965bab6c5d9ca8415e3b870f1dc0df7b2803"
    },
    {
      "subject": "biology",
      "index": 511,
      "question_id": 3127,
      "sha256": "f22e5d43e1b47cc2e2c5615bde04f304e0c6b808b2231b0f72b97705c0954a08",
      "rank_sha256": "10c6df9491fefa039c68a5c2fe4820318aadda5758c8a2b27eea514745d49f24"
    },
    {
      "subject": "business",
      "index": 174,
      "question_id": 244,
      "sha256": "6c905bd88f1e03219c8351e3bfa7489ca29fc8587ac03b9313acb593043e8a0f",
      "rank_sha256": "0a30f03b6617f8f5ed5b226f48e98b3cc15116b3254f3836290f8e4c324ffcaf"
    },
    {
      "subject": "chemistry",
      "index": 187,
      "question_id": 3520,
      "sha256": "04fffabc4fc3aa5b38b09fc3a4a8b8a098d82b0270680d467599aa1231c0b5ee",
      "rank_sha256": "0a197959079b895358499724a9c02700ba78c1e223c8116d3781ad3a6f54ff63"
    },
    {
      "subject": "computer_science",
      "index": 312,
      "question_id": 10263,
      "sha256": "a8c251185905a2f62de53225b47cdd7ac1c4ba73f6cbc9937d97bf2df193a5b8",
      "rank_sha256": "1a06a6e00d65c1e2c4ac8931786dcfd08f6a8eced4258f9fb70ec3fcc088ca39"
    },
    {
      "subject": "economics",
      "index": 187,
      "question_id": 6644,
      "sha256": "3748737d0866b0106f0eee26fab621f669cd6f3905cc780fc52e5c9846caf06d",
      "rank_sha256": "0abbb37ff9d2cf7c43021570ba29057e5fce5b9746113a5182a37d3bd82870fa"
    },
    {
      "subject": "engineering",
      "index": 179,
      "question_id": 11039,
      "sha256": "92397ece48966ff00e9647007a40ba8a63cd1e65cc605d327913e12c2bb8d2f4",
      "rank_sha256": "096b509c6b52aa86d875e7f819fe87b4c19586b458fc9f5be1eb2780aa79b279"
    },
    {
      "subject": "health",
      "index": 529,
      "question_id": 6299,
      "sha256": "32109c99bbbb8b2ad41e1d10861bb2895d799075c717f0aa01d76fd676019643",
      "rank_sha256": "10f9d70cdfa7b55d46bfa6b883c3e2582b70f62c02c2feef988dfe8cfb24cdf6"
    },
    {
      "subject": "history",
      "index": 223,
      "question_id": 4688,
      "sha256": "6aed00f726927391afb7456e97dec210e96a17ef91145c3ce179b8db1d10fb5b",
      "rank_sha256": "1b8f2aeb5cd0eeb1fcc9a4ab472d36784b755d2d2390d108fcf2ad777a716971"
    },
    {
      "subject": "law",
      "index": 345,
      "question_id": 1204,
      "sha256": "633371aa8d9f3a3f5df375f652c87f050e2f9cd2074098375a8f5f9f692b09c5",
      "rank_sha256": "0b56f8e048fc5d96e469430873d4908319846fbf29599c62cf647dcf0c44044e"
    },
    {
      "subject": "math",
      "index": 828,
      "question_id": 8129,
      "sha256": "c9d7e95378feda18454162ae63ab68e180b504158585774cff48cde081866a4f",
      "rank_sha256": "05e4a4d50b925b6ae1bca64752de41ee07f6420bf702c85508e4231bfb3b83b1"
    },
    {
      "subject": "other",
      "index": 696,
      "question_id": 5542,
      "sha256": "7ff8a3f19e5fa29aad0f6c6ee7d5c20a33c51c2d72479965f43573b1abec02e7",
      "rank_sha256": "0be6103d2d3e9459af69883d58face38bf5dad81b0c632eea0c832d64a217bca"
    },
    {
      "subject": "philosophy",
      "index": 174,
      "question_id": 10535,
      "sha256": "52226e578a860e66b15891ff832366c24519b20e4757ded339411e45a8f7b9e7",
      "rank_sha256": "1438c4cbee74bf18383b82b59ff5e9757f4de436d660e9c9fc1a9161426c1602"
    },
    {
      "subject": "physics",
      "index": 168,
      "question_id": 8820,
      "sha256": "c734777d6e9ac9edc890d7b614e08fdb2b4b853afedc81c3b6b4d5e575304a20",
      "rank_sha256": "07e5ab24754ee17c45af083b3c2e57a04be41bcef1a3bb2f75d9192a11f1199b"
    },
    {
      "subject": "psychology",
      "index": 573,
      "question_id": 2391,
      "sha256": "05e64d5e13f85404b82b7dab3991d8d5996e9776623fc266ad37ca88380fd108",
      "rank_sha256": "0e189d0a9f3d2a92adb4e356858216898604cdbf61492537030443db25ba3db0"
    },
    {
      "subject": "biology",
      "index": 119,
      "question_id": 2735,
      "sha256": "c6481d8192cb6ec1f0126fd4fc1027e10585e8e85b45fa8e1785814de48a998c",
      "rank_sha256": "112556e575cb9540e4f07f6793b43bda7a1570c19f0dc051085d2834b7b0236a"
    },
    {
      "subject": "business",
      "index": 466,
      "question_id": 536,
      "sha256": "1ac622b8957c70fd18b554944cd66bb1fcb1501dc38a0e284c3e70438fb088bb",
      "rank_sha256": "0a442aa0f95fbc963b04d4d40c71ea54166577ff8627778cf947b297227b222a"
    },
    {
      "subject": "chemistry",
      "index": 828,
      "question_id": 4161,
      "sha256": "77dcf0ebdb125cde473e5560c088d62f3146917f42afeb19f36185396363890d",
      "rank_sha256": "0a1b9cbbab6dd1decc05e6c70b1ee146d77183b353cee0090527c9faa5e8538e"
    },
    {
      "subject": "computer_science",
      "index": 274,
      "question_id": 10225,
      "sha256": "edcee36928939b0d7f6a4bf077d5ebb7f9dd0b77252125beb1ae536126ec2f0a",
      "rank_sha256": "1a89fed5d90284b689973d7311274f1e1f635529aa3fe57816b7eb4bdb550ee1"
    },
    {
      "subject": "economics",
      "index": 269,
      "question_id": 6726,
      "sha256": "595807eb16c3747dd4efbcd3cb69eab1ca4032ba0b897c3c6514a8214dead3b6",
      "rank_sha256": "0adfec4a1e9103aecfb7bdc637ce449f81e66874ebda77e8b29c087489ae0117"
    },
    {
      "subject": "engineering",
      "index": 615,
      "question_id": 11475,
      "sha256": "14a69cb3952762916d5731cb0aa6117ad6bdd589694805a378f3d89c7226a52e",
      "rank_sha256": "09bc34a8087a369313619e07f47e553271df0d7451e32502f0d69b995d5769da"
    },
    {
      "subject": "health",
      "index": 373,
      "question_id": 6143,
      "sha256": "e703c4923c18de19bae9c5670d54c3ceb418c0032de42e4a43a0e455f3d33d28",
      "rank_sha256": "117269a4b30c8efdef1f79e3c3fc84de2e7c492b031ce85e89de064575993cdd"
    },
    {
      "subject": "history",
      "index": 334,
      "question_id": 4799,
      "sha256": "ab8c50b9bf85ad57ccc6fd60864a49da17dc303b6e57e240b5db7f189ed4533e",
      "rank_sha256": "1bdd29008a233b727c8dd25746a36e0d9c5d9e71d0e154eaa9c47a7ed072efb8"
    },
    {
      "subject": "law",
      "index": 284,
      "question_id": 1143,
      "sha256": "51c1af7dc86a4a9530fa24aa7e882c4240a277fa51ee263f6142ea0644efdee6",
      "rank_sha256": "0b737a0fbaa62e6cce49d55401df046b57ff6f257c5ec2c6a7c0b79b10cb0be3"
    },
    {
      "subject": "math",
      "index": 818,
      "question_id": 8119,
      "sha256": "00abc66a1fb379a6f651ada47018c5d23b69a4560fcf9c5f3a5a5513c8bd94cc",
      "rank_sha256": "05f9d8099c1b4ee9c16c9afe123b6be72ec1d36b7c42c66dd55b0bee7ea04e88"
    },
    {
      "subject": "other",
      "index": 274,
      "question_id": 5120,
      "sha256": "f7e6ec6eb3255bb9fb3fd36bc83cd048465e2046cbe82e6688f51d320bd4b805",
      "rank_sha256": "0c370e4a62387e13e33455c4355a8e50eaa6a51ad406564a7acefc032354d1ba"
    },
    {
      "subject": "philosophy",
      "index": 423,
      "question_id": 10784,
      "sha256": "8ea0cc41d44b54a7071b205dca8e1a60ee1eeb4f0ffefa0815f390961305254c",
      "rank_sha256": "159fa724a565fab5115133eb10a0ac07890c3c6cb4267e07205dec38d485c00e"
    },
    {
      "subject": "physics",
      "index": 536,
      "question_id": 9188,
      "sha256": "391228105e49f28a15b0470f9353c5478406feb750b65f40743166d92b25e500",
      "rank_sha256": "07ead221a46a8117023e36431cee2e268a25cf8d838854e6d532b804c31a7003"
    },
    {
      "subject": "psychology",
      "index": 788,
      "question_id": 2606,
      "sha256": "de8ea1b47beb4a9566c64af641eae09a9a94970ed135345fed0ad87e46fef3e4",
      "rank_sha256": "0e5acfc4bb82f95aeb45afdaedfd57c7dbd95baa8c69513e541180ca09762b65"
    },
    {
      "subject": "biology",
      "index": 652,
      "question_id": 3268,
      "sha256": "54423da79b52f0b62ec225c1d9e22ae4c7d7e7d329e0709653db0b7786c90c3b",
      "rank_sha256": "11371d5834aa26437c8682842f9959b1dfe8e1502cee2d664d1d6b805d8d91ee"
    },
    {
      "subject": "business",
      "index": 304,
      "question_id": 374,
      "sha256": "6cc82dc960c30ffa196ca8ec3da8465c65b852d9313d1ef11aab47720097144b",
      "rank_sha256": "0a8e94064386dcce5391ed12f8baa7b3954288be90bb5c4954c42215e1854b73"
    },
    {
      "subject": "chemistry",
      "index": 553,
      "question_id": 3886,
      "sha256": "1cd6448e173b8ee947041a24f948ff5f8fa9fc2aa463aee679cbd4542202d39c",
      "rank_sha256": "0a7fb54d6eebe4f2c474615528b5a741ac93851833f68b57ad0a18f34f63dc43"
    },
    {
      "subject": "computer_science",
      "index": 116,
      "question_id": 10067,
      "sha256": "dd6525e8249c9f0876cde36708b17420356ec128c07ffb83c0e2beb83fe48159",
      "rank_sha256": "1b401e59b4ac4cf4aac0025e8936d6496c5163c38118109221d89adaef03b9fb"
    },
    {
      "subject": "economics",
      "index": 3,
      "question_id": 6460,
      "sha256": "9417da86d3edbe90114650f15f76bcb69f0e9bb21795e9b437a0983786bc63b2",
      "rank_sha256": "0af75b32fff9eebb6267dd9dabeb7cc20859f59adad914e27a4f88183d290249"
    },
    {
      "subject": "engineering",
      "index": 403,
      "question_id": 11263,
      "sha256": "66fcd6c68d4e5d185b9297ca8b75f7a5e8779867f27e71545a30414a3f6fa141",
      "rank_sha256": "0a488f4efe448f45e8500d3d2424bfe62413dd7c6c9af38ba104b20a25460dbb"
    },
    {
      "subject": "health",
      "index": 137,
      "question_id": 5907,
      "sha256": "b03f620a3f09422954ca8d10f2a686ab1953c278ea5dc8db22df1ba9041ac306",
      "rank_sha256": "11d96300ec0b8bbe6e9ef0923cc42de966c6985346c1f1705d88374871c9279b"
    },
    {
      "subject": "history",
      "index": 70,
      "question_id": 4535,
      "sha256": "3592aafd715f6874a20ced6a96fe9a3cbced58051f3d46760376e46d6755cfb6",
      "rank_sha256": "1c3c470460e39a56eb2759f214f1f986a6f5dab6ad2f54ea809438ac57d96867"
    },
    {
      "subject": "law",
      "index": 66,
      "question_id": 925,
      "sha256": "25085bd5b617c50f5758f94294aef87ea96bfccc44c7b76f71f644a9acf4b037",
      "rank_sha256": "0b81c0f785ed1776f1aee0a407f3fa5c875feedf3e0b46ec9792093f788836cd"
    },
    {
      "subject": "math",
      "index": 1084,
      "question_id": 8385,
      "sha256": "2e8e474ae06f585c4d9db49077eb34e67aa4d5a0873ad0625182242eb1246515",
      "rank_sha256": "05fc2159a0fc681525914265599bb065a3f1e6dd1b835cb60555635061991fe6"
    },
    {
      "subject": "other",
      "index": 41,
      "question_id": 4887,
      "sha256": "13cc75dae58624285515a7fcf79e644c72aacfe839a5790bbbcb5898770b5898",
      "rank_sha256": "0c51f5c99f329f340bef33d0598ad392bc43e50862ac65e1e076a9b70b519c1c"
    },
    {
      "subject": "philosophy",
      "index": 137,
      "question_id": 10498,
      "sha256": "3fa6589096e0a3aa1110e97eaf0e9f5190d9139d5f02d9a2ed8a511ea36ec33a",
      "rank_sha256": "1673b23608809213a1a9aa9a1a7887d2261d389adc80e4d19eea59051c73dcee"
    },
    {
      "subject": "physics",
      "index": 514,
      "question_id": 9166,
      "sha256": "7e45690e5b868a9ba724357718cc3f949773150165f17b62bfa091cfb65484b3",
      "rank_sha256": "080ea0fea0c9e5393c1950214934f040285220cf333d67c87ba6fa9ee2b73837"
    },
    {
      "subject": "psychology",
      "index": 93,
      "question_id": 1911,
      "sha256": "ac1f92016c90082212328081f0d2e200b4d61262f9f7e8e487b3e973240be981",
      "rank_sha256": "0e7c25ea73990d554bfc64ca16e06371a7fcea70c044b42895cfcc2260ac6445"
    },
    {
      "subject": "biology",
      "index": 77,
      "question_id": 2693,
      "sha256": "46faeb77755ccacdc78c3f5ee75c36a2a0b9be9337a8f5a0a21e521b91a1250b",
      "rank_sha256": "1179f437c62b44026a799b5f84750ea94cdccb34497bf546cec0f179883218f8"
    },
    {
      "subject": "business",
      "index": 272,
      "question_id": 342,
      "sha256": "aa56cb1deba88d8c1c00daf7fca6e5d447f449682bd8aa0241a8661adb5a75b3",
      "rank_sha256": "0aee469cc378f5a052abd446579c325a0f16bf9e509fbc6471164016cc058026"
    },
    {
      "subject": "chemistry",
      "index": 84,
      "question_id": 3417,
      "sha256": "2f70d4dbdf85604174db67a2f8cae10a90ce0ce58be5941a365f2df303404161",
      "rank_sha256": "0a8cf44229d8768034c1331630d242668ce3ba8c7ac6087101096e015bc2934a"
    },
    {
      "subject": "computer_science",
      "index": 373,
      "question_id": 10324,
      "sha256": "857453dd4ebdcf4498e9751e8429ea70166081ff3a71ae29b66d12556d0b5045",
      "rank_sha256": "1c5a84324ddc89123ee418141004299e2b4fda83e07b1f2237956888b9134e3e"
    },
    {
      "subject": "economics",
      "index": 540,
      "question_id": 6997,
      "sha256": "6aca74fb52d24b57f8fc4b5fa89655c61cd8387023b2e9ec7b816b1de4caf9c5",
      "rank_sha256": "0afb671dd0e1842c0990868254d5eec4144a6531ecd0e8990231213288dfdbd3"
    },
    {
      "subject": "engineering",
      "index": 320,
      "question_id": 11180,
      "sha256": "3b4eddacea9b284d64a7cc159fb32e513f4367f222378fd86cb6634d02e2503e",
      "rank_sha256": "0aa905d58f82ef5c9c1bbe0d516b72bb833eee01231f99d221f9039eaa0c6df3"
    },
    {
      "subject": "health",
      "index": 557,
      "question_id": 6327,
      "sha256": "cf3867a4f6ce94486b4f3caf7cb72a98534851bd74a314b228c821adbe3698ba",
      "rank_sha256": "12c7ce96512ab16ee649c0d004729a8163d8aa15f2b8abeae4371713760f6bca"
    },
    {
      "subject": "history",
      "index": 187,
      "question_id": 4652,
      "sha256": "c9fdeaeefca0888c23ebda88c980739960839dc3079c865911510c38500a1a5d",
      "rank_sha256": "1e5de1092daafea3148ce1dc768667c8dabcc2d1836734bcf875b174237b9a65"
    },
    {
      "subject": "law",
      "index": 563,
      "question_id": 1422,
      "sha256": "ee293beb53f5f134eb624f7a46130b744c97b22595ec2d1cf0237fc990b47256",
      "rank_sha256": "0bc283271c5ca6cda84c91d3fc594cb6ae367b971a5061712b6a419a63a90272"
    },
    {
      "subject": "math",
      "index": 581,
      "question_id": 7882,
      "sha256": "32185229ccae0393fa976fc4606e192e9a17986e5c253109a0415044d4975118",
      "rank_sha256": "06054b9937aff8b8bb55784415bcd35813021a9b86b7177a16bf8295b18bf438"
    },
    {
      "subject": "other",
      "index": 34,
      "question_id": 4880,
      "sha256": "4f5f8585d0e003b86ebd9055f3e7dec141182152333b8133647c81a1c3983fb7",
      "rank_sha256": "0c673c3c5ca634b16cfd78c5fc5fb64115b42fa2e2080f86a53244ab3840ce59"
    },
    {
      "subject": "philosophy",
      "index": 139,
      "question_id": 10500,
      "sha256": "1c39ccfa3fcd53fb71cc3fb3a0c2b30b9ac98f143cec4d9fda2496aa8ce692b3",
      "rank_sha256": "18f283ebbba0ed4c064863fdd93dabe3b982dbcc2e0511c64a1f1d2d3f62da1e"
    },
    {
      "subject": "physics",
      "index": 819,
      "question_id": 9471,
      "sha256": "16a87186160d3278d7a694a0f110e418e6d53a1ae373242d5d3f03708585f3f5",
      "rank_sha256": "082af7259f96710454c34205d607c6ad6e4f368367ab10b7f292e905f667939c"
    },
    {
      "subject": "psychology",
      "index": 403,
      "question_id": 2221,
      "sha256": "391381cf50b18899c82d1b1ebd69dcf4aef43ca94322c453a8e02655979e49ed",
      "rank_sha256": "0e89881422149b06ccb5ab73e0b4960721ee10991cb0424bd80a1421f65470c7"
    }
  ],
  "excluded": [
    {
      "question_id": 70,
      "reasons": [
        "prior_harness_answer",
        "prior_lite"
      ]
    },
    {
      "question_id": 71,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 90,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 110,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 130,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 150,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 170,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 190,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 209,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 210,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 230,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 250,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 270,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 290,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 310,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 330,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 350,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 352,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 370,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 390,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 410,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 430,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 450,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 470,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 478,
      "reasons": [
        "reserved_calibration"
      ]
    },
    {
      "question_id": 490,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 510,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 530,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 550,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 570,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 590,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 610,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 630,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 650,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 669,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 670,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 690,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 710,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 730,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 746,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 750,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 770,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 790,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 810,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 812,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 813,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 830,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 850,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 870,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 890,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 910,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 930,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 950,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 970,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 990,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1010,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1014,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 1030,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1050,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1070,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1080,
      "reasons": [
        "reserved_calibration"
      ]
    },
    {
      "question_id": 1084,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 1090,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1110,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1130,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1150,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1170,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1190,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1210,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1230,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1246,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 1250,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1270,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1290,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1310,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1330,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1350,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1370,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1390,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1410,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1430,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1450,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1468,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 1470,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1490,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1510,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1530,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1550,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1570,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1590,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1610,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1630,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1650,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1670,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1690,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1710,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1730,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1750,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1770,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1790,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1810,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1830,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1850,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1870,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1890,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1910,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1925,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 1930,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1950,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1970,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 1990,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2010,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2030,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2050,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2070,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2090,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2110,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2130,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2150,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2170,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2190,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2210,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2215,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 2230,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2250,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2270,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2290,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2310,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2330,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2350,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2370,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2390,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2410,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2419,
      "reasons": [
        "reserved_calibration"
      ]
    },
    {
      "question_id": 2430,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2450,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2470,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2490,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2510,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2530,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2550,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2570,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2590,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2610,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2630,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2650,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2652,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2653,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2656,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2657,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2658,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2662,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2670,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2690,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2695,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2710,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2716,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2719,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2722,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2730,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2750,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2770,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2778,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2781,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2783,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2784,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2786,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2789,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2790,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 2791,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 2810,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2830,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2844,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2846,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2847,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2850,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2853,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2854,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2870,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2890,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2902,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2904,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2905,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2910,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2930,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2950,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2951,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2957,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 2958,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 2959,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2960,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 2970,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 2990,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3001,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3003,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3004,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3010,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3030,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3040,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 3049,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3050,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3065,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3067,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3070,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 3071,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3090,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3110,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3124,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 3125,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 3128,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3130,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 3150,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3170,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3179,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3184,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3190,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3210,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3216,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3224,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3225,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3227,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3228,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3230,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3231,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 3250,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3270,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3272,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3273,
      "reasons": [
        "prior_harness_answer",
        "reserved_calibration"
      ]
    },
    {
      "question_id": 3280,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3283,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3286,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3290,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 3310,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3318,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3322,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3324,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3326,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3328,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 3330,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 3350,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3370,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3390,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3410,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3430,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3450,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3470,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3490,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3510,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3530,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3550,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3570,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3590,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3610,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3630,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3650,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3670,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3690,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3710,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3730,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3750,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3770,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3790,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3794,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 3810,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3830,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3850,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3870,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3881,
      "reasons": [
        "reserved_calibration"
      ]
    },
    {
      "question_id": 3890,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3910,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3930,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3950,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3970,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 3990,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4010,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4030,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4050,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4070,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4090,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4110,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4130,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4150,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4170,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4190,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4210,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4230,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4239,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 4250,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4270,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4290,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4310,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4330,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4350,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4370,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4390,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4410,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4430,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4450,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4470,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4490,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4510,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4530,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4550,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4570,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4590,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4592,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 4610,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4630,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4650,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4666,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 4670,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4690,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4710,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4730,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4750,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4770,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4790,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4806,
      "reasons": [
        "reserved_calibration"
      ]
    },
    {
      "question_id": 4810,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4830,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4850,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4870,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4871,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 4890,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4910,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4930,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4950,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4970,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 4990,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5010,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5030,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5050,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5070,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5090,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5110,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5130,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5150,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5170,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5190,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5210,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5230,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5250,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5270,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5290,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5310,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5330,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5350,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5370,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5375,
      "reasons": [
        "reserved_calibration"
      ]
    },
    {
      "question_id": 5390,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5410,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5430,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5450,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5470,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5490,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5510,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5530,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5546,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 5550,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5570,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5590,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5610,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5630,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5650,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5670,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5690,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5710,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5730,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5750,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5770,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5790,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5810,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5830,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5845,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 5850,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5870,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5890,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5910,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5930,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5950,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5970,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5990,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 5992,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 6006,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 6010,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6021,
      "reasons": [
        "reserved_calibration"
      ]
    },
    {
      "question_id": 6030,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6050,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6070,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6090,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6110,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6130,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6150,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6170,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6190,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6210,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6230,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6250,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6270,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6285,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 6290,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 6310,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6330,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6350,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6358,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 6370,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6390,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6410,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6430,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6450,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6470,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6490,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6510,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6514,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 6521,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 6530,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6540,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 6550,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6570,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6590,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6610,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 6630,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6650,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6670,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6690,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6710,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6730,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6749,
      "reasons": [
        "reserved_calibration"
      ]
    },
    {
      "question_id": 6750,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6770,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6790,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6810,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6815,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 6830,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6850,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6870,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6890,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6910,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6930,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6950,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6970,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 6990,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7010,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7030,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7050,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7053,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 7070,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7090,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7110,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7115,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 7130,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7150,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7170,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7190,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7210,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7230,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7250,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7259,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 7264,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 7270,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7290,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7310,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7330,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7341,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 7350,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7353,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 7370,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7390,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7410,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7430,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7442,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 7444,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 7450,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7470,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7490,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7510,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7530,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7550,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7570,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7590,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7610,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7625,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 7630,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7650,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7670,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7690,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7710,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7730,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7750,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7770,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7777,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 7790,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7810,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7830,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7850,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7870,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7890,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7910,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7930,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7950,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7970,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 7973,
      "reasons": [
        "reserved_calibration"
      ]
    },
    {
      "question_id": 7990,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8010,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8030,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8050,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8070,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8090,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8110,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8130,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8150,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8164,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 8170,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8190,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8210,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8230,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8250,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8270,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8271,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 8290,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8310,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8330,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8350,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8370,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8390,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8410,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8430,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8450,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8470,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8490,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8510,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8530,
      "reasons": [
        "prior_harness_answer",
        "prior_lite"
      ]
    },
    {
      "question_id": 8550,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8570,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8590,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8610,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8630,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8650,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8670,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8690,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8710,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8728,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 8730,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8750,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8770,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8790,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8810,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8822,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 8823,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 8824,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 8830,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8850,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8870,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8890,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8910,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8930,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8950,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8964,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 8970,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 8990,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9010,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9030,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9050,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9070,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9090,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9110,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9130,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9150,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9170,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9190,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9210,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9230,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9236,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 9243,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 9250,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9270,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9290,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9310,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9330,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9350,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9370,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9390,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9410,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9430,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9434,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 9450,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9470,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9484,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 9490,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9502,
      "reasons": [
        "reserved_calibration"
      ]
    },
    {
      "question_id": 9510,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9530,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9550,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9570,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9590,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9610,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9623,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 9630,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9650,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9666,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 9670,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9690,
      "reasons": [
        "duplicate_test_question",
        "prior_lite"
      ]
    },
    {
      "question_id": 9710,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9730,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9750,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9763,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 9770,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9790,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9804,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 9810,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9830,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9850,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9870,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9890,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9910,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9926,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 9929,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 9930,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9950,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9970,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 9981,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 9990,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10010,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10030,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10044,
      "reasons": [
        "reserved_calibration"
      ]
    },
    {
      "question_id": 10050,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10070,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10086,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 10090,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10110,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10130,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10150,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10170,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10190,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10210,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10221,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 10223,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 10230,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10250,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10270,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10290,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10310,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10330,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10350,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10370,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10390,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10410,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10430,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10438,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 10450,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10470,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10485,
      "reasons": [
        "reserved_calibration"
      ]
    },
    {
      "question_id": 10490,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10510,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10530,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10550,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10570,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10590,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10591,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 10610,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10630,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10650,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10670,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10690,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10710,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10730,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10750,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10770,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10790,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10810,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10830,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10850,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10870,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10890,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10910,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10930,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10950,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10970,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 10990,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11010,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11030,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11050,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11070,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11090,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11110,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11130,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11150,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11170,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11190,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11210,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11230,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11250,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11270,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11290,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11310,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11330,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11350,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11370,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11377,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 11390,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11410,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11430,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11450,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11470,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11490,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11504,
      "reasons": [
        "duplicate_test_question"
      ]
    },
    {
      "question_id": 11510,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11530,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11550,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11570,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11590,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11610,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11630,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11650,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11659,
      "reasons": [
        "reserved_calibration"
      ]
    },
    {
      "question_id": 11670,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11690,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11710,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11730,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11735,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 11750,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11767,
      "reasons": [
        "prior_harness_answer"
      ]
    },
    {
      "question_id": 11770,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11790,
      "reasons": [
        "prior_lite"
      ]
    },
    {
      "question_id": 11810,
      "reasons": [
        "prior_lite"
      ]
    }
  ],
  "schema_version": 1,
  "id": "chat-5shot-fp8-expanded-v1",
  "builder_sha256": "9115a27f877401231a480d0e8925931f6c67b706476e6f568dbbfea6f632d527",
  "dataset": "li-lab/MMLU-ProX",
  "subset": "ru",
  "dataset_revision": "8e6106a6c6ce1c5027e66cc338143cf997b2aa09",
  "dataset_files": {
    "ru/test-00000-of-00001.parquet": "00fbd67cab968df4f7f136235d6c9f8c33fb4044933aa2f5c15b0423a1eb7dbf",
    "ru/validation-00000-of-00001.parquet": "22102323dcb75063ff2f23085a5ab444cabe888305e7f70ae613135f989f15cc"
  },
  "seed": 20261007,
  "per_subject": 40,
  "selection_rule": "ascending canonical JSON SHA256(seed,subject,question_id)",
  "order": "rank round, then alphabetical subject",
  "normalization": "NFKC, whitespace collapse, casefold",
  "budget_gpu_hours": 12,
  "exposure_sources": {
    "lapa/data/mmluprox_ru/test.jsonl": "14c5736f06f7e99896288f7a5e7317050005b81d086a4b288ccbe3e58d62d1f0",
    "LAPA_MMLUPROX_smoke_20260924T133439Z_0d88b2e3.zip": "e288a9ce113e76980b6753044ad515ed09cab13eeaf8bc9edf0894beaefa7647",
    "LAPA_MMLUPROX_portion_failed_20260930T070726Z_19bceb09.zip": "a4e9b74c2e5cb332a63b45f7bddbf7296106171015ac9f679249ca2df5cc55bb",
    "LAPA_MMLUPROX_debug_20260930T143103Z_3d55eed4.zip": "453fdee00d597a855cb20eff1f8601250f3f0a99e0ab654503d756da2890a04e",
    "LAPA_MMLUPROX_manual_20260930T143125Z_e0734cb8.zip": "d850ca0bfe304c85ce0cfe1e13eae1c239217a7437b892595f9913003927acd6",
    "LAPA_MMLUPROX_manual_20260930T144500Z_e0003078.zip": "683518b4fafd5114c18521fe56138d0b2b5d6c5b255d1b84a172333a874755f9",
    "LAPA_MMLUPROX_progress_20260930T145758Z_aa01bc2a.zip": "d67f0865a795daf5709918c2ab4e30f78c702ec001be3c4234e24742da17e38e",
    "LAPA_MMLUPROX_baseline_20260930T175106Z_9a7146f9.zip": "7d857e967f4d263114cb4b3247647b3036834582f5ea4d9cc027f52d185e826d",
    "20261007T123126Z_60b69041_f63890cb.zip": "894a64bf4d1f94f5d6723c5e0b41341c8684eed7469205264e8d702f8f08eee4",
    "20261007T125518Z_030918b4_518ada8b.zip": "0f2aac645df55f47c7715bfb10d7d8ac39fc6f46b1cc45f7a0232ce06d493b4b",
    "20261007T155728Z_92e7ef35_aac4c7ee.zip": "7104decef24f9f29c885a82bbf1b6d8bd73e1d00a296cd79fd481fa90754cf01"
  },
  "calibration": [
    {
      "subject": "biology",
      "question_id": 3273,
      "index": 657,
      "sha256": "e9f216bfdd60cbb2cb91a2654cff5c21497bb062046a7b6eae20573613f376d3"
    },
    {
      "subject": "business",
      "question_id": 478,
      "index": 408,
      "sha256": "0ceee92224cb1212dc5cc5d8f91687571d7210d22afceed0663b56d8db2e7c5a"
    },
    {
      "subject": "chemistry",
      "question_id": 3881,
      "index": 548,
      "sha256": "eb8562cf90fef498a87b06795c8b43384e1ac07ee09f5543abe081bcc9bb2f89"
    },
    {
      "subject": "computer_science",
      "question_id": 10044,
      "index": 93,
      "sha256": "1f5df94a308a053e1f81f655eda8c0434adfd8ff40fba7566ce32b93eae0b6f8"
    },
    {
      "subject": "economics",
      "question_id": 6749,
      "index": 292,
      "sha256": "78f865f5d671bc77f09a75aebf042b929ff023f842419c1abfb938fc46349b44"
    },
    {
      "subject": "engineering",
      "question_id": 11659,
      "index": 799,
      "sha256": "fdd3f999058ebfeb59091471ac5c0f722849b01853b73ddbe5220cc183a78e66"
    },
    {
      "subject": "health",
      "question_id": 6021,
      "index": 251,
      "sha256": "e9cd343801e6cf3da9c676c88f67ae0b9ab673f59152975528d68599a7f50151"
    },
    {
      "subject": "history",
      "question_id": 4806,
      "index": 341,
      "sha256": "bd97117397a29c46bdd8685d59509102e9f0785ea21c1cc61c64b2cd69ee27d1"
    },
    {
      "subject": "law",
      "question_id": 1080,
      "index": 221,
      "sha256": "fce9733331ef41251f444ea2c51c46e3c783eb494636cc63819fca8b82e132dd"
    },
    {
      "subject": "math",
      "question_id": 7973,
      "index": 672,
      "sha256": "dbe21c95dfe7ca0bd3afcbd3a13a3dd1cf0c2f98656e4f551cb59eb87c983f8c"
    },
    {
      "subject": "other",
      "question_id": 5375,
      "index": 529,
      "sha256": "3ca03e5c35c22c3d3cd2f2b40c016848f712ee9af39c4756f47c25a2ab74dd32"
    },
    {
      "subject": "philosophy",
      "question_id": 10485,
      "index": 124,
      "sha256": "fbac2683998236a4a08798844f53c06d42740f906d53b3903c095bf04cb72f4f"
    },
    {
      "subject": "physics",
      "question_id": 9502,
      "index": 850,
      "sha256": "92b10e8627876818b3beaeb68b95e3ee99707439bd7249166541eee356e3c318"
    },
    {
      "subject": "psychology",
      "question_id": 2419,
      "index": 601,
      "sha256": "0f32df6e46138d7de4fd61fbfa975789b8dce653e5fc84a2aa874d1c12de8760"
    }
  ],
  "demonstrations": {
    "biology": [
      {
        "question_id": 20,
        "sha256": "69b6e2cb57393ddcaaeef15b92680be213e24f57f2e30fa0eeebf5d4cbb42db5"
      },
      {
        "question_id": 21,
        "sha256": "4100ef35443e2d04926f062e38f8c2e109aad7d3e5017738095443f292328830"
      },
      {
        "question_id": 22,
        "sha256": "5de289ab16e93824d192583326f79cdb9381c0feb4a7d6c9827e594b03d14254"
      },
      {
        "question_id": 23,
        "sha256": "d477de323babf793a9dd16f2c63a2978c9fb0bcc90cbcfc927091131e51e960c"
      },
      {
        "question_id": 24,
        "sha256": "6b03e8e7b8dfd124109f87e9f77595e8c99cca876763347cb5d5ee6cba944d00"
      }
    ],
    "business": [
      {
        "question_id": 15,
        "sha256": "0d27d6fa2fecb38c87f9d9978299c18189f7d0608889e4e0fcb56917aab431e1"
      },
      {
        "question_id": 16,
        "sha256": "6379c85ce3553ded59fea67dafc8e061a71450714b427759e59d85839761a76f"
      },
      {
        "question_id": 17,
        "sha256": "9e1cb3638dffcd3ea3f39ee11182f2204b6d45a3b9f51a424f2f6ab80b1d4ea1"
      },
      {
        "question_id": 18,
        "sha256": "c470e2c52752fb603bb33c3c5cf42640286ffcaa27624efa006a030d4deba8f6"
      },
      {
        "question_id": 19,
        "sha256": "5ff8bebf0f09cff13d82ea91ded9ee1d75331c407de35e08293e57e62282ef6c"
      }
    ],
    "chemistry": [
      {
        "question_id": 25,
        "sha256": "cb4d00b3ead4eca572c85958a6ec97301204b6d9cdd14a62bba493d42c7f5e8b"
      },
      {
        "question_id": 26,
        "sha256": "4a7c7552e9f109074421a83d25d361d77e049a260f88ba3e56d291e54a0d2925"
      },
      {
        "question_id": 27,
        "sha256": "90b1edcbe9636a8870a6e0c7c4a9ed59eaf49d49fb9fd2e8b0dcb9b901b8326f"
      },
      {
        "question_id": 28,
        "sha256": "0e386b9ca94dbeaa52e0f856e1c122cf4bd0e8732c87694eb58a7e0646992e5a"
      },
      {
        "question_id": 29,
        "sha256": "8580b7f66cbc5154b9199167dd33306b0c1daef78198b181d6d88806e161b530"
      }
    ],
    "computer_science": [
      {
        "question_id": 30,
        "sha256": "1ec5dab72781ea0c14a6227dff6d3e18515b265518926b617cdcadd88e25ffec"
      },
      {
        "question_id": 31,
        "sha256": "48ee6654d66d52a78712c47742c6dde964fe244ef455253de96c8127aab79a7d"
      },
      {
        "question_id": 32,
        "sha256": "fdde1aae2d9e3c4af5e03f5587f1fe13ff8f81aaa5ef8b229a15fec46a01e645"
      },
      {
        "question_id": 33,
        "sha256": "e6a6573f75ddb9adc3c13dfb86d9c064dbf4f8692bc1da31c8ccba268e759983"
      },
      {
        "question_id": 34,
        "sha256": "ca1e4f8de1fbab81319b9e7f2e7de38545d970e62842b4444f0197a4853524ce"
      }
    ],
    "economics": [
      {
        "question_id": 35,
        "sha256": "337afa4916d7740a60a1a62399f429877eec1c3c7b7bbc600432171c0681c977"
      },
      {
        "question_id": 36,
        "sha256": "b87e5f3b5052698c051b0e3db6ec601384a4dae644aeef741b010c2e2821639c"
      },
      {
        "question_id": 37,
        "sha256": "ca33bf5426ccc5247bd304a04b83c6cf9d645e36af47e40fa201bdb0789dc485"
      },
      {
        "question_id": 38,
        "sha256": "490e576d6f99103ff21b7a71d646b21fbda20f049c01a366033de51c057676a5"
      },
      {
        "question_id": 39,
        "sha256": "11bac6ff7445110e610a201911439b8db539c9d4041355014ac93554b916df3a"
      }
    ],
    "engineering": [
      {
        "question_id": 40,
        "sha256": "4f5467a90517c773ea042dbe1619338a8379a403d4e603ac827fbf9a87d3f4c9"
      },
      {
        "question_id": 41,
        "sha256": "e08999dece2097ccff66ee93eb7395fdcb794de91ba31516597d7232c14bc791"
      },
      {
        "question_id": 42,
        "sha256": "a41c80ec2fb237088916fd95d6bb7a538522cdfbf912c23ac983df8af64ba8ca"
      },
      {
        "question_id": 43,
        "sha256": "d9b76832569e0a0d4ab2fa30545220dc8efcf83cdaab72cf182146fdf68062bd"
      },
      {
        "question_id": 44,
        "sha256": "0d2a31625c4ff2310d7569445a6ebfab79b8a43f289ec2214a5342c9a0b53590"
      }
    ],
    "health": [
      {
        "question_id": 5,
        "sha256": "08bd44c53a090694baa227cbc0a279d23ccbc34382f2fe4f819cda1bccfac90a"
      },
      {
        "question_id": 6,
        "sha256": "0e76c50cfe835b546e728270b46082c67d0d806483a49faf8d0b29bb4dab58db"
      },
      {
        "question_id": 7,
        "sha256": "2c1ea397736c379e8c7b5452adbaa1e2e7e007d6b7b098ef5aad2b9600fa72b4"
      },
      {
        "question_id": 8,
        "sha256": "52cf270bc917695b0afec0bd666ddcc59e8b7cbe2ccae65331985e1e345e43b6"
      },
      {
        "question_id": 9,
        "sha256": "5af7a6a378b61670f393a420126f62e68b90a2de015fd5705ca6a1e78224a981"
      }
    ],
    "history": [
      {
        "question_id": 55,
        "sha256": "03910dee5e03c4665073e313f55b75523bba40b61b3dd95c6a7dc0725f7de20f"
      },
      {
        "question_id": 56,
        "sha256": "9ec32acdd0b3cd3ae4a5dce639bc2f803631ed1a62d42609c08392bc9b5f7d67"
      },
      {
        "question_id": 57,
        "sha256": "2375a0e4023da76f0a90cb7c5a04c4ed723e4f54a8ca9ffb3bd3761939026060"
      },
      {
        "question_id": 58,
        "sha256": "201f06ad7eab68ca8cd952cffa941efe97c58158853c785c1eea146edd4def07"
      },
      {
        "question_id": 59,
        "sha256": "28f1166fd764e2950661306b4e6cf76821f8c6dc2b7f17f1b5175b078eef6e80"
      }
    ],
    "law": [
      {
        "question_id": 65,
        "sha256": "4f04168ebdb29a7410bc981cf8f60f52e296a1c87a80662f50a181827addfcf1"
      },
      {
        "question_id": 66,
        "sha256": "98e31bf2a22fd4c84845c0ea419f860835642612efd908820d45affe97659597"
      },
      {
        "question_id": 67,
        "sha256": "2e09706b5bfec318a3682a60a0edc09a610bd80540b8ec79c78d74d0f558ecbd"
      },
      {
        "question_id": 68,
        "sha256": "77550ad5bc104a34daad5baca31f471f2c005870536196b8276f38c4f71ee441"
      },
      {
        "question_id": 69,
        "sha256": "b231753ed38b68bf1cb9865bcda580a4078177992a94a221ec30488785939d9b"
      }
    ],
    "math": [
      {
        "question_id": 0,
        "sha256": "0a48ec20b9eb38403cf8c457fe671d69b310ac797a0a3b3278c57a5f063579d2"
      },
      {
        "question_id": 1,
        "sha256": "9809ac4f335512d357e00f72c091fa7ef78e8e67ea10da062b2c8204824d5330"
      },
      {
        "question_id": 2,
        "sha256": "01251ce6f86f7cfe83309805a3f73e47f584e2312f2b3a3f7fc61ab4a72ac128"
      },
      {
        "question_id": 3,
        "sha256": "d1680c312847f0862ac9f5ad9cca7ead0333f0a7e7a81086d998947eca537518"
      },
      {
        "question_id": 4,
        "sha256": "32c1b8a582e4a90feb3205bdc505d9f0a868f92236252551f0833050edac380c"
      }
    ],
    "other": [
      {
        "question_id": 50,
        "sha256": "59a7e44ffafe17c6b89fc7b02a55e3f3c179a4f6785ae2315e1c527e17193575"
      },
      {
        "question_id": 51,
        "sha256": "58aa32714ce8e869b5f97075fc760b0dc76bfaa6009e79c06b93d4227de375ff"
      },
      {
        "question_id": 52,
        "sha256": "ae946a1503559238b306fc0c013459f2962051deab81e8b9f32ff4889ecdee43"
      },
      {
        "question_id": 53,
        "sha256": "ee03c47f31aa934d1eba8b7512d9fe31ac192a38364e59dff0ec63e27835b6f8"
      },
      {
        "question_id": 54,
        "sha256": "c152c39d3472246bbcdf5abce006fe1b00b915f502c0e6b9a95a116630ce40b2"
      }
    ],
    "philosophy": [
      {
        "question_id": 45,
        "sha256": "0877511c6b104ef2f5b23949843041573aaa104b745b60779b04c16613b4c57f"
      },
      {
        "question_id": 46,
        "sha256": "7ae37e0b1b24a1be72ebe03a8fc1055849657c22bf705aabcfb426aa3800f608"
      },
      {
        "question_id": 47,
        "sha256": "374a0f5bce02ab7e56d65c4487cb76a97c3d5083f8fe85b7b9ee0ca2943c1151"
      },
      {
        "question_id": 48,
        "sha256": "b7787f96bc618cd2294a8d0b146e684423a197925d23756b275f0c35237968fb"
      },
      {
        "question_id": 49,
        "sha256": "597d247642b76e1df311933d0b12c605b37a94bbc95a07e5540f90246c518acc"
      }
    ],
    "physics": [
      {
        "question_id": 10,
        "sha256": "5d5cf6a356742ecfcb1f4e26f06a2047e70b556a1602fbd7b58e9eff925e23c0"
      },
      {
        "question_id": 11,
        "sha256": "8db7056d5dcc45e6d197fb7a26b029acb84cae97db0ad2c3c077180462a18092"
      },
      {
        "question_id": 12,
        "sha256": "6287b45474de5c320d8988d20d1709ca3231720e249643981d5dd2804ca8cbc5"
      },
      {
        "question_id": 13,
        "sha256": "6088d123e5ca87797e367ce4f1476c9436cc9c1db923890de9256645a07da06d"
      },
      {
        "question_id": 14,
        "sha256": "5f32029da27cf01b69269aa7f1a6a74087712aa177ad6f7020bb2d14e3439707"
      }
    ],
    "psychology": [
      {
        "question_id": 60,
        "sha256": "3da7a0e5f5bee6e2dbd818f08df95058d4097bc9b607692aa456db28432d54b6"
      },
      {
        "question_id": 61,
        "sha256": "529f5b792a77051942eddb32b254755c048eb3699b1fa754576c035fa966f503"
      },
      {
        "question_id": 62,
        "sha256": "59c6c72600be61e59f436c97e697e15a023dbedc01d082a39384aa5cabb5de93"
      },
      {
        "question_id": 63,
        "sha256": "65e200bf947c6948df3e2e971b890e0e7520c58d4606ddd810bd4ffd97abd058"
      },
      {
        "question_id": 64,
        "sha256": "137fdaac1d958cd074fd13f42ced0aca0485c6574107b4535ec2e9092e0f7756"
      }
    ]
  },
  "configuration": {
    "model": "Qwen/Qwen3.8-27B-FP8",
    "model_revision": "017b9c7af6b5689d5dd426a76e0bc077eb5ca20a",
    "tokenizer_revision": "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0",
    "harness_revision": "d6de81643928d653435c431bae19945d41d32520",
    "sources": {
      "lm_eval_tasks/mmlu_prox_ru_chat/_mmlu_prox_ru_chat.yaml": "852bf00302ac68b796a07121646afeb1c054f165b1b7e1585c89a7a549193669",
      "lm_eval_tasks/mmlu_prox_ru_chat/_ru_chat_template.yaml": "ba71b82a8872d81e841ca566cf1228fbf6bac288963c40669112c4d241156093",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_biology.yaml": "5c5652af779b6f76263c8b2c50d1e556b49ed9ae135a569de7a729f819d2db23",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_business.yaml": "c3ee2987b8742f81fa4eb59262d3bd247a65efa6a9cb2a75ed7d75427bf08d75",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_chemistry.yaml": "5be779e17271bb81d0ebe8702a24c3ac399287d0df4836ab81524ea6a098257a",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_computer_science.yaml": "1816af75a73635c7705c35aef3072f15a22ee54aa305819880927dcf7e3a9ae5",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_economics.yaml": "ef1fcf83eadc67e89bb3b4381bad0eac16585c3bfc266bf5d124f99d4030d487",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_engineering.yaml": "fe1bcfa0009c42c7998c37ede092bcf3bb9b896dbbc5885980f92a89ea8f05b5",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_health.yaml": "80e6d0e0613752a2454df265c58f34f9448c4791e347893abda72305e6fbc64a",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_history.yaml": "047c600f10c0b38e0b3eff2c428079793c3d89ecd2bcc1b43019df6c83f84956",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_law.yaml": "7679544fafe2ada9c1e667e4adc13db1f319f420ecb97908ef48198737ba2289",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_math.yaml": "af96df4b28b32daa1846bc0f29264709ebe2e7a7c4789f137fa900ae61d49694",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_other.yaml": "87a294c66ccf38044a6016ad0ae0c5cb5a4c3f092ebd2c40dbdc6c4de3ff1580",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_philosophy.yaml": "fb0599414d22276942170f10fc4238a590e4981d04be08ceae9202af54fe0fe2",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_physics.yaml": "bfb64e6779bef91b5092dea22efee61f8ebf1e7888861558bd308a3a6d4e7196",
      "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_psychology.yaml": "cb4802cc2f6f07283d8a433cd55512d436a34d653d6ee5d09f1e0cf561e6a751",
      "lm_eval_tasks/mmlu_prox_ru_chat/utils.py": "99d10756ca88eee8fa3ff9fce3a4d4efb746b17a7a9b9feb7fcfbb76b15e5f3a",
      "nf4_reference.json": "500024adaad1814e3e38ba737ec37ebd19010f73055cb4b4ded25262d76fcfbc",
      "requirements.txt": "0d8268adba83ce6c9062c5085e76c516a7e0b79e33791e33dd7686a452708052",
      "baseline.py": "11a0b7d0d8c61e04c853e899d0db27bce5cbb993270f477215bc94a19362668f",
      "run.py": "ece2f0033cbae9db06a8aa69f4daf893c6d3ccd1829c160bcc285c824f8aa0d9",
      "model_sources.json": "ce5701c04db49306083db628881b4da6fa630c518ddcfec395f43d7505f18f87",
      "protocol.md": "439d1e186de9150c614c7fbce8eb21231574b04da2b3579f155bdaf388f4247e"
    },
    "shots": 5,
    "chat": true,
    "thinking": false,
    "max_gen_toks": 2048,
    "max_model_len": 32768,
    "batch_size": 1,
    "temperature": 0,
    "seed": 1234
  },
  "primary_metric": "mean of 14 subject accuracies (equal strata)",
  "scope": "eligible unexposed RU pool, not full benchmark score"
}
''',
    'source_manifest.json': r'''{
  "lm_eval_tasks/mmlu_prox_ru_chat/_mmlu_prox_ru_chat.yaml": "852bf00302ac68b796a07121646afeb1c054f165b1b7e1585c89a7a549193669",
  "lm_eval_tasks/mmlu_prox_ru_chat/_ru_chat_template.yaml": "ba71b82a8872d81e841ca566cf1228fbf6bac288963c40669112c4d241156093",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_biology.yaml": "5c5652af779b6f76263c8b2c50d1e556b49ed9ae135a569de7a729f819d2db23",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_business.yaml": "c3ee2987b8742f81fa4eb59262d3bd247a65efa6a9cb2a75ed7d75427bf08d75",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_chemistry.yaml": "5be779e17271bb81d0ebe8702a24c3ac399287d0df4836ab81524ea6a098257a",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_computer_science.yaml": "1816af75a73635c7705c35aef3072f15a22ee54aa305819880927dcf7e3a9ae5",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_economics.yaml": "ef1fcf83eadc67e89bb3b4381bad0eac16585c3bfc266bf5d124f99d4030d487",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_engineering.yaml": "fe1bcfa0009c42c7998c37ede092bcf3bb9b896dbbc5885980f92a89ea8f05b5",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_health.yaml": "80e6d0e0613752a2454df265c58f34f9448c4791e347893abda72305e6fbc64a",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_history.yaml": "047c600f10c0b38e0b3eff2c428079793c3d89ecd2bcc1b43019df6c83f84956",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_law.yaml": "7679544fafe2ada9c1e667e4adc13db1f319f420ecb97908ef48198737ba2289",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_math.yaml": "af96df4b28b32daa1846bc0f29264709ebe2e7a7c4789f137fa900ae61d49694",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_other.yaml": "87a294c66ccf38044a6016ad0ae0c5cb5a4c3f092ebd2c40dbdc6c4de3ff1580",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_philosophy.yaml": "fb0599414d22276942170f10fc4238a590e4981d04be08ceae9202af54fe0fe2",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_physics.yaml": "bfb64e6779bef91b5092dea22efee61f8ebf1e7888861558bd308a3a6d4e7196",
  "lm_eval_tasks/mmlu_prox_ru_chat/mmlu_prox_ru_chat_psychology.yaml": "cb4802cc2f6f07283d8a433cd55512d436a34d653d6ee5d09f1e0cf561e6a751",
  "lm_eval_tasks/mmlu_prox_ru_chat/utils.py": "99d10756ca88eee8fa3ff9fce3a4d4efb746b17a7a9b9feb7fcfbb76b15e5f3a",
  "requirements.txt": "0d8268adba83ce6c9062c5085e76c516a7e0b79e33791e33dd7686a452708052",
  "baseline.py": "4f20df851c435b79f59cb2db590c8c2978782ee364f21aba85bc709935cfb223",
  "run.py": "af0964f31d7aa2d84a09fa869c40bd9ae84129e24eb23abadf608f7586113f4e",
  "model_sources.json": "ce5701c04db49306083db628881b4da6fa630c518ddcfec395f43d7505f18f87",
  "protocol.md": "dd484a4a234cf6c2b2a227af7786f5ed164fd522dd9eebe8f864a9ff0bc38d02",
  "evaluate.py": "677065b71902fe7bd65a5943c89ecbc419c4b60a06dbcf9481f45a97ba491a07",
  "session.py": "c51e2514f4fc7c34243679869be5a32de12b72bb2bef793f7349e92a9d736525",
  "selection.json": "ea2fa68075e80de45e42e286e99c07dabf8c81a6b31fd851306d23db5cbdd988"
}
''',
}
for directory in (RUN_ROOT, STUDY / 'source'):
    for name, source in SOURCES.items():
        path = directory / name
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.exists():
            if path.read_text() != source:
                raise RuntimeError(f'Saved source differs: {path}')
        else:
            temporary = path.with_suffix(path.suffix + '.tmp')
            temporary.write_text(source, encoding='utf-8')
            os.replace(temporary, path)
expected = json.loads(SOURCES['source_manifest.json'])
for directory in (RUN_ROOT, STUDY / 'source'):
    for name, digest in expected.items():
        actual = hashlib.sha256((directory / name).read_bytes()).hexdigest()
        if actual != digest:
            raise RuntimeError(f'Source checksum mismatch: {name}')
sys.path.insert(0, str(RUN_ROOT))


In [ ]:
def export_results():
    name = 'LAPA_FP8_560_' + uuid4().hex + '.zip'
    archive = WORK_ROOT / name
    hashes = {}
    with zipfile.ZipFile(archive, 'x', zipfile.ZIP_DEFLATED) as output:
        for directory in (STUDY / 'source', RESULTS_DIR,
                          STUDY / 'sessions', STUDY / 'budget'):
            for path in sorted(directory.rglob('*')):
                if not path.is_file() or path.suffix == '.tmp':
                    continue
                relative = str(path.relative_to(STUDY))
                data = path.read_bytes()
                hashes[relative] = hashlib.sha256(data).hexdigest()
                output.writestr(relative, data)
        output.writestr('export_manifest.json', json.dumps({
            'schema_version': 1, 'files': hashes,
        }, indent=2))
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    print('ZIP:', archive, 'SHA-256:', digest)
    try:
        destination = STUDY / 'exports' / name
        destination.parent.mkdir(exist_ok=True)
        temporary = destination.with_suffix('.partial')
        shutil.copyfile(archive, temporary)
        if hashlib.sha256(temporary.read_bytes()).hexdigest() != digest:
            raise OSError('Drive export checksum mismatch')
        os.replace(temporary, destination)
        print('Drive:', destination)
    except OSError as error:
        print('Drive export failed; download the local ZIP:', error)
    try:
        from google.colab import files
        files.download(str(archive))
    except Exception as error:
        print('Download via Files:', archive, error)
    return archive


def summary_ready():
    manifest = RESULTS_DIR / 'final_manifest.json'
    if not manifest.exists():
        return False
    files = json.loads(manifest.read_text())
    if 'summary.json' not in files:
        raise ValueError('Final manifest has no summary')
    for name, expected in files.items():
        path = RESULTS_DIR / name
        if not path.resolve().is_relative_to(RESULTS_DIR.resolve()):
            raise ValueError('Unsafe final manifest path')
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError(f'Final result checksum mismatch: {name}')
    return True


def stop_process(process):
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(process.pid, sig)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            continue


def run_command(args, log_name):
    if BUDGET.closed:
        raise RuntimeError('Restart Python, then Run all to resume')
    log_path = SESSION_DIR / log_name
    if BUDGET.tick() <= 120:
        raise TimeoutError('Total budget exhausted')
    with log_path.open('a', encoding='utf-8') as stream:
        process = subprocess.Popen(
            args, cwd=RUN_ROOT, stdout=stream, stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        timer = threading.Timer(
            max(0, BUDGET.deadline - time.monotonic() - 120),
            stop_process, args=(process,),
        )
        timer.daemon = True
        try:
            timer.start()
            position = 0
            while True:
                left = BUDGET.tick() - 120
                if left <= 0:
                    raise TimeoutError('Total budget exhausted')
                try:
                    code = process.wait(timeout=min(20, left))
                    break
                except subprocess.TimeoutExpired:
                    with log_path.open() as reader:
                        reader.seek(position)
                        print(reader.read()[-2000:], end='', flush=True)
                        position = reader.tell()
        finally:
            timer.cancel()
            stop_process(process)
            if timer.ident is not None:
                timer.join(timeout=6)
        print(log_path.read_text()[-5000:])
        if code:
            raise RuntimeError(f'Process failed; see {log_path}')


def start_budget():
    from session import Budget

    return Budget(STUDY / 'budget', SESSION_STARTED)


BUDGET = start_budget()
print(f'Remaining budget: {max(0, BUDGET.tick()) / 3600:.2f} hours')


## Подготовка
Зависимости и модель — как в проверенном FP8 baseline.


In [ ]:
try:
    if not summary_ready():
        gpu = subprocess.check_output([
            'nvidia-smi', '--query-gpu=name,memory.total,driver_version',
            '--format=csv,noheader,nounits',
        ], text=True)
        fields = gpu.strip().split(',')
        if len(fields) != 3 or 'A100' not in fields[0]:
            raise RuntimeError('One A100 40 GB is required')
        if not 35000 < int(fields[1]) < 45000:
            raise RuntimeError('A100 40 GB is required')
        active = subprocess.check_output([
            'nvidia-smi', '--query-compute-apps=pid',
            '--format=csv,noheader,nounits',
        ], text=True).strip()
        if active:
            raise RuntimeError('GPU is busy; stop the previous run first')
        (SESSION_DIR / 'gpu.txt').write_text(gpu)
        print(gpu)
        ENV_PYTHON = WORK_ROOT / 'venv' / 'bin' / 'python'
        if not ENV_PYTHON.exists():
            venv.EnvBuilder(with_pip=False).create(ENV_PYTHON.parent.parent)
        run_command([
            sys.executable, '-m', 'pip', '--python', str(ENV_PYTHON),
            'install', '-r', str(RUN_ROOT / 'requirements.txt'),
        ], 'install.log')
        run_command([
            sys.executable, '-m', 'pip', '--python', str(ENV_PYTHON), 'check',
        ], 'pip_check.log')
        run_command([
            sys.executable, '-m', 'pip', '--python', str(ENV_PYTHON),
            'freeze', '--all',
        ], 'packages.txt')
except BaseException:
    try:
        BUDGET.tick(closed=True)
    finally:
        export_results()
    raise


## Прогон
Все 560 вопросов автоматически. Завершённые ответы сохраняются в Drive.
После обрыва: перезапустите Python и выполните все ячейки снова.
После аварии подождите две минуты: проверяется активность прошлой сессии.
Неучтённый интервал до продолжения консервативно входит в бюджет.


In [ ]:
try:
    if not summary_ready():
        run_command([
            str(ENV_PYTHON), str(RUN_ROOT / 'run.py'),
            str(RESULTS_DIR), str(BUDGET.deadline),
        ], 'run.log')
    summary = json.loads((RESULTS_DIR / 'summary.json').read_text())
    print(json.dumps(summary, ensure_ascii=False, indent=2))
finally:
    try:
        BUDGET.tick(closed=True)
    finally:
        archive = export_results()


ZIP скачивается в конце и сохраняется в Drive. Затем отключите GPU.
Повторное скачивание: `export_results()`.
Учёт времени — по работе ноутбука с резервом 2 минуты на сессию,
а не по счёту Colab. GPU-работоспособность новой версии ещё не проверена.
